In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [1]:
# %cd /kaggle/working

# !git clone https://github.com/ThaoTo2806/WeedsGalore_Ipr28.git
# %cd /kaggle/working/WeedsGalore_Ipr28

# !ls -lah
%cd /kaggle/working
!rm -rf WeedsGalore_Ipr28
!git clone -q https://github.com/ThaoTo2806/WeedsGalore_Ipr28.git
%cd /kaggle/working/WeedsGalore_Ipr28
!git log --oneline -1
REPO = '/kaggle/working/WeedsGalore_Ipr28'

/kaggle/working
/kaggle/working/WeedsGalore_Ipr28
1da3d0c (HEAD -> main, origin/main, origin/HEAD) Exp26 doc


In [2]:
!pip install -q "transformers==4.45.2" torchmetrics absl-py opencv-python-headless pytest scikit-learn thop
import transformers, torchmetrics, sklearn
print('transformers', transformers.__version__, '| torchmetrics', torchmetrics.__version__, '| sklearn', sklearn.__version__)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 69.4 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 27.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 76.5 MB/s eta 0:00:00:00:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
litellm 1.85.7 requires tokenizers<1.0,>=0.21.0, but you have tokenizers 0.20.3 which is incompatible.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.
transformers 4.45.2 | torchmetrics 1.9.0 | sklearn 1.6.1


In [ ]:
# !pip install -q transformers==4.45.2

In [ ]:
# !pip install -q datasets huggingface_hub

In [ ]:
# from huggingface_hub import snapshot_download

# dataset_path = snapshot_download(
#     repo_id="Voxel51/weeds-galore",
#     repo_type="dataset",
#     local_dir="/kaggle/working/weeds-galore"
# )

# print(dataset_path)

In [ ]:
# %cd /kaggle/working/WeedsGalore_Ipr28

# !pip install -q absl-py==1.3.0 torchmetrics==0.11.4 tensorboard==2.11.0

In [ ]:
# %cd /kaggle/working/WeedsGalore_Ipr28

# !pip install -q --upgrade "tensorboard>=2.20,<2.21"

In [ ]:
# !cp -r /kaggle/input/datasets/thothanh28/weedsgalore-dataset/weedsgalore-dataset /kaggle/working/weedsgalore-dataset

In [ ]:
# !rm -rf /kaggle/working/weedsgalore-dataset

In [3]:
%cd /kaggle/working

!wget -O weedsgalore-dataset.zip \
"https://doidata.gfz.de/weedsgalore_e_celikkan_2024/weedsgalore-dataset.zip"

/kaggle/working
--2026-11-13 06:18:39--  https://doidata.gfz.de/weedsgalore_e_celikkan_2024/weedsgalore-dataset.zip
Resolving doidata.gfz.de (doidata.gfz.de)... 139.17.228.163
Connecting to doidata.gfz.de (doidata.gfz.de)|139.17.228.163|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 336989804 (321M) [application/zip]
Saving to: ‘weedsgalore-dataset.zip’

weedsgalore-dataset 100%[===================>] 321.38M  31.1MB/s    in 11s     

2026-11-13 06:18:51 (28.4 MB/s) - ‘weedsgalore-dataset.zip’ saved [336989804/336989804]



In [4]:
!unzip -q /kaggle/working/weedsgalore-dataset.zip \
    -d /kaggle/working/

In [5]:
import glob, os
# 1) Ưu tiên dataset đã Add Input (thư mục chứa các ngày 2023-05-25, ...)
hits = sorted(glob.glob('/kaggle/input/**/2023-05-25/semantics', recursive=True))
if hits:
    DS = os.path.dirname(os.path.dirname(hits[0]))
elif os.path.isdir('/kaggle/working/weedsgalore-dataset/2023-05-25'):
    DS = '/kaggle/working/weedsgalore-dataset'
else:
    # 2) Nếu chưa có: tải theo cách bạn đã dùng trước đây vào /kaggle/working/weedsgalore-dataset
    #    (trang dữ liệu: https://doidata.gfz.de/weedsgalore_e_celikkan_2024/)
    raise SystemExit('Chưa tìm thấy dataset: hãy Add Input hoặc tải về /kaggle/working/weedsgalore-dataset')
print('DS =', DS)
for d in sorted(os.listdir(DS)):
    p = os.path.join(DS, d)
    if os.path.isdir(p):
        print(d, sorted(os.listdir(p)))

DS = /kaggle/input/datasets/thothanh28/weedsgalore-dataset/weedsgalore-dataset
2023-05-25 ['images', 'instances', 'semantics']
2023-05-30 ['images', 'instances', 'semantics']
2023-06-06 ['images', 'instances', 'semantics']
2023-06-15 ['images', 'instances', 'semantics']
splits ['test.txt', 'train.txt', 'val.txt']


In [6]:
import inspect, os, sys
REPO = '/kaggle/working/WeedsGalore_Ipr28'
os.chdir(REPO)
from transformers.models.segformer.modeling_segformer import (
    SegformerEncoder, SegformerEfficientSelfAttention, SegformerOverlapPatchEmbeddings)
src_attn = inspect.getsource(SegformerEfficientSelfAttention.forward)
src_enc = inspect.getsource(SegformerEncoder.forward)
checks = {
    'Q computed before SR (query on un-reduced hidden_states)':
        src_attn.find('self.query(hidden_states)') < src_attn.find('self.sr('),
    'SR conv called on 4D tensor then layer_norm': 'self.sr(hidden_states)' in src_attn and 'self.layer_norm(hidden_states)' in src_attn,
    'encoder loop: embedding -> blocks -> norm -> reshape': all(s in src_enc for s in ['embedding_layer(hidden_states)', 'blk(hidden_states, height, width', 'norm_layer(hidden_states)']),
}
for k, v in checks.items():
    print('OK ' if v else 'FAIL', k)
os.makedirs('results/env', exist_ok=True)
with open('results/env/hf_source_check.txt', 'w') as f:
    f.write(src_attn + '\n\n' + src_enc)

import sys
if f'{REPO}/src' not in sys.path:
    sys.path.insert(0, f'{REPO}/src')
assert os.path.exists(f'{REPO}/src/nets/ccmc_sr.py'), 'Repo chưa có code mới: commit + push rồi chạy lại S2'
from nets.ccmc_sr import CCMCSegFormer
m = CCMCSegFormer(prior_mode='learned_weed')
for name, mod in m.named_modules():
    if name.endswith('.sr'):
        print(name, type(mod).__name__, tuple(mod.weight.shape), 'stride', mod.stride)
assert all(checks.values()), 'API HF khác giả định -> dừng lại, gửi file results/env/hf_source_check.txt để review'

OK  Q computed before SR (query on un-reduced hidden_states)
OK  SR conv called on 4D tensor then layer_norm
OK  encoder loop: embedding -> blocks -> norm -> reshape


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/14.4M [00:00<?, ?B/s]

encoder.encoder.block.0.0.attention.self.sr Conv2d (32, 32, 8, 8) stride (8, 8)
encoder.encoder.block.0.1.attention.self.sr Conv2d (32, 32, 8, 8) stride (8, 8)
encoder.encoder.block.1.0.attention.self.sr Conv2d (64, 64, 4, 4) stride (4, 4)
encoder.encoder.block.1.1.attention.self.sr Conv2d (64, 64, 4, 4) stride (4, 4)
encoder.encoder.block.2.0.attention.self.sr Conv2d (160, 160, 2, 2) stride (2, 2)
encoder.encoder.block.2.1.attention.self.sr Conv2d (160, 160, 2, 2) stride (2, 2)


In [7]:
%cd /kaggle/working/WeedsGalore_Ipr28
!mkdir -p results/env
!python -m pytest -q tests/test_ccmc_sr.py 2>&1 | tee results/env/unit_tests.log | tail -n 15

/kaggle/working
......................................................                   [100%]
54 passed in 39.47s


In [8]:
from pathlib import Path
import sys, json, hashlib

P = Path("/kaggle/working/WeedsGalore_Ipr28")
R = P / "results/exp12"
ARMS = P / "configs/exp12/exp12_arms.json"
COMMON = P / "configs/exp12/exp12_common_config.json"

assert ARMS.is_file(), ARMS
assert COMMON.is_file(), COMMON

sys.path.insert(0, str(P / "src"))
from exp12_config import load_json, check_run_config

arms = load_json(ARMS)
common = load_json(COMMON)

print("REGISTERED ARMS:")
for arm_id, spec in arms["arms"].items():
    print(
        arm_id,
        "| seeds:", spec.get("seeds"),
        "| backbone:", spec.get("flags", {}).get("backbone"),
        "| FSP:", spec.get("flags", {}).get("use_fsp"),
    )

print("\nB2 CHECKPOINT AVAILABILITY:")
for s in range(5, 10):
    ckpt = R / "P_B2_LLR" / f"seed{s}" / "final_ema.pth"
    cfg = ckpt.parent / "run_config.json"

    if ckpt.is_file() and cfg.is_file():
        errors = check_run_config(cfg, common, arms)
        status = "READY" if not errors else f"CONFIG ERROR: {errors}"
    else:
        status = "MISSING"

    print(f"seed{s}: {status}")

print("\nTEST HAS NOT BEEN OPENED BY THIS CELL")

REGISTERED ARMS:
P_B0_LLR | seeds: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9] | backbone: b0 | FSP: None
P_B0_LLR_X15 | seeds: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9] | backbone: b0 | FSP: None
P_B1_LLR | seeds: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9] | backbone: b1 | FSP: None
P_B1_LLR_X15 | seeds: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9] | backbone: b1 | FSP: None
P_B2_LLR | seeds: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9] | backbone: b2 | FSP: None
P_B2_LLR_X15 | seeds: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9] | backbone: b2 | FSP: None
S0_B0 | seeds: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9] | backbone: b0 | FSP: None
S1_B0 | seeds: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9] | backbone: b0 | FSP: None
S2_B0 | seeds: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9] | backbone: b0 | FSP: None
S0_B1 | seeds: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9] | backbone: b1 | FSP: None
S1_B1 | seeds: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9] | backbone: b1 | FSP: None
S2_B1 | seeds: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9] | backbone: b1 | FSP: None
L12_B0_REF | seeds: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9] | backbone: b0 | FSP: None

In [9]:
from pathlib import Path

P = Path("/kaggle/working/WeedsGalore_Ipr28")
roots = [
    P / "results",
    Path("/kaggle/input/datasets/thanhthao28"),
    Path("/kaggle/input/datasets/thothanh28"),
]

print("=== SEARCH B2 CHECKPOINTS ===")

found = []

for root in roots:
    if not root.exists():
        continue

    for ckpt in root.rglob("final_ema.pth"):
        path = str(ckpt)
        if "B2" not in path.upper():
            continue
        if "exp12" not in path.lower():
            continue

        for seed in range(5, 10):
            if f"seed{seed}/" in path:
                found.append((seed, ckpt))

for seed in range(5, 10):
    matches = [p for s, p in found if s == seed]

    print(f"\nSEED {seed}")
    if matches:
        for p in matches:
            print("FOUND:", p)
            print(
                "CONFIG:",
                (p.parent / "run_config.json").is_file()
            )
    else:
        print("NOT FOUND")

print("\nTEST NOT ACCESSED")

=== SEARCH B2 CHECKPOINTS ===

SEED 5
NOT FOUND

SEED 6
NOT FOUND

SEED 7
NOT FOUND

SEED 8
NOT FOUND

SEED 9
NOT FOUND

TEST NOT ACCESSED


In [10]:
#TEST B2 seed 0–4, D4
# ============================================================
# EXP12 B2 + RAW LLR | EXPLORATORY TEST | SEEDS 0..4
# D4 = 4 rotations x horizontal flip (8 transforms)
# NO TRAINING | NO CHECKPOINT SELECTION ON TEST
# ============================================================

from pathlib import Path
import sys, json, hashlib, gc
import numpy as np
import pandas as pd
import torch
from scipy.stats import t as student_t

P = Path("/kaggle/working/WeedsGalore_Ipr28")
DS = Path(
    "/kaggle/input/datasets/thothanh28/"
    "weedsgalore-dataset/weedsgalore-dataset"
)
OUT = P / "results/exp12/EXPLORATORY_TEST_B2_SEEDS_0_4"
OUT.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(P / "src"))

from scripts.exp22_run import load_base, data
from datasets import WeedsGaloreDataset

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

SEEDS = list(range(5))
ARM = "P_B2_LLR"

# ------------------------------------------------------------
# 1. DISCOVER CHECKPOINTS — EXACT ARM ONLY
# ------------------------------------------------------------
roots = [
    P / "results/exp12",
    Path("/kaggle/input/datasets/thanhthao28"),
    Path("/kaggle/input/datasets/thothanh28"),
]

candidates = []

for root in roots:
    if root.exists():
        candidates.extend(root.rglob("final_ema.pth"))

ckpts = {}

for seed in SEEDS:
    matches = [
        p for p in candidates
        if ARM in p.parts
        and f"seed{seed}" in p.parts
        and p.parent.name == f"seed{seed}"
    ]

    # Ưu tiên checkpoint trong project, sau đó backup.
    matches = sorted(
        set(matches),
        key=lambda p: (
            not str(p).startswith(str(P)),
            str(p)
        )
    )

    print(f"seed{seed}:")
    if not matches:
        print("  MISSING")
        continue

    ckpts[seed] = matches[0]
    print("  SELECTED:", matches[0])

    if len(matches) > 1:
        print("  OTHER COPIES:", len(matches) - 1)

assert len(ckpts) == 5, (
    "Thiếu checkpoint B2 seed 0–4. "
    "Dừng trước khi mở TEST."
)

# ------------------------------------------------------------
# 2. FREEZE CHECKPOINT MANIFEST
# ------------------------------------------------------------
def sha256_file(path):
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(4 * 1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

manifest = {
    "experiment": "EXP12_B2_EXPLORATORY_TEST",
    "arm": ARM,
    "seeds": SEEDS,
    "tta": "D4_8_transforms",
    "checkpoints": {
        str(seed): {
            "path": str(ckpts[seed]),
            "sha256": sha256_file(ckpts[seed]),
        }
        for seed in SEEDS
    },
    "protocol": "exploratory_not_confirmation",
}

manifest_path = OUT / "locked_manifest.json"
manifest_path.write_text(
    json.dumps(manifest, indent=2),
    encoding="utf-8"
)

print("\nCHECKPOINT MANIFEST LOCKED")
for seed in SEEDS:
    print(
        seed,
        manifest["checkpoints"][str(seed)]["sha256"][:16]
    )

# ------------------------------------------------------------
# 3. DEFINE D4 AND VERIFY INVERSION
# ------------------------------------------------------------
def transform_d4(x, rotation, flip):
    if flip:
        x = torch.flip(x, dims=(-1,))
    return torch.rot90(x, k=rotation, dims=(-2, -1))

def inverse_d4(x, rotation, flip):
    x = torch.rot90(
        x, k=(-rotation) % 4, dims=(-2, -1)
    )
    if flip:
        x = torch.flip(x, dims=(-1,))
    return x

dummy = torch.arange(35).reshape(1, 1, 5, 7)

for flip in (False, True):
    for k in range(4):
        restored = inverse_d4(
            transform_d4(dummy, k, flip), k, flip
        )
        assert torch.equal(dummy, restored)

print("D4 INVERSE CHECK PASS")

@torch.inference_mode()
def predict_d4(model, x):
    acc = None

    for flip in (False, True):
        for k in range(4):
            xx = transform_d4(x, k, flip)

            logits = model(xx).float()

            logits = inverse_d4(
                logits, k, flip
            )

            if acc is None:
                acc = logits
            else:
                acc += logits

    return (acc / 8.0).argmax(dim=1)

def update_cm(cm, gt, pred):
    gt = gt.cpu().numpy().astype(np.int64)
    pred = pred.cpu().numpy().astype(np.int64)

    valid = (gt >= 0) & (gt < 3)

    values = (
        3 * gt[valid] + pred[valid]
    )

    cm += np.bincount(
        values, minlength=9
    ).reshape(3, 3)

def calculate_metrics(cm):
    cm = cm.astype(np.float64)
    tp = np.diag(cm)

    denominator = (
        cm.sum(axis=1)
        + cm.sum(axis=0)
        - tp
    )

    iou = np.divide(
        tp,
        denominator,
        out=np.zeros(3, dtype=np.float64),
        where=denominator > 0
    ) * 100

    return {
        "mIoU": float(iou.mean()),
        "IoU_bg": float(iou[0]),
        "IoU_crop": float(iou[1]),
        "IoU_weed": float(iou[2]),
    }

# ------------------------------------------------------------
# 4. TEST DATASET
# ------------------------------------------------------------
ds = WeedsGaloreDataset(
    str(DS), None, 5, 3,
    False, "test", False
)

assert len(ds) == 26, (
    f"Expected 26 TEST images, got {len(ds)}"
)

print("TEST IMAGES:", len(ds))

# ------------------------------------------------------------
# 5. RUN ONCE; SAVE EACH SEED IMMEDIATELY
# ------------------------------------------------------------
results = []

for seed in SEEDS:
    print("\n" + "=" * 60)
    print(f"TEST B2 SEED {seed} / D4")
    print("=" * 60)

    seed_output = OUT / f"seed{seed}_metrics.json"

    # Không ghi đè nếu cell bị chạy lại.
    if seed_output.exists():
        r = json.loads(seed_output.read_text())

        assert (
            r["checkpoint_sha256"]
            == manifest["checkpoints"][str(seed)]["sha256"]
        )

        print("ALREADY EVALUATED — REUSING SAVED METRICS")
        results.append(r)
        continue

    model = load_base(str(ckpts[seed]), DEVICE)
    model.eval()

    cm = np.zeros((3, 3), dtype=np.int64)

    for i in range(len(ds)):
        x, y = data(ds, i, DEVICE)
        pred = predict_d4(model, x)

        update_cm(cm, y, pred)

        if (i + 1) % 5 == 0 or i + 1 == len(ds):
            print(
                f"seed{seed}: {i+1}/{len(ds)}",
                flush=True
            )

    metrics = calculate_metrics(cm)

    r = {
        "seed": seed,
        "arm": ARM,
        "split": "test",
        "tta": "d4_8_transforms",
        "n_images": len(ds),
        "checkpoint_sha256":
            manifest["checkpoints"][str(seed)]["sha256"],
        "confusion_matrix": cm.tolist(),
        **metrics,
    }

    seed_output.write_text(
        json.dumps(r, indent=2),
        encoding="utf-8"
    )

    results.append(r)

    print(
        "mIoU:", round(metrics["mIoU"], 5),
        "| Crop:", round(metrics["IoU_crop"], 5),
        "| Weed:", round(metrics["IoU_weed"], 5),
    )

    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# ------------------------------------------------------------
# 6. FIVE-SEED SUMMARY
# ------------------------------------------------------------
df = pd.DataFrame(results).sort_values("seed")

metric_names = [
    "mIoU", "IoU_bg", "IoU_crop", "IoU_weed"
]

summary = {}

for metric in metric_names:
    values = df[metric].to_numpy(dtype=np.float64)

    mean = float(values.mean())
    sd = float(values.std(ddof=1))
    half_width = float(
        student_t.ppf(0.975, len(values)-1)
        * sd / np.sqrt(len(values))
    )

    summary[metric] = {
        "mean": mean,
        "std": sd,
        "ci95_low": mean - half_width,
        "ci95_high": mean + half_width,
    }

report = {
    "experiment": "EXP12_B2_EXPLORATORY_TEST",
    "split": "test",
    "seeds": SEEDS,
    "tta": "D4_8_transforms",
    "results": results,
    "summary": summary,
    "protocol": "exploratory_not_independent_confirmation",
}

df.to_csv(
    OUT / "test_per_seed.csv",
    index=False
)

(OUT / "test_summary.json").write_text(
    json.dumps(report, indent=2),
    encoding="utf-8"
)

print("\n" + "=" * 68)
print("EXP12 B2 — EXPLORATORY TEST D4 — SEEDS 0–4")
print("=" * 68)

print(
    df[
        ["seed", "mIoU",
         "IoU_bg", "IoU_crop", "IoU_weed"]
    ].round(5).to_string(index=False)
)

print("\nFIVE-SEED SUMMARY:")

for metric, stats in summary.items():
    print(
        f"{metric:9s}: "
        f"{stats['mean']:.5f} ± {stats['std']:.5f}"
        f" | 95% CI "
        f"[{stats['ci95_low']:.5f}, "
        f"{stats['ci95_high']:.5f}]"
    )

print("\nSAVED:", OUT)
print("EXPLORATORY TEST COMPLETE")

seed0:
  SELECTED: /kaggle/input/datasets/thanhthao28/exp12-backup/exp12/P_B2_LLR/seed0/final_ema.pth
seed1:
  SELECTED: /kaggle/input/datasets/thanhthao28/exp12-backup/exp12/P_B2_LLR/seed1/final_ema.pth
seed2:
  SELECTED: /kaggle/input/datasets/thanhthao28/exp12-backup/exp12/P_B2_LLR/seed2/final_ema.pth
seed3:
  SELECTED: /kaggle/input/datasets/thanhthao28/exp12-backup/exp12/P_B2_LLR/seed3/final_ema.pth
seed4:
  SELECTED: /kaggle/input/datasets/thanhthao28/exp12-backup/exp12/P_B2_LLR/seed4/final_ema.pth

CHECKPOINT MANIFEST LOCKED
0 b80a5f7794efb6ba
1 d3730eafd7d50976
2 85297b87b8588356
3 97ff2643f0a1695f
4 23503a41d8db8da5
D4 INVERSE CHECK PASS
TEST IMAGES: 26

TEST B2 SEED 0 / D4


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/99.0M [00:00<?, ?B/s]

seed0: 5/26
seed0: 10/26
seed0: 15/26
seed0: 20/26
seed0: 25/26
seed0: 26/26
mIoU: 86.3062 | Crop: 79.5892 | Weed: 80.7068

TEST B2 SEED 1 / D4
seed1: 5/26
seed1: 10/26
seed1: 15/26
seed1: 20/26
seed1: 25/26
seed1: 26/26
mIoU: 86.15637 | Crop: 79.21226 | Weed: 80.62919

TEST B2 SEED 2 / D4
seed2: 5/26
seed2: 10/26
seed2: 15/26
seed2: 20/26
seed2: 25/26
seed2: 26/26
mIoU: 86.47875 | Crop: 79.70084 | Weed: 81.09035

TEST B2 SEED 3 / D4
seed3: 5/26
seed3: 10/26
seed3: 15/26
seed3: 20/26
seed3: 25/26
seed3: 26/26
mIoU: 86.15888 | Crop: 78.89938 | Weed: 80.93496

TEST B2 SEED 4 / D4
seed4: 5/26
seed4: 10/26
seed4: 15/26
seed4: 20/26
seed4: 25/26
seed4: 26/26
mIoU: 86.15223 | Crop: 78.96297 | Weed: 80.85576

EXP12 B2 — EXPLORATORY TEST D4 — SEEDS 0–4
 seed     mIoU   IoU_bg  IoU_crop  IoU_weed
    0 86.30620 98.62261  79.58920  80.70680
    1 86.15637 98.62766  79.21226  80.62919
    2 86.47875 98.64506  79.70084  81.09035
    3 86.15888 98.64229  78.89938  80.93496
    4 86.15223 98.63795  

In [16]:
%cd /kaggle/working/WeedsGalore_Ipr28
!rm -f results/exp12/PER_IMAGE_AUDIT/perimg_*
!python src/scripts/exp12_eval_per_image.py --dataset_path=$DS --arms=P_B2_LLR --splits=val

/kaggle/working
D4 inverse check PASS
P_B2_LLR: seed0=b80a5f7794ef, seed1=d3730eafd7d5, seed2=85297b87b858, seed3=97ff2643f0a1, seed4=23503a41d8db

=== P_B2_LLR seed0 ===
  val: modes=['prob', 'logit']
    5/26
    10/26
    15/26
    20/26
    25/26
    26/26
  -> perimg_P_B2_LLR_val_prob_seed0.csv: mIoU 84.937 (bg 98.41 crop 78.17 weed 78.23)
  -> perimg_P_B2_LLR_val_logit_seed0.csv: mIoU 84.940 (bg 98.41 crop 78.18 weed 78.23)

=== P_B2_LLR seed1 ===
  val: modes=['prob', 'logit']
    5/26
    10/26
    15/26
    20/26
    25/26
    26/26
  -> perimg_P_B2_LLR_val_prob_seed1.csv: mIoU 84.873 (bg 98.40 crop 78.18 weed 78.03)
  -> perimg_P_B2_LLR_val_logit_seed1.csv: mIoU 84.866 (bg 98.40 crop 78.17 weed 78.03)

=== P_B2_LLR seed2 ===
  val: modes=['prob', 'logit']
    5/26
    10/26
    15/26
    20/26
    25/26
    26/26
  -> perimg_P_B2_LLR_val_prob_seed2.csv: mIoU 84.869 (bg 98.40 crop 78.26 weed 77.94)
  -> perimg_P_B2_LLR_val_logit_seed2.csv: mIoU 84.855 (bg 98.40 crop 78.24 weed

In [18]:
%cd /kaggle/working/WeedsGalore_Ipr28
D = "results/exp12/PER_IMAGE_AUDIT"
!python src/scripts/exp12_bootstrap.py --per_image_dir={D} --split=val  --mode=prob  --arms=P_B2_LLR,P_B0_LLR --thresholds=85
!python src/scripts/exp12_bootstrap.py --per_image_dir={D} --split=test --mode=logit --arms=P_B2_LLR,P_B0_LLR --thresholds=86

/kaggle/working

split=val mode=prob images=26 seeds=[0, 1, 2, 3, 4] draws=10000 (images only, seeds fixed)

P_B2_LLR: mIoU 84.72  image-bootstrap 95% CI [83.32, 86.15]  P(>85)=0.366
   seed-only t-interval (what the TEST cell printed): 84.72 ± 0.35 [84.28, 85.16]
   IoU_crop: 77.90 [75.91, 80.63]
   IoU_weed: 77.86 [75.29, 80.29]

P_B0_LLR: mIoU 84.11  image-bootstrap 95% CI [82.68, 85.68]  P(>85)=0.157
   seed-only t-interval (what the TEST cell printed): 84.11 ± 0.27 [83.78, 84.44]
   IoU_crop: 76.65 [74.33, 79.97]
   IoU_weed: 77.29 [74.75, 79.52]

DELTA P_B2_LLR - P_B0_LLR: +0.61 pp  95% CI [+0.33, +0.83]  P(delta<=0)=0.0000

Saved: results/exp12/PER_IMAGE_AUDIT/bootstrap_val_prob.json

split=test mode=logit images=26 seeds=[0, 1, 2, 3, 4] draws=10000 (images only, seeds fixed)

P_B2_LLR: mIoU 86.25  image-bootstrap 95% CI [83.77, 87.43]  P(>86)=0.564
   seed-only t-interval (what the TEST cell printed): 86.25 ± 0.14 [86.07, 86.43]
   IoU_crop: 79.27 [76.87, 81.43]
   IoU_weed: 80

In [19]:
!python src/scripts/exp12_bootstrap.py --per_image_dir={D} --split=test --mode=logit --arms=P_B2_LLR,P_B0_LLR --resample_seeds


split=test mode=logit images=26 seeds=[0, 1, 2, 3, 4] draws=10000 (images+seeds)

P_B2_LLR: mIoU 86.25  image-bootstrap 95% CI [83.78, 87.43]  P(>86)=0.562
   seed-only t-interval (what the TEST cell printed): 86.25 ± 0.14 [86.07, 86.43]
   IoU_crop: 79.27 [76.82, 81.45]
   IoU_weed: 80.84 [73.96, 84.24]

P_B0_LLR: mIoU 85.53  image-bootstrap 95% CI [83.13, 86.75]  P(>86)=0.208
   seed-only t-interval (what the TEST cell printed): 85.53 ± 0.17 [85.32, 85.74]
   IoU_crop: 77.80 [74.88, 80.48]
   IoU_weed: 80.17 [73.07, 83.72]

DELTA P_B2_LLR - P_B0_LLR: +0.72 pp  95% CI [+0.46, +1.00]  P(delta<=0)=0.0000

Saved: results/exp12/PER_IMAGE_AUDIT/bootstrap_test_logit.json


In [11]:
import shutil

src = "/kaggle/working/WeedsGalore_Ipr28/results/exp12"
dst = "/kaggle/working/exp12_test.zip"

shutil.make_archive(
    dst.replace(".zip", ""),
    "zip",
    root_dir=src
)

print("ZIP CREATED:", dst)

ZIP CREATED: /kaggle/working/exp12_test.zip


In [9]:
from pathlib import Path
import subprocess, sys

P = Path("/kaggle/working/WeedsGalore_Ipr28")

DS = (
    "/kaggle/input/datasets/thothanh28/"
    "weedsgalore-dataset/weedsgalore-dataset"
)

BASE = (
    "/kaggle/input/datasets/thothanh28/"
    "exp15-ab/results/exp15/"
    "E15_A_CW_LOVASZ/seed0/final_ema.pth"
)

OUT = P / "results/exp26/E26_PAIR/seed0"

common = [
    sys.executable,
    str(P / "src/scripts/exp26_run.py"),
    f"--dataset_path={DS}",
    f"--base_ckpt={BASE}",
    f"--out_dir={OUT}",
    "--seed=0",
    "--epochs=12",
    "--lr=1e-5",
    "--batch_size=1",
    "--grad_accum=2",
    "--boundary_weight=0.10",
    "--lovasz_weight=0.10",
]

subprocess.run(
    common + ["--mode=preflight"],
    cwd=P, check=True
)

for arm in ["control", "boundary"]:
    print("\nTRAINING EXP26:", arm.upper())

    subprocess.run(
        common + [
            "--mode=fit",
            f"--arm={arm}",
        ],
        cwd=P,
        check=True
    )

print("\nEXP26 BOTH ARMS TRAIN PASS")
print("RESULTS:", OUT)
print("VAL/TEST NOT ACCESSED")

EXP26 PREFLIGHT PASS base_sha256= 1e991c8624f9ffe8ba4243b09e04e4e6160b885a44d8c47e4ee8873945df945e split_hash= 675667edf3f35f16

TRAINING EXP26: CONTROL
EXP26 control epoch 1/12: train_loss=0.21664
EXP26 control epoch 2/12: train_loss=0.20342
EXP26 control epoch 3/12: train_loss=0.19887
EXP26 control epoch 4/12: train_loss=0.19449
EXP26 control epoch 5/12: train_loss=0.19304
EXP26 control epoch 6/12: train_loss=0.19120
EXP26 control epoch 7/12: train_loss=0.18953
EXP26 control epoch 8/12: train_loss=0.18928
EXP26 control epoch 9/12: train_loss=0.18856
EXP26 control epoch 10/12: train_loss=0.18699
EXP26 control epoch 11/12: train_loss=0.18666
EXP26 control epoch 12/12: train_loss=0.18665
EXP26 FIT PASS control FINAL EMA: /kaggle/working/WeedsGalore_Ipr28/results/exp26/E26_PAIR/seed0/control/final_ema.pth VAL/TEST NOT ACCESSED

TRAINING EXP26: BOUNDARY
EXP26 boundary epoch 1/12: train_loss=0.26071
EXP26 boundary epoch 2/12: train_loss=0.24480
EXP26 boundary epoch 3/12: train_loss=0.23900

In [10]:
#Cell đánh giá Validation D4
import json

subprocess.run(
    common + [
        "--mode=val",
        "--val_once",
        "--tta=d4",
    ],
    cwd=P,
    check=True
)

result = json.loads(
    (OUT / "val_exp26_compare.json").read_text()
)

print("\n=== EXP26 VALIDATION D4 ===")
print("EXP15-A BASE:", result["base"]["mIoU"])

for arm, r in result["arms"].items():
    print(
        arm,
        "| mIoU:", round(r["mIoU"], 5),
        "| delta:", round(r["delta_miou_pp"], 5),
        "| Crop:", round(r["IoU_crop"], 5),
        "| Weed:", round(r["IoU_weed"], 5),
    )

print("\nTEST SEALED")

EXP26 VAL 1 / 26
EXP26 VAL 2 / 26
EXP26 VAL 3 / 26
EXP26 VAL 4 / 26
EXP26 VAL 5 / 26
EXP26 VAL 6 / 26
EXP26 VAL 7 / 26
EXP26 VAL 8 / 26
EXP26 VAL 9 / 26
EXP26 VAL 10 / 26
EXP26 VAL 11 / 26
EXP26 VAL 12 / 26
EXP26 VAL 13 / 26
EXP26 VAL 14 / 26
EXP26 VAL 15 / 26
EXP26 VAL 16 / 26
EXP26 VAL 17 / 26
EXP26 VAL 18 / 26
EXP26 VAL 19 / 26
EXP26 VAL 20 / 26
EXP26 VAL 21 / 26
EXP26 VAL 22 / 26
EXP26 VAL 23 / 26
EXP26 VAL 24 / 26
EXP26 VAL 25 / 26
EXP26 VAL 26 / 26
{
  "split": "val",
  "tta": "d4",
  "n_images": 26,
  "seed": 0,
  "base_sha256": "1e991c8624f9ffe8ba4243b09e04e4e6160b885a44d8c47e4ee8873945df945e",
  "base": {
    "mIoU": 85.0247410542669,
    "IoU_bg": 98.39647892976096,
    "IoU_crop": 78.4831128838509,
    "IoU_weed": 78.19463134918882
  },
  "arms": {
    "control": {
      "mIoU": 84.154896260505,
      "IoU_bg": 98.38899718152977,
      "IoU_crop": 76.70176117928244,
      "IoU_weed": 77.37393042070278,
      "delta_miou_pp": -0.8698447937618994
    },
    "boundary": {
     

In [11]:
import shutil

src = "/kaggle/working/WeedsGalore_Ipr28/results/exp26"
dst = "/kaggle/working/exp26.zip"

shutil.make_archive(
    dst.replace(".zip", ""),
    "zip",
    root_dir=src
)

print("ZIP CREATED:", dst)

ZIP CREATED: /kaggle/working/exp26.zip


In [8]:
#Cache + kiểm toán lỗi quanh biên
from pathlib import Path
import subprocess, sys, json

P = Path("/kaggle/working/WeedsGalore_Ipr28")

DS = (
    "/kaggle/input/datasets/thothanh28/"
    "weedsgalore-dataset/weedsgalore-dataset"
)
BASE = (
    "/kaggle/input/datasets/thothanh28/"
    "exp15-ab/results/exp15/"
    "E15_A_CW_LOVASZ/seed0/final_ema.pth"
)

OUT = P / "results/exp25/E25_B2_OCC_BOUNDARY/seed0"
OLD = P / "results/exp23/E23_TWO_ARM/seed0"

common = [
    sys.executable,
    str(P / "src/scripts/exp25_run.py"),
    f"--dataset_path={DS}",
    f"--base_ckpt={BASE}",
    f"--out_dir={OUT}",
    "--seed=0",
]

cmd = common + ["--mode=cache"]

if (OLD / "cache_meta.json").is_file():
    cmd += [f"--reuse_exp23_cache={OLD}"]

subprocess.run(cmd, cwd=P, check=True)

subprocess.run(
    common + ["--mode=audit"],
    cwd=P, check=True
)

audit = json.loads(
    (OUT / "boundary_error_audit.json").read_text()
)

print("\n=== EXP25 BOUNDARY FEASIBILITY ===")
print("BG-Vegetation errors:", audit["bgveg_errors"])

for band, stats in audit["bins"].items():
    print(
        band,
        "pixels:",
        stats["bgveg_errors"],
        "share:",
        round(stats["share_of_bgveg_pct"], 3),
        "%"
    )

EXP25 CACHE 10 / 104
EXP25 CACHE 20 / 104
EXP25 CACHE 30 / 104
EXP25 CACHE 40 / 104
EXP25 CACHE 50 / 104
EXP25 CACHE 60 / 104
EXP25 CACHE 70 / 104
EXP25 CACHE 80 / 104
EXP25 CACHE 90 / 104
EXP25 CACHE 100 / 104
EXP25 CACHE 104 / 104
EXP25 CACHE PASS — TRAIN ONLY / TEST SEALED
AUDIT 20 / 104
AUDIT 40 / 104
AUDIT 60 / 104
AUDIT 80 / 104
AUDIT 100 / 104
{
  "split": "train",
  "images": 104,
  "error_total": 484356,
  "bgveg_errors": 460691,
  "crop_weed_errors": 23665,
  "bins": {
    "0-3": {
      "bgveg_errors": 424752,
      "share_of_bgveg_pct": 92.198892533173,
      "pixels_in_band": 3943122
    },
    "4-10": {
      "bgveg_errors": 22303,
      "share_of_bgveg_pct": 4.841205927617427,
      "pixels_in_band": 5299301
    },
    "11-25": {
      "bgveg_errors": 7230,
      "share_of_bgveg_pct": 1.5693816462661523,
      "pixels_in_band": 9402990
    },
    "26+": {
      "bgveg_errors": 6406,
      "share_of_bgveg_pct": 1.3905198929434264,
      "pixels_in_band": 18794587
    }
  

In [9]:
#Huấn luyện đủ ba nhánh exp25
for arm in ["occ", "boundary", "stage_boundary"]:
    print("\n" + "=" * 60)
    print("TRAINING:", arm.upper())
    print("=" * 60)

    subprocess.run(
        common + [
            "--mode=fit",
            f"--arm={arm}",
            "--epochs=15",
            "--patch=128",
            "--patches_per_image=3",
            "--holdout_every=3",
        ],
        cwd=P,
        check=True
    )

print("\n=== EXP25 TRAIN HOLDOUT RESULTS ===")

for arm in ["occ", "boundary", "stage_boundary"]:
    history = json.loads(
        (OUT / arm / "history.json").read_text()
    )
    best = max(
        history,
        key=lambda x: x["delta_miou_pp"]
    )

    print(
        arm,
        "| best epoch:", best["epoch"] + 1,
        "| delta mIoU:", round(best["delta_miou_pp"], 5),
        "| corrected:", best["corrected"],
        "| damaged:", best["damaged"]
    )

print("\nEXP25 THREE ARMS TRAIN PASS")


TRAINING: OCC
occ epoch 3/15: holdout delta=-0.18065 pp; correct=6475, damaged=9491; best=-0.18065
occ epoch 6/15: holdout delta=-0.19283 pp; correct=7314, damaged=10705; best=-0.18065
occ epoch 9/15: holdout delta=-0.08802 pp; correct=8803, damaged=11578; best=-0.08802
occ epoch 12/15: holdout delta=-0.24434 pp; correct=10624, damaged=15376; best=-0.08802
occ epoch 15/15: holdout delta=-0.10723 pp; correct=10218, damaged=12888; best=-0.08802
EXP25 FIT PASS occ TRAIN HOLDOUT ONLY, BACKBONE IN-SAMPLE

TRAINING: BOUNDARY
boundary epoch 3/15: holdout delta=-0.14750 pp; correct=4985, damaged=7317; best=-0.14750
boundary epoch 6/15: holdout delta=-0.06229 pp; correct=4208, damaged=5637; best=-0.06229
boundary epoch 9/15: holdout delta=-0.13196 pp; correct=6740, damaged=9367; best=-0.06229
boundary epoch 12/15: holdout delta=-0.27620 pp; correct=9173, damaged=13719; best=-0.06229
boundary epoch 15/15: holdout delta=-0.20565 pp; correct=8668, damaged=11770; best=-0.06229
EXP25 FIT PASS bound

In [10]:
#So sánh Validation D4
subprocess.run(
    common + [
        "--mode=val",
        "--val_once",
        "--tta=d4",
    ],
    cwd=P,
    check=True
)

result = json.loads(
    (OUT / "val_exp25_compare.json").read_text()
)

print("\n=== EXP25 VALIDATION D4 ===")
print("BASE:", result["base"]["mIoU"])

for arm, r in result["arms"].items():
    print(
        arm,
        "| mIoU:", round(r["mIoU"], 5),
        "| delta:", round(r["delta_miou_pp"], 5),
        "| Crop:", round(r["IoU_crop"], 5),
        "| Weed:", round(r["IoU_weed"], 5),
        "| corrected:", r["corrected"],
        "| damaged:", r["damaged"]
    )

print("\nTEST SEALED")

EXP25 VAL 1 / 26
EXP25 VAL 2 / 26
EXP25 VAL 3 / 26
EXP25 VAL 4 / 26
EXP25 VAL 5 / 26
EXP25 VAL 6 / 26
EXP25 VAL 7 / 26
EXP25 VAL 8 / 26
EXP25 VAL 9 / 26
EXP25 VAL 10 / 26
EXP25 VAL 11 / 26
EXP25 VAL 12 / 26
EXP25 VAL 13 / 26
EXP25 VAL 14 / 26
EXP25 VAL 15 / 26
EXP25 VAL 16 / 26
EXP25 VAL 17 / 26
EXP25 VAL 18 / 26
EXP25 VAL 19 / 26
EXP25 VAL 20 / 26
EXP25 VAL 21 / 26
EXP25 VAL 22 / 26
EXP25 VAL 23 / 26
EXP25 VAL 24 / 26
EXP25 VAL 25 / 26
EXP25 VAL 26 / 26
{
  "split": "val",
  "tta": "d4",
  "n_images": 26,
  "seed": 0,
  "base_sha256": "1e991c8624f9ffe8ba4243b09e04e4e6160b885a44d8c47e4ee8873945df945e",
  "base": {
    "mIoU": 85.0247410542669,
    "IoU_bg": 98.39647892976096,
    "IoU_crop": 78.4831128838509,
    "IoU_weed": 78.19463134918882
  },
  "arms": {
    "occ": {
      "mIoU": 84.82538176217639,
      "IoU_bg": 98.33670890592886,
      "IoU_crop": 78.36561541702181,
      "IoU_weed": 77.77382096357847,
      "delta_miou_pp": -0.19935929209050585,
      "corrected": 8901,
     

In [11]:
import shutil

src = "/kaggle/working/WeedsGalore_Ipr28/results/exp25"
dst = "/kaggle/working/exp25.zip"

shutil.make_archive(
    dst.replace(".zip", ""),
    "zip",
    root_dir=src
)

print("ZIP CREATED:", dst)

ZIP CREATED: /kaggle/working/exp25.zip


In [8]:
from pathlib import Path
import subprocess
import sys

P = Path("/kaggle/working/WeedsGalore_Ipr28")

assert (P / "src/scripts/exp22_run.py").is_file(), (
    "Thiếu exp22_run.py — Exp23 cần file này để nạp Exp15-A"
)

subprocess.run(
    [
        sys.executable,
        "-m", "pytest", "-q",
        "tests/test_exp23_pixel.py",
    ],
    cwd=P,
    check=True,
)

print("EXP23 PREFLIGHT PASS")

....                                                                     [100%]
4 passed in 2.91s
EXP23 PREFLIGHT PASS


In [9]:
#Cache train và chạy cả hai phương án exp23
from pathlib import Path
import subprocess
import sys

P = Path("/kaggle/working/WeedsGalore_Ipr28")

DS = (
    "/kaggle/input/datasets/thothanh28/"
    "weedsgalore-dataset/weedsgalore-dataset"
)

BASE = (
    "/kaggle/input/datasets/thothanh28/"
    "exp15-ab/results/exp15/"
    "E15_A_CW_LOVASZ/seed0/final_ema.pth"
)

OUT = P / "results/exp23/E23_TWO_ARM/seed0"

assert Path(BASE).is_file()
assert Path(DS).is_dir()

common = [
    sys.executable,
    "src/scripts/exp23_run.py",
    f"--dataset_path={DS}",
    f"--base_ckpt={BASE}",
    f"--out_dir={OUT}",
    "--seed=0",
]

# Stage 1: Tạo cache từ 104 ảnh train
subprocess.run(
    common + ["--mode=cache"],
    cwd=P,
    check=True
)

# Stage 2: Exp23-A — Không dùng ngày
subprocess.run(
    common + [
        "--mode=fit",
        "--arm=nodate",
        "--epochs=20",
    ],
    cwd=P,
    check=True
)

# Stage 3: Exp23-B — Có dùng ngày
subprocess.run(
    common + [
        "--mode=fit",
        "--arm=stage",
        "--epochs=20",
    ],
    cwd=P,
    check=True
)

print("\nEXP23 BOTH ARMS TRAIN PASS")
print("RESULTS:", OUT)

cached 10 / 104 positive pixels: 1328
cached 20 / 104 positive pixels: 1635
cached 30 / 104 positive pixels: 2719
cached 40 / 104 positive pixels: 172
cached 50 / 104 positive pixels: 633
cached 60 / 104 positive pixels: 548
cached 70 / 104 positive pixels: 683
cached 80 / 104 positive pixels: 168
cached 90 / 104 positive pixels: 315
cached 100 / 104 positive pixels: 294
cached 104 / 104 positive pixels: 2716
EXP23 CACHE PASS: 104 TRAIN images. NO VAL/TEST.
nodate epoch 5/20: holdout delta=+0.0158 pp; corrected=692, damaged=432; best=+0.0158
nodate epoch 10/20: holdout delta=-0.1067 pp; corrected=4929, damaged=4907; best=+0.0158
nodate epoch 15/20: holdout delta=-0.2879 pp; corrected=7669, damaged=9289; best=+0.0158
nodate epoch 20/20: holdout delta=-0.3928 pp; corrected=9473, damaged=11728; best=+0.0158
EXP23 FIT PASS nodate — holdout is policy-only; base trained on 104.
stage epoch 5/20: holdout delta=+0.0094 pp; corrected=599, damaged=409; best=+0.0094
stage epoch 10/20: holdout del

In [10]:
#Xem kết quả nội bộ của hai phương án exp23
import json

for arm in ["nodate", "stage"]:
    path = OUT / arm / "policy_history.json"
    assert path.is_file(), path

    history = json.loads(path.read_text())
    best = max(history, key=lambda x: x["delta_miou_pp"])

    print("\n" + "=" * 55)
    print("ARM:", arm.upper())
    print("=" * 55)
    print("BEST EPOCH:", best["epoch"])
    print("HOLDOUT BASE:", best["base"]["mIoU"])
    print("HOLDOUT NEW:", best["new"]["mIoU"])
    print("DELTA mIoU (pp):", best["delta_miou_pp"])
    print("CORRECTED:", best["corrected"])
    print("DAMAGED:", best["damaged"])
    print("NET CORRECTED:", best["net_corrected"])


ARM: NODATE
BEST EPOCH: 4
HOLDOUT BASE: 86.28470853369932
HOLDOUT NEW: 86.30051924888284
DELTA mIoU (pp): 0.01581071518351962
CORRECTED: 692
DAMAGED: 432
NET CORRECTED: 260

ARM: STAGE
BEST EPOCH: 4
HOLDOUT BASE: 86.28470853369932
HOLDOUT NEW: 86.29410487806233
DELTA mIoU (pp): 0.009396344363011622
CORRECTED: 599
DAMAGED: 409
NET CORRECTED: 190


In [11]:
#Đánh giá cả hai phương án trên validation D4
subprocess.run(
    common + [
        "--mode=val",
        "--val_once",
        "--tta=d4",
    ],
    cwd=P,
    check=True
)

result = json.loads(
    (OUT / "val_exp23_compare.json").read_text()
)

print("\n" + "=" * 60)
print("EXP23 VALIDATION D4 — TWO ARM COMPARISON")
print("=" * 60)

print("EXP15-A BASE:", result["base"]["mIoU"])

for arm in ["nodate", "stage"]:
    r = result["policy_arm_comparison"][arm]

    print("\n", arm.upper())
    print("mIoU:", r["mIoU"])
    print("Delta (pp):", r["delta_miou_pp"])
    print("Crop IoU:", r["IoU_crop"])
    print("Weed IoU:", r["IoU_weed"])
    print("Corrected:", r["corrected"])
    print("Damaged:", r["damaged"])
    print("Net corrected:", r["net_corrected"])

print("\nTEST SEALED")

evaluated val 1 / 26
evaluated val 2 / 26
evaluated val 3 / 26
evaluated val 4 / 26
evaluated val 5 / 26
evaluated val 6 / 26
evaluated val 7 / 26
evaluated val 8 / 26
evaluated val 9 / 26
evaluated val 10 / 26
evaluated val 11 / 26
evaluated val 12 / 26
evaluated val 13 / 26
evaluated val 14 / 26
evaluated val 15 / 26
evaluated val 16 / 26
evaluated val 17 / 26
evaluated val 18 / 26
evaluated val 19 / 26
evaluated val 20 / 26
evaluated val 21 / 26
evaluated val 22 / 26
evaluated val 23 / 26
evaluated val 24 / 26
evaluated val 25 / 26
evaluated val 26 / 26
{
  "split": "val",
  "tta": "d4",
  "n_images": 26,
  "seed": 0,
  "base_sha256": "1e991c8624f9ffe8ba4243b09e04e4e6160b885a44d8c47e4ee8873945df945e",
  "base": {
    "mIoU": 85.0247410542669,
    "IoU_bg": 98.39647892976096,
    "IoU_crop": 78.4831128838509,
    "IoU_weed": 78.19463134918882
  },
  "policy_arm_comparison": {
    "nodate": {
      "mIoU": 85.02522754207621,
      "IoU_bg": 98.39716918617667,
      "IoU_crop": 78.4833

In [12]:
# ============================================================
# EXP24 PRE-AUDIT — CROP ROW GEOMETRY + GEO METADATA
# TRAIN ONLY | NO VAL | NO TEST | NO MODEL TRAINING
# ============================================================

from pathlib import Path
from collections import Counter, defaultdict
import sys, json
import numpy as np
import pandas as pd
import torch
from scipy.ndimage import rotate, gaussian_filter1d
from scipy.signal import find_peaks

P = Path("/kaggle/working/WeedsGalore_Ipr28")
DS = Path(
    "/kaggle/input/datasets/thothanh28/"
    "weedsgalore-dataset/weedsgalore-dataset"
)

OUT = P / "results/exp24/geometry_audit"
OUT.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(P / "src"))
from datasets import WeedsGaloreDataset

# ------------------------------------------------------------
# 1. TRAIN DATASET
# ------------------------------------------------------------
ds = WeedsGaloreDataset(
    str(DS), None, 5, 3,
    False, "train", False
)

assert len(ds) == 104

print("TRAIN IMAGES:", len(ds))

# ------------------------------------------------------------
# 2. CROP-ROW GEOMETRY
# ------------------------------------------------------------
# Tìm hướng mà các vùng Crop có cấu trúc hàng rõ nhất.
# Dùng GT train chỉ để audit, KHÔNG phải inference.
#
# Ảnh được lấy mẫu xuống 150x150 để chạy nhanh.
# Tại mỗi góc, chiếu mask Crop theo trục ngang.
# Dùng số đỉnh và độ nổi bật để đo cấu trúc ứng viên.

ANGLES = np.arange(0, 180, 5)

def row_geometry_score(mask_crop):
    small = mask_crop[::4, ::4].astype(np.float32)

    scores = []

    for angle in ANGLES:
        rotated = rotate(
            small,
            angle=float(angle),
            reshape=False,
            order=0,
            mode="constant",
            cval=0.0
        )

        projection = rotated.sum(axis=0)
        projection = gaussian_filter1d(
            projection, sigma=2.0
        )

        mean = float(projection.mean())
        std = float(projection.std())

        score = std / (mean + 1e-6)
        scores.append(score)

    best_idx = int(np.argmax(scores))
    best_angle = float(ANGLES[best_idx])

    rotated = rotate(
        small,
        angle=best_angle,
        reshape=False,
        order=0,
        mode="constant",
        cval=0.0
    )

    projection = gaussian_filter1d(
        rotated.sum(axis=0),
        sigma=2.0
    )

    prominence = max(
        1.0,
        float(projection.max()) * 0.20
    )

    peaks, _ = find_peaks(
        projection,
        prominence=prominence,
        distance=7
    )

    return {
        "best_angle": best_angle,
        "row_score": float(scores[best_idx]),
        "peak_count": int(len(peaks)),
        "orientation_contrast": float(
            max(scores) / (np.median(scores) + 1e-6)
        ),
    }

records = []

for i in range(len(ds)):
    sample = ds[i]

    # Dataset trả về (image, label)
    x, y = sample[:2]

    if torch.is_tensor(y):
        mask = y.detach().cpu().numpy()
    else:
        mask = np.asarray(y)

    mask = np.squeeze(mask)
    assert mask.shape == (600, 600), mask.shape

    name = str(ds.img_list[i])
    date = name[:10]

    crop = (mask == 1)
    weed = (mask == 2)

    geom = row_geometry_score(crop)

    records.append({
        "image": name,
        "date": date,
        "crop_fraction": float(crop.mean()),
        "weed_fraction": float(weed.mean()),
        **geom
    })

    if (i + 1) % 20 == 0:
        print("GEOMETRY AUDITED:", i + 1, "/", len(ds))

df = pd.DataFrame(records)

print("\n" + "=" * 65)
print("CROP ROW GEOMETRY")
print("=" * 65)

print(
    df.groupby("date")[
        [
            "crop_fraction",
            "weed_fraction",
            "row_score",
            "orientation_contrast",
            "peak_count"
        ]
    ].agg(["mean", "median"]).round(3).to_string()
)

print("\nPEAK COUNT DISTRIBUTION:")
print(df["peak_count"].value_counts().sort_index())

print("\nIMAGES WITH >= 2 ROW PEAK CANDIDATES:",
      int((df["peak_count"] >= 2).sum()), "/", len(df))

print("\nTOP 10 ROW CANDIDATES:")
print(
    df.sort_values(
        ["peak_count", "orientation_contrast"],
        ascending=False
    )[
        ["image", "best_angle", "peak_count",
         "orientation_contrast", "row_score"]
    ].head(10).to_string(index=False)
)

# ------------------------------------------------------------
# 3. GEO-METADATA AUDIT
# ------------------------------------------------------------
# Chỉ liệt kê tệp; không đọc ảnh VAL/TEST.
# Không dùng số hậu tố filename làm tọa độ.

print("\n" + "=" * 65)
print("GEO METADATA AUDIT")
print("=" * 65)

GEO_EXTENSIONS = {
    ".tif", ".tiff", ".tfw", ".wld",
    ".prj", ".shp", ".shx", ".geojson",
    ".gpkg", ".kml", ".kmz", ".vrt",
    ".csv", ".json", ".xml"
}

metadata_files = []

# Kiểm tra file metadata ở root và thư mục theo ngày.
# Không quét sâu vào thư mục ảnh/nhãn.
scan_dirs = [DS] + [
    DS / date
    for date in sorted(df["date"].unique())
]

for folder in scan_dirs:
    if not folder.is_dir():
        continue

    for f in folder.iterdir():
        if f.is_file() and f.suffix.lower() in GEO_EXTENSIONS:
            metadata_files.append(str(f.relative_to(DS)))

print("GEO/METADATA FILE COUNT:", len(metadata_files))
for f in metadata_files[:60]:
    print(" ", f)

if not metadata_files:
    print(
        "Không thấy metadata ở các thư mục đã kiểm tra. "
        "Chưa loại trừ metadata trong thư mục con hoặc nguồn gốc khác."
    )

# ------------------------------------------------------------
# 4. SAVE
# ------------------------------------------------------------
csv_path = OUT / "train_row_geometry.csv"
json_path = OUT / "geometry_audit_summary.json"

df.to_csv(csv_path, index=False)

summary = {
    "split": "train",
    "n_images": len(ds),
    "date_counts": df["date"].value_counts().to_dict(),
    "images_with_two_or_more_candidate_peaks": int(
        (df["peak_count"] >= 2).sum()
    ),
    "median_orientation_contrast": float(
        df["orientation_contrast"].median()
    ),
    "median_row_score": float(
        df["row_score"].median()
    ),
    "metadata_files": metadata_files,
    "geometry_uses_train_ground_truth": True,
    "row_registration_verified": False,
    "temporal_pairing_verified": False,
    "validation_accessed": False,
    "test_accessed": False
}

json_path.write_text(
    json.dumps(summary, indent=2),
    encoding="utf-8"
)

print("\n" + "=" * 65)
print("EXP24 AUDIT COMPLETE")
print("=" * 65)

print("Median orientation contrast:",
      round(summary["median_orientation_contrast"], 4))
print("Images with >=2 candidate peaks:",
      summary["images_with_two_or_more_candidate_peaks"])
print("Metadata files found:",
      len(metadata_files))

print("\nSAVED:")
print(csv_path)
print(json_path)

print("\nTRAIN ONLY — VAL/TEST SEALED")

TRAIN IMAGES: 104
GEOMETRY AUDITED: 20 / 104
GEOMETRY AUDITED: 40 / 104
GEOMETRY AUDITED: 60 / 104
GEOMETRY AUDITED: 80 / 104
GEOMETRY AUDITED: 100 / 104

CROP ROW GEOMETRY
           crop_fraction        weed_fraction        row_score        orientation_contrast        peak_count       
                    mean median          mean median      mean median                 mean median       mean median
date                                                                                                               
2023-05-25         0.007  0.007         0.000  0.000     2.126  2.051                2.165  2.246      2.500    2.5
2023-05-30         0.012  0.012         0.005  0.004     2.105  1.993                1.805  1.734      1.688    1.5
2023-06-06         0.033  0.032         0.032  0.025     1.741  1.756                1.971  2.041      1.938    2.0
2023-06-15         0.109  0.121         0.202  0.229     1.272  1.144                1.960  2.105      2.000    2.0

PEAK COUNT DIS

In [13]:
# ============================================================
# EXP24-B: GT-FREE ROW PRIOR FEASIBILITY
# TRAIN ONLY — NO VAL / NO TEST / NO TRAINING
# Uses Exp15-A predictions to estimate crop-row structure.
# GT used ONLY for auditing the resulting geometric signal.
# ============================================================

from pathlib import Path
from collections import defaultdict
import sys, json
import numpy as np
import torch
import pandas as pd
from scipy.ndimage import gaussian_filter1d
from scipy.signal import find_peaks
from sklearn.metrics import roc_auc_score

P = Path("/kaggle/working/WeedsGalore_Ipr28")
DS = Path(
    "/kaggle/input/datasets/thothanh28/"
    "weedsgalore-dataset/weedsgalore-dataset"
)
BASE = Path(
    "/kaggle/input/datasets/thothanh28/"
    "exp15-ab/results/exp15/"
    "E15_A_CW_LOVASZ/seed0/final_ema.pth"
)
OUT = P / "results/exp24/row_prior_audit"
OUT.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(P / "src"))

from scripts.exp22_run import load_base, data
from datasets import WeedsGaloreDataset

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = load_base(str(BASE), DEVICE)
model.eval()

ds = WeedsGaloreDataset(
    str(DS), None, 5, 3, False, "train", False
)
assert len(ds) == 104

ANGLES = np.arange(0, 180, 5, dtype=np.float32)

def estimate_rows_from_prediction(pred):
    """
    Input: predicted class mask only.
    No GT is used in this function.

    Output:
      dist_map: normalized distance to nearest candidate crop row
      info: estimated angle, number of peaks, etc.

    The result is a geometric proxy, NOT verified crop rows.
    """
    crop = (pred == 1).astype(np.float32)
    H, W = crop.shape

    # Smooth to suppress isolated predicted crop pixels.
    from scipy.ndimage import gaussian_filter

    crop_smooth = gaussian_filter(crop, sigma=3.0)

    yy, xx = np.indices((H, W))
    cx = (W - 1) / 2
    cy = (H - 1) / 2
    x = xx - cx
    y = yy - cy

    best = None

    for angle in ANGLES:
        theta = np.deg2rad(angle)
        t = x * np.cos(theta) + y * np.sin(theta)

        # Projection bins along candidate normal direction.
        ti = np.rint(t + H + W).astype(np.int32)
        n_bins = 2 * (H + W) + 1

        profile = np.bincount(
            ti.ravel(),
            weights=crop_smooth.ravel(),
            minlength=n_bins
        )

        # Normalize for varying line lengths within image.
        support = np.bincount(
            ti.ravel(),
            minlength=n_bins
        )

        profile = profile / np.maximum(support, 1)
        profile = gaussian_filter1d(profile, sigma=5)

        # Restrict to reasonably supported bins.
        good = support >= 0.35 * max(H, W)
        vals = profile[good]

        if len(vals) == 0:
            continue

        score = float(vals.std() / (vals.mean() + 1e-6))

        if best is None or score > best["score"]:
            best = {
                "score": score,
                "angle": float(angle),
                "profile": profile,
                "support": support,
                "ti": ti,
                "good": good
            }

    if best is None or crop.sum() < 30:
        return None, {
            "status": "insufficient_crop",
            "crop_pixels": int(crop.sum())
        }

    profile = best["profile"].copy()
    good = best["good"]

    # Peak detection inside supported part of projection.
    prominence = max(
        0.015,
        float(profile[good].max()) * 0.20
    )

    peaks, _ = find_peaks(
        profile,
        prominence=prominence,
        distance=35
    )

    peaks = peaks[good[peaks]]

    if len(peaks) == 0:
        return None, {
            "status": "no_row_peaks",
            "crop_pixels": int(crop.sum()),
            "angle": best["angle"]
        }

    # Distance from every pixel to nearest detected line,
    # measured in projection-coordinate pixels.
    axis = best["ti"].astype(np.float32)
    distance = np.full((H, W), np.inf, dtype=np.float32)

    for p in peaks:
        distance = np.minimum(
            distance,
            np.abs(axis - float(p))
        )

    return distance, {
        "status": "ok",
        "crop_pixels": int(crop.sum()),
        "angle": best["angle"],
        "peak_count": int(len(peaks)),
        "score": float(best["score"]),
    }


rows = []
all_labels = []
all_scores = []

for i in range(len(ds)):
    x, y = data(ds, i, DEVICE)

    with torch.inference_mode():
        logits = model(x).float()
        pred = logits.argmax(dim=1)[0].cpu().numpy()

    gt = y[0].cpu().numpy()
    name = str(ds.img_list[i])

    distance, info = estimate_rows_from_prediction(pred)

    record = {
        "image": name,
        "date": name[:10],
        **info
    }

    if distance is not None:
        # Evaluate ONLY pixels labeled Crop or Weed.
        vegetation = (gt == 1) | (gt == 2)

        crop_d = distance[(gt == 1)]
        weed_d = distance[(gt == 2)]

        record["crop_mean_distance"] = (
            float(crop_d.mean()) if len(crop_d) else np.nan
        )
        record["weed_mean_distance"] = (
            float(weed_d.mean()) if len(weed_d) else np.nan
        )

        # Lower distance -> predicts Crop.
        # For each image, summarize class separation.
        if np.any(gt == 1) and np.any(gt == 2):
            ys = (gt[vegetation] == 1).astype(np.uint8)
            sc = -distance[vegetation]

            record["crop_vs_weed_auc"] = float(
                roc_auc_score(ys, sc)
            )

            # Subsample for an aggregate AUC without
            # collecting all pixels from every image.
            rng = np.random.default_rng(1000 + i)
            ids = rng.choice(
                len(ys),
                size=min(12000, len(ys)),
                replace=False
            )

            all_labels.append(ys[ids])
            all_scores.append(sc[ids])
        else:
            record["crop_vs_weed_auc"] = np.nan

    rows.append(record)

    if (i + 1) % 10 == 0 or i + 1 == len(ds):
        print(f"AUDITED {i+1}/{len(ds)}", flush=True)

df = pd.DataFrame(rows)

usable = df[df["status"] == "ok"]

if all_labels:
    aggregate_auc = roc_auc_score(
        np.concatenate(all_labels),
        np.concatenate(all_scores)
    )
else:
    aggregate_auc = None

print("\n" + "=" * 65)
print("EXP24 GT-FREE ROW PRIOR AUDIT")
print("=" * 65)
print("Images with estimated rows:", len(usable), "/", len(ds))
print("Median estimated peaks:",
      usable["peak_count"].median() if len(usable) else None)
print("Median geometry score:",
      usable["score"].median() if len(usable) else None)
print("Median per-image Crop-vs-Weed AUC:",
      usable["crop_vs_weed_auc"].median()
      if len(usable) else None)
print("Pooled sampled Crop-vs-Weed AUC:", aggregate_auc)

print("\nBY DATE:")
if len(usable):
    print(
        usable.groupby("date")[
            ["peak_count", "crop_vs_weed_auc",
             "crop_mean_distance", "weed_mean_distance"]
        ].median().round(4).to_string()
    )

print("\nSTATUS DISTRIBUTION:")
print(df["status"].value_counts().to_string())

summary = {
    "split": "train",
    "n_images": len(ds),
    "usable_images": len(usable),
    "median_per_image_auc": (
        float(usable["crop_vs_weed_auc"].median())
        if len(usable) else None
    ),
    "pooled_sampled_auc": (
        float(aggregate_auc)
        if aggregate_auc is not None else None
    ),
    "row_estimated_from": "Exp15-A predictions only",
    "GT_used_for": "audit metrics only",
    "validation_accessed": False,
    "test_accessed": False
}

df.to_csv(OUT / "row_prior_per_image.csv", index=False)
(OUT / "row_prior_summary.json").write_text(
    json.dumps(summary, indent=2)
)

print("\nSAVED:", OUT)
print("EXP24 ROW PRIOR AUDIT PASS")

AUDITED 10/104
AUDITED 20/104
AUDITED 30/104
AUDITED 40/104
AUDITED 50/104
AUDITED 60/104
AUDITED 70/104
AUDITED 80/104
AUDITED 90/104
AUDITED 100/104
AUDITED 104/104

EXP24 GT-FREE ROW PRIOR AUDIT
Images with estimated rows: 101 / 104
Median estimated peaks: 2.0
Median geometry score: 2.0019795647477143
Median per-image Crop-vs-Weed AUC: 0.869294832121717
Pooled sampled Crop-vs-Weed AUC: 0.8643869132027646

BY DATE:
            peak_count  crop_vs_weed_auc  crop_mean_distance  weed_mean_distance
date                                                                            
2023-05-25         3.0            0.8993              8.7810             61.9937
2023-05-30         2.0            0.8748             15.0785            104.6053
2023-06-06         2.0            0.8457             20.4774             86.1311
2023-06-15         2.0            0.8141             31.8304             85.0306

STATUS DISTRIBUTION:
status
ok              101
no_row_peaks      3

SAVED: /kaggle/working/

In [14]:
# ============================================================
# EXP24-C: DOES ROW GEOMETRY HELP ON MODEL ERRORS?
# TRAIN ONLY | NO TRAINING | NO VAL/TEST
# ============================================================
from pathlib import Path
from collections import defaultdict
import json
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import roc_auc_score

OUT = Path(
    "/kaggle/working/WeedsGalore_Ipr28/"
    "results/exp24/row_error_audit"
)
OUT.mkdir(parents=True, exist_ok=True)

assert len(ds) == 104
model.eval()

rng = np.random.default_rng(240)
groups = {
    "rescue_crop": defaultdict(list),
    "reject_false_crop": defaultdict(list),
}
counts = defaultdict(int)
records = []

def add_samples(group, gt, pred, distance, logits, date, name):
    """
    rescue_crop:
      Among predicted Weed pixels, identify GT Crop.
      Positive GT Crop versus negative GT Weed.

    reject_false_crop:
      Among predicted Crop pixels, identify GT Weed.
      Positive GT Weed versus negative GT Crop.
    """
    if group == "rescue_crop":
        select = pred == 2
        positive = gt == 1
        negative = gt == 2
        geom_score = -distance
        logit_score = logits[1] - logits[2]
    else:
        select = pred == 1
        positive = gt == 2
        negative = gt == 1
        geom_score = distance
        logit_score = logits[2] - logits[1]

    pos_mask = select & positive
    neg_mask = select & negative

    n_pos = int(pos_mask.sum())
    n_neg = int(neg_mask.sum())

    counts[group + "_positive"] += n_pos
    counts[group + "_negative"] += n_neg

    # Không dùng GT để dựng row prior.
    # GT chỉ dùng để xác định trường hợp đúng/sai cho audit.
    for mask, label in [(pos_mask, 1), (neg_mask, 0)]:
        indices = np.flatnonzero(mask)
        if len(indices) == 0:
            continue

        indices = rng.choice(
            indices,
            size=min(3000, len(indices)),
            replace=False,
        )

        groups[group]["y"].append(
            np.full(len(indices), label, dtype=np.uint8)
        )
        groups[group]["row"].append(
            geom_score.ravel()[indices].astype(np.float32)
        )
        groups[group]["logit"].append(
            logit_score.ravel()[indices].astype(np.float32)
        )
        groups[group]["date"].append(
            np.full(len(indices), date)
        )

for i in range(len(ds)):
    x, y = data(ds, i, DEVICE)

    with torch.inference_mode():
        logits_t = model(x).float()
        pred = logits_t.argmax(dim=1)[0].cpu().numpy()

    gt = y[0].cpu().numpy()
    logits = logits_t[0].cpu().numpy()

    distance, info = estimate_rows_from_prediction(pred)

    if distance is None:
        records.append({
            "image": str(ds.img_list[i]),
            "status": info["status"]
        })
        continue

    name = str(ds.img_list[i])
    date = name[:10]

    add_samples(
        "rescue_crop",
        gt, pred, distance, logits, date, name
    )
    add_samples(
        "reject_false_crop",
        gt, pred, distance, logits, date, name
    )

    records.append({
        "image": name,
        "date": date,
        "status": "ok",
    })

    if (i + 1) % 10 == 0 or i + 1 == len(ds):
        print(f"AUDITED {i+1}/{len(ds)}", flush=True)

def evaluate_group(group_name):
    g = groups[group_name]
    if not g["y"]:
        return None

    yy = np.concatenate(g["y"])
    rr = np.concatenate(g["row"])
    ll = np.concatenate(g["logit"])
    dd = np.concatenate(g["date"])

    if len(np.unique(yy)) < 2:
        return None

    result = {
        "n_sampled": int(len(yy)),
        "positive_count_full": counts[group_name + "_positive"],
        "negative_count_full": counts[group_name + "_negative"],
        "row_auc": float(roc_auc_score(yy, rr)),
        "logit_auc": float(roc_auc_score(yy, ll)),
        "by_date": {},
    }

    for date in sorted(np.unique(dd)):
        m = dd == date
        if len(np.unique(yy[m])) == 2:
            result["by_date"][date] = {
                "row_auc": float(roc_auc_score(yy[m], rr[m])),
                "logit_auc": float(roc_auc_score(yy[m], ll[m])),
                "n_sampled": int(m.sum()),
                "positive_sampled": int(yy[m].sum()),
            }

    return result

results = {
    "rescue_crop": evaluate_group("rescue_crop"),
    "reject_false_crop": evaluate_group("reject_false_crop"),
    "split": "train",
    "row_from_predictions_only": True,
    "gt_used_for_audit_only": True,
    "validation_accessed": False,
    "test_accessed": False,
}

print("\n" + "=" * 65)
print("EXP24 ERROR-CONDITIONAL ROW AUDIT")
print("=" * 65)

for key, title in [
    ("rescue_crop", "PREDICTED WEED: identify missed Crop"),
    ("reject_false_crop", "PREDICTED CROP: identify actual Weed"),
]:
    r = results[key]
    print("\n" + title)
    if r is None:
        print("INSUFFICIENT CLASS SUPPORT")
        continue

    print("Wrong pixels:", r["positive_count_full"])
    print("Correct comparator pixels:", r["negative_count_full"])
    print("ROW AUC:", round(r["row_auc"], 5))
    print("LOGIT AUC:", round(r["logit_auc"], 5))

    print("BY DATE:")
    for date, v in r["by_date"].items():
        print(
            date,
            "row=", round(v["row_auc"], 4),
            "logit=", round(v["logit_auc"], 4),
            "n=", v["n_sampled"],
            "wrong=", v["positive_sampled"],
        )

path = OUT / "row_error_conditional_summary.json"
path.write_text(json.dumps(results, indent=2))

print("\nSAVED:", path)
print("EXP24 ERROR AUDIT PASS — TEST SEALED")

AUDITED 10/104
AUDITED 20/104
AUDITED 30/104
AUDITED 40/104
AUDITED 50/104
AUDITED 60/104
AUDITED 70/104
AUDITED 80/104
AUDITED 90/104
AUDITED 100/104
AUDITED 104/104

EXP24 ERROR-CONDITIONAL ROW AUDIT

PREDICTED WEED: identify missed Crop
Wrong pixels: 12064
Correct comparator pixels: 1415567
ROW AUC: 0.75364
LOGIT AUC: 0.93642
BY DATE:
2023-05-25 row= 0.897 logit= 0.6305 n= 13916 wrong= 215
2023-05-30 row= 0.7691 logit= 0.8263 n= 66021 wrong= 418
2023-06-06 row= 0.7934 logit= 0.921 n= 93346 wrong= 3346
2023-06-15 row= 0.7504 logit= 0.9583 n= 32085 wrong= 8085

PREDICTED CROP: identify actual Weed
Wrong pixels: 11311
Correct comparator pixels: 828532
ROW AUC: 0.67285
LOGIT AUC: 0.96953
BY DATE:
2023-05-25 row= 0.6936 logit= 0.9496 n= 66734 wrong= 222
2023-05-30 row= 0.674 logit= 0.914 n= 88215 wrong= 622
2023-06-06 row= 0.5423 logit= 0.9728 n= 93019 wrong= 3019
2023-06-15 row= 0.5218 logit= 0.9572 n= 31448 wrong= 7448

SAVED: /kaggle/working/WeedsGalore_Ipr28/results/exp24/row_error_a

In [15]:
# ================================================================
# EXP24-E: BACKGROUND <-> VEGETATION ERROR AUDIT
# TRAIN ONLY | NO VAL/TEST | NO TRAINING
# Base: Exp15-A, 5-band SegFormer-B2
# ================================================================

from pathlib import Path
from collections import defaultdict
import sys, json
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

P = Path("/kaggle/working/WeedsGalore_Ipr28")
DS = Path(
    "/kaggle/input/datasets/thothanh28/"
    "weedsgalore-dataset/weedsgalore-dataset"
)
BASE = Path(
    "/kaggle/input/datasets/thothanh28/"
    "exp15-ab/results/exp15/"
    "E15_A_CW_LOVASZ/seed0/final_ema.pth"
)
OUT = P / "results/exp24/bg_vegetation_audit"
OUT.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(P / "src"))

from scripts.exp22_run import load_base, data
from datasets import WeedsGaloreDataset

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

assert BASE.is_file()
model = load_base(str(BASE), DEVICE)
model.eval()

ds = WeedsGaloreDataset(
    str(DS), None, 5, 3,
    False, "train", False
)
assert len(ds) == 104

CLASSES = ["Background", "Crop", "Weed"]
DATES = [
    "2023-05-25",
    "2023-05-30",
    "2023-06-06",
    "2023-06-15"
]

# Những mức chỉnh nhỏ để đo độ nhạy của dự đoán.
# Đây là oracle GT trên TRAIN, KHÔNG phải mô hình inference.
BIAS_LEVELS = [0.3, 0.6, 1.0]

cm_all = np.zeros((3, 3), dtype=np.int64)
cm_date = defaultdict(lambda: np.zeros((3, 3), dtype=np.int64))

# Mỗi hướng nhầm:
# margin = logit[predicted_class] - logit[true_class].
# Margin càng nhỏ -> lỗi càng gần ranh giới quyết định.
error_margins = defaultdict(list)
error_confidences = defaultdict(list)

# Đếm số lỗi có thể sửa bởi việc tăng logit lớp thật.
correctable_counts = defaultdict(
    lambda: {b: 0 for b in BIAS_LEVELS}
)

per_image = []

def accumulate_hist(container, key, values, limit=50000, seed=0):
    """
    Giới hạn số điểm lưu để không tốn RAM.
    Đếm confusion và correctable vẫn dùng toàn bộ pixel.
    """
    values = values.detach().float().cpu().numpy().ravel()
    if len(values) > limit:
        rng = np.random.default_rng(seed)
        values = rng.choice(values, size=limit, replace=False)
    container[key].append(values)

for i in range(len(ds)):
    x, y = data(ds, i, DEVICE)

    with torch.inference_mode():
        logits = model(x).float()
        assert logits.shape[1] == 3
        assert logits.shape[-2:] == y.shape[-2:]

        pred = logits.argmax(dim=1)
        probs = logits.softmax(dim=1)
        confidence = probs.max(dim=1).values

        gt = y.long()
        valid = (gt >= 0) & (gt < 3)

        gt_flat = gt[valid]
        pred_flat = pred[valid]

        cm = torch.bincount(
            (gt_flat * 3 + pred_flat),
            minlength=9
        ).reshape(3, 3).cpu().numpy()

        cm_all += cm

        name = str(ds.img_list[i])
        date = name[:10]
        cm_date[date] += cm

        image_record = {
            "image": name,
            "date": date,
            "valid_pixels": int(valid.sum().item()),
            "errors": int(((pred != gt) & valid).sum().item()),
            "accuracy": float(
                ((pred == gt) & valid).sum().item()
                / max(valid.sum().item(), 1)
            )
        }

        for true_cls in range(3):
            for predicted_cls in range(3):
                if true_cls == predicted_cls:
                    continue

                key = f"{CLASSES[true_cls]}->{CLASSES[predicted_cls]}"

                mask = (
                    valid &
                    (gt == true_cls) &
                    (pred == predicted_cls)
                )

                n = int(mask.sum().item())
                image_record[key] = n

                if n == 0:
                    continue

                # Logit gap between predicted class and GT class
                margin = (
                    logits[:, predicted_cls]
                    - logits[:, true_cls]
                )[mask]

                accumulate_hist(
                    error_margins, key, margin,
                    seed=10000 + i * 10 + true_cls * 3 + predicted_cls
                )

                accumulate_hist(
                    error_confidences, key, confidence[mask],
                    seed=20000 + i * 10 + true_cls * 3 + predicted_cls
                )

                # Bất kể các lớp khác: tăng đúng logit GT thêm bias.
                # Chỉ là oracle khả năng sửa dưới từng bias.
                for bias in BIAS_LEVELS:
                    adjusted_gt_logit = (
                        logits.gather(
                            1, gt.clamp(0, 2).unsqueeze(1)
                        ).squeeze(1)
                        + bias
                    )

                    # Tăng lớp thật vượt qua mọi lớp khác?
                    max_other = logits.clone()
                    max_other.scatter_(
                        1,
                        gt.clamp(0, 2).unsqueeze(1),
                        float("-inf")
                    )
                    max_other = max_other.max(dim=1).values

                    fixable = (
                        mask &
                        (adjusted_gt_logit > max_other)
                    )

                    correctable_counts[key][bias] += int(
                        fixable.sum().item()
                    )

    per_image.append(image_record)

    if (i + 1) % 10 == 0 or i + 1 == len(ds):
        print(f"AUDITED {i+1:3d}/{len(ds)}", flush=True)

# ================================================================
# SUMMARY
# ================================================================

error_rows = []

for true_cls in range(3):
    for predicted_cls in range(3):
        if true_cls == predicted_cls:
            continue

        key = f"{CLASSES[true_cls]}->{CLASSES[predicted_cls]}"
        total = int(cm_all[true_cls, predicted_cls])

        margins = (
            np.concatenate(error_margins[key])
            if error_margins[key] else np.array([])
        )
        confs = (
            np.concatenate(error_confidences[key])
            if error_confidences[key] else np.array([])
        )

        item = {
            "direction": key,
            "errors": total,
            "fraction_of_all_errors": float(
                total / max(cm_all.sum() - np.trace(cm_all), 1)
            ),
            "median_margin": (
                float(np.median(margins))
                if margins.size else None
            ),
            "median_confidence": (
                float(np.median(confs))
                if confs.size else None
            )
        }

        for bias in BIAS_LEVELS:
            item[f"fixable_plus_{bias}"] = int(
                correctable_counts[key][bias]
            )
            item[f"fixable_pct_{bias}"] = float(
                100 * correctable_counts[key][bias]
                / max(total, 1)
            )

        error_rows.append(item)

df = pd.DataFrame(error_rows)
per_df = pd.DataFrame(per_image)

# ================================================================
# MIOU FROM CONFUSION
# ================================================================

def metrics_from_cm(cm):
    cm = cm.astype(np.float64)
    tp = np.diag(cm)
    denom = cm.sum(axis=1) + cm.sum(axis=0) - tp
    iou = np.divide(
        tp, denom,
        out=np.zeros_like(tp),
        where=denom > 0
    )
    return {
        "mIoU": float(iou.mean() * 100),
        "IoU_bg": float(iou[0] * 100),
        "IoU_crop": float(iou[1] * 100),
        "IoU_weed": float(iou[2] * 100)
    }

metric_all = metrics_from_cm(cm_all)

print("\n" + "=" * 72)
print("EXP24-E: BACKGROUND <-> VEGETATION ERROR AUDIT")
print("=" * 72)
print("\nTRAIN mIoU:", round(metric_all["mIoU"], 5))

print("\nCONFUSION MATRIX (rows=GT, columns=prediction):")
print(pd.DataFrame(cm_all, index=CLASSES, columns=CLASSES).to_string())

print("\nERROR DIRECTIONS:")
cols = [
    "direction", "errors", "fraction_of_all_errors",
    "median_margin", "median_confidence",
    "fixable_pct_0.3", "fixable_pct_0.6", "fixable_pct_1.0"
]
print(df[cols].round(4).to_string(index=False))

bgveg = int(
    cm_all[0, 1] + cm_all[0, 2]
    + cm_all[1, 0] + cm_all[2, 0]
)
cw = int(cm_all[1, 2] + cm_all[2, 1])
total_errors = int(cm_all.sum() - np.trace(cm_all))

print("\nERROR GROUP SUMMARY")
print("BG <-> VEGETATION errors:", bgveg)
print("CROP <-> WEED errors:", cw)
print("TOTAL errors:", total_errors)
print("BG <-> VEGETATION share (%):",
      round(100 * bgveg / max(total_errors, 1), 3))

print("\nMETRICS BY DATE:")
for date in DATES:
    cm = cm_date[date]
    print(date, metrics_from_cm(cm),
          "errors:", int(cm.sum() - np.trace(cm)))

# ================================================================
# SAVE REPORTS
# ================================================================

df.to_csv(OUT / "error_directions.csv", index=False)
per_df.to_csv(OUT / "per_image_errors.csv", index=False)

summary = {
    "experiment": "EXP24_BG_VEGETATION_AUDIT",
    "split": "train",
    "n_images": len(ds),
    "base_ckpt": str(BASE),
    "metrics": metric_all,
    "confusion_matrix": cm_all.tolist(),
    "error_directions": error_rows,
    "bg_vegetation_errors": bgveg,
    "crop_weed_errors": cw,
    "total_errors": total_errors,
    "bg_vegetation_share_pct": (
        100 * bgveg / max(total_errors, 1)
    ),
    "by_date": {
        date: {
            "metrics": metrics_from_cm(cm_date[date]),
            "confusion_matrix": cm_date[date].tolist()
        }
        for date in DATES
    },
    "GT_used_only_for_metrics_and_oracle": True,
    "validation_accessed": False,
    "test_accessed": False,
}

(OUT / "bg_vegetation_summary.json").write_text(
    json.dumps(summary, indent=2),
    encoding="utf-8"
)

print("\nSAVED:", OUT)
print("EXP24 BG-VEGETATION AUDIT PASS — TEST SEALED")

AUDITED  10/104
AUDITED  20/104
AUDITED  30/104
AUDITED  40/104
AUDITED  50/104
AUDITED  60/104
AUDITED  70/104
AUDITED  80/104
AUDITED  90/104
AUDITED 100/104
AUDITED 104/104

EXP24-E: BACKGROUND <-> VEGETATION ERROR AUDIT

TRAIN mIoU: 87.61103

CONFUSION MATRIX (rows=GT, columns=prediction):
            Background    Crop     Weed
Background    34671291   86616   193025
Crop             61590  841686    12183
Weed            119470   11485  1442654

ERROR DIRECTIONS:
       direction  errors  fraction_of_all_errors  median_margin  median_confidence  fixable_pct_0.3  fixable_pct_0.6  fixable_pct_1.0
Background->Crop   86616                  0.1788         1.7250             0.8308          10.4172          20.0679          31.9514
Background->Weed  193025                  0.3985         1.6007             0.8224          11.2467          21.5770          33.9852
Crop->Background   61590                  0.1272         1.5412             0.8074          11.5912          22.2845        

In [17]:
import shutil

src = "/kaggle/working/WeedsGalore_Ipr28/results/exp23"
dst = "/kaggle/working/exp23.zip"

shutil.make_archive(
    dst.replace(".zip", ""),
    "zip",
    root_dir=src
)

print("ZIP CREATED:", dst)

ZIP CREATED: /kaggle/working/exp23.zip


In [7]:
import subprocess, sys

subprocess.run(
    [sys.executable, "-m", "pytest", "-q",
     "tests/test_exp22_bandit.py"],
    cwd="/kaggle/working/WeedsGalore_Ipr28",
    check=True
)

print("EXP22 PREFLIGHT PASS")

....                                                                     [100%]
4 passed in 2.88s
EXP22 PREFLIGHT PASS


In [8]:
# EXP22 — RL AGENT TRAIN (EXP15-A B2 CHECKPOINT)

from pathlib import Path
import subprocess
import sys

P = Path("/kaggle/working/WeedsGalore_Ipr28")

DS = Path(
    "/kaggle/input/datasets/thothanh28/"
    "weedsgalore-dataset/weedsgalore-dataset"
)

# Ưu tiên Exp15-A: B2 + Crop/Weed Lovasz
EXP15_A = Path(
    "/kaggle/input/datasets/thothanh28/"
    "exp15-ab/results/exp15/"
    "E15_A_CW_LOVASZ/seed0/final_ema.pth"
)

# Dự phòng Exp12: B2 + Raw LLR
EXP12_B2 = Path(
    "/kaggle/input/datasets/thanhthao28/"
    "exp12-backup/exp12/"
    "P_B2_LLR/seed0/final_ema.pth"
)

assert P.is_dir(), f"Project không tồn tại: {P}"
assert DS.is_dir(), f"Dataset không tồn tại: {DS}"

SCRIPT = P / "src/scripts/exp22_run.py"
assert SCRIPT.is_file(), f"Thiếu script: {SCRIPT}"

# Chỉ chọn B2 phù hợp
if EXP15_A.is_file():
    BASE = EXP15_A
    BASE_NAME = "EXP15_A_B2"
elif EXP12_B2.is_file():
    BASE = EXP12_B2
    BASE_NAME = "EXP12_B2_LLR"
else:
    raise FileNotFoundError(
        "Không tìm thấy checkpoint Exp15-A hoặc Exp12 B2."
    )

print("=" * 65)
print("EXP22 REINFORCEMENT LEARNING")
print("=" * 65)
print("BASE MODEL:", BASE_NAME)
print("CHECKPOINT:", BASE)
print("DATASET:", DS)
print("SCRIPT:", SCRIPT)

OUT = P / "results/exp22/E22_B2_RL/seed0"
OUT.mkdir(parents=True, exist_ok=True)

common = [
    sys.executable,
    str(SCRIPT),
    f"--dataset_path={DS}",
    f"--base_ckpt={BASE}",
    f"--out_dir={OUT}",
    "--seed=0",
]

# ============================================================
# STAGE 1 — Cache feature, logits và reward trên TRAIN
# ============================================================
print("\n" + "=" * 65)
print("STAGE 1/2: CACHE TRAIN")
print("=" * 65)

subprocess.run(
    common + ["--mode=cache_train"],
    cwd=str(P),
    check=True,
)

# ============================================================
# STAGE 2 — Huấn luyện RL policy
# ============================================================
print("\n" + "=" * 65)
print("STAGE 2/2: TRAIN RL AGENT")
print("=" * 65)

subprocess.run(
    common + [
        "--mode=fit",
        "--epochs=80",
    ],
    cwd=str(P),
    check=True,
)

print("\n" + "=" * 65)
print("EXP22 RL TRAIN PASS")
print("BASE MODEL:", BASE_NAME)
print("RESULTS:", OUT)
print("=" * 65)

EXP22 REINFORCEMENT LEARNING
BASE MODEL: EXP15_A_B2
CHECKPOINT: /kaggle/input/datasets/thothanh28/exp15-ab/results/exp15/E15_A_CW_LOVASZ/seed0/final_ema.pth
DATASET: /kaggle/input/datasets/thothanh28/weedsgalore-dataset/weedsgalore-dataset
SCRIPT: /kaggle/working/WeedsGalore_Ipr28/src/scripts/exp22_run.py

STAGE 1/2: CACHE TRAIN
cached 1 / 104
cached 11 / 104
cached 21 / 104
cached 31 / 104
cached 41 / 104
cached 51 / 104
cached 61 / 104
cached 71 / 104
cached 81 / 104
cached 91 / 104
cached 101 / 104
EXP22 CACHE PASS 104

STAGE 2/2: TRAIN RL AGENT
policy epoch 1/80: holdout reward=0.00000 best=0.00000
policy epoch 11/80: holdout reward=0.00000 best=0.00000
policy epoch 21/80: holdout reward=0.00000 best=0.00000
policy epoch 31/80: holdout reward=0.00000 best=0.00000
policy epoch 41/80: holdout reward=0.00000 best=0.00000
policy epoch 51/80: holdout reward=0.00000 best=0.00000
policy epoch 61/80: holdout reward=0.00000 best=0.00000
policy epoch 71/80: holdout reward=0.00000 best=0.0000

In [9]:
#Kiểm tra reward
import json
from pathlib import Path

OUT = Path(
    "/kaggle/working/WeedsGalore_Ipr28/"
    "results/exp22/E22_B2_RL/seed0"
)

history_path = OUT / "policy_history.json"
assert history_path.is_file(), "Chưa tìm thấy policy_history.json"

history = json.loads(history_path.read_text())

best = max(
    history,
    key=lambda x: x["policy_holdout_mean_reward"]
)

print("BEST EPOCH:", best["epoch"])
print("HOLDOUT REWARD:", best["policy_holdout_mean_reward"])
print("KEEP FRACTION:", best["policy_holdout_keep_fraction"])

BEST EPOCH: 0
HOLDOUT REWARD: 0.0
KEEP FRACTION: 1.0


In [10]:
import json
from pathlib import Path
import numpy as np

OUT = Path(
    "/kaggle/working/WeedsGalore_Ipr28/"
    "results/exp22/E22_B2_RL/seed0"
)

history = json.loads(
    (OUT / "policy_history.json").read_text()
)

print("TOTAL EPOCHS:", len(history))
print("\nFIRST EPOCH:")
print(json.dumps(history[0], indent=2))

print("\nLAST EPOCH:")
print(json.dumps(history[-1], indent=2))

print("\n=== REWARD DIAGNOSTICS ===")

rewards = np.array([
    x["policy_holdout_mean_reward"]
    for x in history
], dtype=float)

keeps = np.array([
    x["policy_holdout_keep_fraction"]
    for x in history
], dtype=float)

print("Maximum reward:", rewards.max())
print("Minimum reward:", rewards.min())
print("Mean reward:", rewards.mean())
print("Minimum keep fraction:", keeps.min())
print("Final keep fraction:", keeps[-1])

print("\nEpochs with positive reward:",
      int((rewards > 0).sum()))

print("Epochs with non-KEEP actions:",
      int((keeps < 1).sum()))

TOTAL EPOCHS: 80

FIRST EPOCH:
{
  "epoch": 0,
  "policy_holdout_mean_reward": 0.0,
  "policy_holdout_keep_fraction": 1.0,
  "loss": -0.0008786660270873524
}

LAST EPOCH:
{
  "epoch": 79,
  "policy_holdout_mean_reward": 0.0,
  "policy_holdout_keep_fraction": 1.0,
  "loss": -5.513306668295367e-05
}

=== REWARD DIAGNOSTICS ===
Maximum reward: 0.0
Minimum reward: 0.0
Mean reward: 0.0
Minimum keep fraction: 1.0
Final keep fraction: 1.0

Epochs with positive reward: 0
Epochs with non-KEEP actions: 0


In [11]:
#Cell chẩn đoán Oracle Action — không train lại
from pathlib import Path
import torch
import numpy as np
import hashlib

OUT = Path(
    "/kaggle/working/WeedsGalore_Ipr28/"
    "results/exp22/E22_B2_RL/seed0"
)

CACHE = OUT / "train_cache"
paths = sorted(CACHE.glob("*.pt"))
assert len(paths) == 104, f"Expected 104, found {len(paths)}"

records = []
for p in paths:
    d = torch.load(p, map_location="cpu", weights_only=False)
    r = d["rewards"].float()

    assert r.ndim == 2 and r.shape[1] == 5
    assert torch.isfinite(r).all()

    records.append((d["name"], r))

# Giống nguyên tắc split_indices trong exp22_run.py:
# 20 ảnh dành cho policy holdout
order = sorted(
    records,
    key=lambda z: hashlib.sha256(
        str(z[0]).encode()
    ).hexdigest()
)

groups = {
    "FIT_84": order[20:],
    "HOLDOUT_20": order[:20],
    "ALL_104": order
}

action_names = [
    "KEEP",
    "CROP +0.6",
    "CROP -0.6",
    "WEED +0.6",
    "WEED -0.6"
]

for group_name, group in groups.items():
    rewards = torch.cat([r for _, r in group], dim=0)

    oracle_reward, oracle_actions = rewards.max(dim=1)
    best_edit_reward = rewards[:, 1:].max(dim=1).values

    print("\n" + "=" * 60)
    print(group_name)
    print("=" * 60)

    print("Total regions:", len(rewards))
    print("KEEP reward:", rewards[:, 0].mean().item())

    print("ORACLE mean reward:",
          round(oracle_reward.mean().item(), 6))

    print("ORACLE edit fraction:",
          round((oracle_actions != 0).float().mean().item(), 4))

    print("Regions with profitable edit:",
          round((best_edit_reward > 0).float().mean().item(), 4))

    print("Regions with any positive edit:",
          int((best_edit_reward > 0).sum()))

    print("\nREWARD BY ACTION:")
    for i, name in enumerate(action_names):
        a = rewards[:, i]
        print(
            f"{name:12s} "
            f"mean={a.mean().item():+.6f} "
            f"positive={(a > 0).float().mean().item():.4f} "
            f"max={a.max().item():+.5f}"
        )

print("\nEXP22 ORACLE AUDIT PASS")


FIT_84
Total regions: 52500
KEEP reward: 0.0
ORACLE mean reward: 6.6e-05
ORACLE edit fraction: 0.0078
Regions with profitable edit: 0.0078
Regions with any positive edit: 407

REWARD BY ACTION:
KEEP         mean=+0.000000 positive=0.0000 max=+0.00000
CROP +0.6    mean=-0.015522 positive=0.0006 max=+0.04576
CROP -0.6    mean=-0.015231 positive=0.0013 max=+0.02927
WEED +0.6    mean=-0.016211 positive=0.0012 max=+0.03708
WEED -0.6    mean=-0.015338 positive=0.0050 max=+0.13431

HOLDOUT_20
Total regions: 12500
KEEP reward: 0.0
ORACLE mean reward: 2.4e-05
ORACLE edit fraction: 0.0042
Regions with profitable edit: 0.0042
Regions with any positive edit: 52

REWARD BY ACTION:
KEEP         mean=+0.000000 positive=0.0000 max=+0.00000
CROP +0.6    mean=-0.015372 positive=0.0002 max=+0.01712
CROP -0.6    mean=-0.015203 positive=0.0003 max=+0.00410
WEED +0.6    mean=-0.016304 positive=0.0004 max=+0.01365
WEED -0.6    mean=-0.015462 positive=0.0034 max=+0.03708

ALL_104
Total regions: 65000
KEEP re

In [12]:
#Exp22 Pixel-wise Oracle Audit
# ============================================================
# EXP22 -> EXP23: PIXEL-WISE ORACLE AUDIT
# Base: Exp15-A B2 + Raw LLR
# TRAIN ONLY (104 images) — NO VAL / NO TEST / NO TRAINING
# ============================================================

from pathlib import Path
import sys, json, csv, hashlib
import numpy as np
import torch

P = Path("/kaggle/working/WeedsGalore_Ipr28")
DS = (
    "/kaggle/input/datasets/thothanh28/"
    "weedsgalore-dataset/weedsgalore-dataset"
)
BASE = Path(
    "/kaggle/input/datasets/thothanh28/"
    "exp15-ab/results/exp15/"
    "E15_A_CW_LOVASZ/seed0/final_ema.pth"
)

OUT = P / "results/exp22/pixel_oracle_audit"
OUT.mkdir(parents=True, exist_ok=True)

assert BASE.is_file(), f"Missing checkpoint: {BASE}"
assert (P / "src/scripts/exp22_run.py").is_file()

sys.path.insert(0, str(P / "src"))

# Dùng đúng loader Exp22 đã chạy thành công trên Kaggle
from scripts.exp22_run import load_base, data, confusion, miou
from datasets import WeedsGaloreDataset

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DEVICE:", DEVICE)
print("BASE:", BASE)

model = load_base(str(BASE), DEVICE)

ds = WeedsGaloreDataset(
    DS, None, 5, 3,
    False, "train", False
)
assert len(ds) == 104

# Đúng 5 hành động của Exp22 nhưng áp dụng theo từng pixel.
ACTIONS = [
    ("KEEP", None, 0.0),
    ("CROP_PLUS", 1, +0.6),
    ("CROP_MINUS", 1, -0.6),
    ("WEED_PLUS", 2, +0.6),
    ("WEED_MINUS", 2, -0.6),
]

DAMAGE = 1.5
ACTION_COST = 0.015

base_cm = torch.zeros((3, 3), dtype=torch.float64)
oracle_cm = torch.zeros((3, 3), dtype=torch.float64)

total_pixels = 0
total_errors = 0
total_correctable = 0
total_oracle_edits = 0
total_oracle_corrected = 0
total_oracle_damaged = 0

action_counts = torch.zeros(5, dtype=torch.long)
rows = []

for i in range(len(ds)):
    x, y = data(ds, i, DEVICE)

    with torch.inference_mode():
        logits = model(x).float()

        assert logits.shape[1] == 3
        assert logits.shape[-2:] == y.shape[-2:]

        base_pred = logits.argmax(dim=1)

        valid = (y >= 0) & (y < 3)
        base_correct = (base_pred == y) & valid
        base_error = (base_pred != y) & valid

        # KEEP có reward = 0.
        best_reward = torch.zeros_like(
            base_pred, dtype=torch.float32
        )
        best_action = torch.zeros_like(
            base_pred, dtype=torch.long
        )
        best_pred = base_pred.clone()

        # Có tồn tại hành động nào sửa đúng pixel sai không?
        correctable = torch.zeros_like(
            base_pred, dtype=torch.bool
        )

        for action_id, (name, channel, bias) in enumerate(ACTIONS):
            if action_id == 0:
                continue

            # Thay đổi một logit như Exp22.
            changed = logits.clone()
            changed[:, channel] += bias

            candidate_pred = changed.argmax(dim=1)

            corrected = (
                base_error &
                (candidate_pred == y)
            )

            damaged = (
                base_correct &
                (candidate_pred != y)
            )

            correctable |= corrected

            # Reward pixel-wise:
            # +1 nếu sửa đúng
            # -1.5 nếu làm hỏng pixel đúng
            # -0.015 nếu can thiệp
            reward = (
                corrected.float()
                - DAMAGE * damaged.float()
                - ACTION_COST
            )

            # Không chọn hành động cho pixel không hợp lệ.
            reward = torch.where(
                valid,
                reward,
                torch.full_like(reward, -1e9)
            )

            better = reward > best_reward

            best_reward = torch.where(
                better, reward, best_reward
            )
            best_action = torch.where(
                better, action_id, best_action
            )
            best_pred = torch.where(
                better, candidate_pred, best_pred
            )

        base_cm += confusion(base_pred, y)
        oracle_cm += confusion(best_pred, y)

        n_valid = int(valid.sum().item())
        n_error = int(base_error.sum().item())
        n_correctable = int(correctable.sum().item())
        n_edits = int(((best_action != 0) & valid).sum().item())

        n_corrected = int(
            (base_error & (best_pred == y)).sum().item()
        )
        n_damaged = int(
            (base_correct & (best_pred != y)).sum().item()
        )

        total_pixels += n_valid
        total_errors += n_error
        total_correctable += n_correctable
        total_oracle_edits += n_edits
        total_oracle_corrected += n_corrected
        total_oracle_damaged += n_damaged

        for a in range(5):
            action_counts[a] += int(
                ((best_action == a) & valid).sum().item()
            )

        rows.append({
            "image": str(ds.img_list[i]),
            "valid_pixels": n_valid,
            "base_errors": n_error,
            "correctable_errors": n_correctable,
            "oracle_edits": n_edits,
            "oracle_corrected": n_corrected,
            "oracle_damaged": n_damaged,
            "correctable_error_pct": (
                100 * n_correctable / max(n_error, 1)
            ),
        })

    if (i + 1) % 10 == 0 or i + 1 == len(ds):
        print(
            f"Audited {i+1:3d}/{len(ds)} "
            f"| correctable errors: {total_correctable:,}",
            flush=True
        )

baseline = miou(base_cm)
oracle = miou(oracle_cm)

summary = {
    "experiment": "EXP22_PIXEL_ORACLE",
    "split": "train",
    "n_images": len(ds),
    "checkpoint": str(BASE),
    "checkpoint_sha256": hashlib.sha256(
        BASE.read_bytes()
    ).hexdigest(),
    "seed": 0,
    "tta": "none",
    "action_bias": 0.6,
    "damage": DAMAGE,
    "action_cost": ACTION_COST,
    "total_valid_pixels": total_pixels,
    "total_base_errors": total_errors,
    "total_correctable_errors": total_correctable,
    "correctable_error_pct": (
        100 * total_correctable / max(total_errors, 1)
    ),
    "oracle_edits": total_oracle_edits,
    "oracle_corrected": total_oracle_corrected,
    "oracle_damaged": total_oracle_damaged,
    "oracle_edit_pct": (
        100 * total_oracle_edits / max(total_pixels, 1)
    ),
    "base": baseline,
    "pixel_oracle": oracle,
    "oracle_miou_gain_pp": (
        oracle["mIoU"] - baseline["mIoU"]
    ),
    "action_counts": {
        ACTIONS[a][0]: int(action_counts[a])
        for a in range(5)
    },
    "oracle_uses_train_ground_truth": True,
    "deployable_without_ground_truth": False,
    "validation_accessed": False,
    "test_accessed": False,
}

json_path = OUT / "pixel_oracle_summary.json"
csv_path = OUT / "pixel_oracle_per_image.csv"

json_path.write_text(
    json.dumps(summary, indent=2),
    encoding="utf-8"
)

with csv_path.open("w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=list(rows[0]))
    writer.writeheader()
    writer.writerows(rows)

print("\n" + "=" * 65)
print("EXP22 PIXEL-WISE ORACLE AUDIT")
print("=" * 65)

print("BASELINE mIoU:", round(baseline["mIoU"], 4))
print("ORACLE mIoU:", round(oracle["mIoU"], 4))
print("ORACLE GAIN (pp):", round(
    summary["oracle_miou_gain_pp"], 4
))

print("\nERROR ANALYSIS")
print("Total baseline errors:", total_errors)
print("Correctable errors:", total_correctable)
print("Correctable error %:", round(
    summary["correctable_error_pct"], 3
))
print("Oracle edits:", total_oracle_edits)
print("Oracle corrected:", total_oracle_corrected)
print("Oracle damaged:", total_oracle_damaged)

print("\nACTION COUNTS")
for name, count in summary["action_counts"].items():
    print(f"{name:14s}: {count:,}")

print("\nSAVED:")
print(json_path)
print(csv_path)

print("\nTRAIN ONLY. VAL / TEST NOT ACCESSED.")
print("PIXEL ORACLE AUDIT PASS")

DEVICE: cuda
BASE: /kaggle/input/datasets/thothanh28/exp15-ab/results/exp15/E15_A_CW_LOVASZ/seed0/final_ema.pth
Audited  10/104 | correctable errors: 36,988
Audited  20/104 | correctable errors: 50,441
Audited  30/104 | correctable errors: 64,733
Audited  40/104 | correctable errors: 79,206
Audited  50/104 | correctable errors: 84,266
Audited  60/104 | correctable errors: 88,838
Audited  70/104 | correctable errors: 94,419
Audited  80/104 | correctable errors: 97,591
Audited  90/104 | correctable errors: 99,803
Audited 100/104 | correctable errors: 102,336
Audited 104/104 | correctable errors: 105,859

EXP22 PIXEL-WISE ORACLE AUDIT
BASELINE mIoU: 87.611
ORACLE mIoU: 90.0519
ORACLE GAIN (pp): 2.4409

ERROR ANALYSIS
Total baseline errors: 484369
Correctable errors: 105859
Correctable error %: 21.855
Oracle edits: 105859
Oracle corrected: 105859
Oracle damaged: 0

ACTION COUNTS
KEEP          : 37,334,141
CROP_PLUS     : 15,750
CROP_MINUS    : 19,177
WEED_PLUS     : 29,419
WEED_MINUS    : 

In [13]:
from pathlib import Path
import zipfile

P = Path("/kaggle/working/WeedsGalore_Ipr28")
OUT = Path("/kaggle/working/EXP22_TO_EXP23_BACKUP.zip")

sources = [
    P / "results/exp22/pixel_oracle_audit",
    P / "results/exp22/E22_B2_RL/seed0",
    P / "src/nets/exp22_bandit.py",
    P / "src/scripts/exp22_run.py",
]

with zipfile.ZipFile(
    OUT, "w", compression=zipfile.ZIP_DEFLATED
) as z:
    for src in sources:
        if src.is_file():
            z.write(src, src.relative_to(P))
        elif src.is_dir():
            for f in src.rglob("*"):
                if f.is_file():
                    z.write(f, f.relative_to(P))

with zipfile.ZipFile(OUT) as z:
    assert z.testzip() is None
    print("BACKUP PASS:", len(z.namelist()), "files")

print("DOWNLOAD:", OUT)
print("SIZE MB:", round(OUT.stat().st_size / 1024**2, 2))

BACKUP PASS: 111 files
DOWNLOAD: /kaggle/working/EXP22_TO_EXP23_BACKUP.zip
SIZE MB: 3.31


In [14]:
from pathlib import Path
from collections import Counter
import json

DS = Path(
    "/kaggle/input/datasets/thothanh28/"
    "weedsgalore-dataset/weedsgalore-dataset"
)
P = Path("/kaggle/working/WeedsGalore_Ipr28")

split_file = DS / "splits/train.txt"
assert split_file.is_file(), split_file

entries = [
    s.strip()
    for s in split_file.read_text().splitlines()
    if s.strip()
]

print("TRAIN ENTRIES:", len(entries))
assert len(entries) == 104

dates = ["2023-05-25", "2023-05-30",
         "2023-06-06", "2023-06-15"]

records = []
for entry in entries:
    found = [d for d in dates if d in entry]
    if len(found) != 1:
        raise ValueError(f"Không xác định được ngày: {entry}")
    records.append({
        "date": found[0],
        "filename": Path(entry).name
    })

print("\n=== TRAIN IMAGES PER DATE ===")
counts = Counter(r["date"] for r in records)
for d in dates:
    print(d, counts[d])

print("\n=== EXAMPLE FILENAMES ===")
for d in dates:
    print("\n", d)
    for r in [x for x in records if x["date"] == d][:4]:
        print(" ", r["filename"])

print("\n=== TRAIN METADATA DIRECTORIES ===")
for d in dates:
    folder = DS / d
    print(d, "exists:", folder.is_dir())
    if folder.is_dir():
        log_dir = folder / "logs"
        print(" log files:",
              [p.name for p in log_dir.iterdir()][:10]
              if log_dir.is_dir() else "NONE")

report = {
    "train_count": len(records),
    "date_counts": dict(counts),
    "train_records": records,
    "test_opened": False,
    "validation_opened": False,
    "temporal_pairing_verified": False,
    "registration_verified": False
}

out = P / "results/exp23/temporal_data_audit.json"
out.parent.mkdir(parents=True, exist_ok=True)
out.write_text(json.dumps(report, indent=2))

print("\nTEMPORAL METADATA AUDIT PASS")
print("SAVED:", out)

TRAIN ENTRIES: 104

=== TRAIN IMAGES PER DATE ===
2023-05-25 32
2023-05-30 32
2023-06-06 32
2023-06-15 8

=== EXAMPLE FILENAMES ===

 2023-05-25
  2023-05-25_0150
  2023-05-25_0486
  2023-05-25_0776
  2023-05-25_0158

 2023-05-30
  2023-05-30_0078
  2023-05-30_0432
  2023-05-30_0760
  2023-05-30_0086

 2023-06-06
  2023-06-06_0197
  2023-06-06_0473
  2023-06-06_0760
  2023-06-06_0188

 2023-06-15
  2023-06-15_0472
  2023-06-15_0763
  2023-06-15_0157
  2023-06-15_1056

=== TRAIN METADATA DIRECTORIES ===
2023-05-25 exists: True
 log files: NONE
2023-05-30 exists: True
 log files: NONE
2023-06-06 exists: True
 log files: NONE
2023-06-15 exists: True
 log files: NONE

TEMPORAL METADATA AUDIT PASS
SAVED: /kaggle/working/WeedsGalore_Ipr28/results/exp23/temporal_data_audit.json


In [15]:
from pathlib import Path
from collections import defaultdict, Counter
import json

P = Path("/kaggle/working/WeedsGalore_Ipr28")
report = json.loads(
    (P / "results/exp23/temporal_data_audit.json").read_text()
)

groups = defaultdict(list)

for r in report["train_records"]:
    name = r["filename"]
    date, suffix = name.rsplit("_", 1)
    groups[suffix].append((date, name))

repeated = {
    k: sorted(v)
    for k, v in groups.items()
    if len(set(d for d, _ in v)) >= 2
}

print("TOTAL UNIQUE SUFFIXES:", len(groups))
print("SUFFIXES REPEATED ACROSS DATES:", len(repeated))

print("\nREPEAT DISTRIBUTION:")
print(dict(sorted(Counter(
    len(set(d for d, _ in v))
    for v in groups.values()
).items())))

print("\nEXAMPLES:")
for suffix, items in list(sorted(repeated.items()))[:20]:
    print(suffix, "=>", [x[1] for x in items])

# Lưu kết quả để chuẩn bị Exp23
out = P / "results/exp23/temporal_suffix_audit.json"
out.write_text(json.dumps({
    "n_unique_suffixes": len(groups),
    "n_repeated_suffixes": len(repeated),
    "repeated": repeated,
    "same_suffix_proves_same_location": False,
    "registration_verified": False,
    "split": "train",
    "test_accessed": False,
}, indent=2))

print("\nSAVED:", out)
print("EXP23 SUFFIX AUDIT PASS")

TOTAL UNIQUE SUFFIXES: 98
SUFFIXES REPEATED ACROSS DATES: 6

REPEAT DISTRIBUTION:
{1: 92, 2: 6}

EXAMPLES:
0147 => ['2023-05-25_0147', '2023-05-30_0147']
0150 => ['2023-05-25_0150', '2023-05-30_0150']
0454 => ['2023-05-30_0454', '2023-06-15_0454']
0728 => ['2023-06-06_0728', '2023-06-15_0728']
0760 => ['2023-05-30_0760', '2023-06-06_0760']
0945 => ['2023-05-30_0945', '2023-06-06_0945']

SAVED: /kaggle/working/WeedsGalore_Ipr28/results/exp23/temporal_suffix_audit.json
EXP23 SUFFIX AUDIT PASS


In [8]:
#Preflight và kiểm tra nhãn gốc
from pathlib import Path
import subprocess
import sys

P = Path("/kaggle/working/WeedsGalore_Ipr28")

DS = (
    "/kaggle/input/datasets/thothanh28/"
    "weedsgalore-dataset/weedsgalore-dataset"
)

required = [
    "src/nets/exp21_hier.py",
    "src/train_exp21.py",
    "src/evaluate_exp21.py",
    "src/scripts/exp21_label_audit.py",
    "src/scripts/exp21_arms.py",
    "src/scripts/exp21_seed0.py",
    "tests/test_exp21_hier.py",
]

for rel in required:
    print("PASS" if (P / rel).is_file() else "MISSING", rel)

assert all((P / rel).is_file() for rel in required)

commands = [
    [
        "src/scripts/exp21_label_audit.py",
        f"--dataset_path={DS}",
        f"--project={P}",
    ],
    ["src/scripts/exp21_arms.py"],
    ["-m", "pytest", "-q", "tests/test_exp21_hier.py"],
    [
        "-m", "py_compile",
        "src/train_exp21.py",
        "src/evaluate_exp21.py",
        "src/nets/exp21_hier.py",
    ],
]

for cmd in commands:
    subprocess.run(
        [sys.executable, *cmd],
        cwd=P,
        check=True,
    )

print("\nEXP21 PREFLIGHT PASS")

PASS src/nets/exp21_hier.py
PASS src/train_exp21.py
PASS src/evaluate_exp21.py
PASS src/scripts/exp21_label_audit.py
PASS src/scripts/exp21_arms.py
PASS src/scripts/exp21_seed0.py
PASS tests/test_exp21_hier.py
EXP21 LABEL AUDIT PASS: 104 train images, fine IDs 0..5
PIXEL COUNTS: {0: 34950932, 1: 915459, 2: 1016336, 3: 336285, 4: 67903, 5: 153085}
TEST SEALED; label IDs are not species names until confirmed in official documentation
EXP21 ARMS PASS: /kaggle/working/WeedsGalore_Ipr28/configs/exp21/exp21_arms.json
...                                                                      [100%]
3 passed in 3.15s

EXP21 PREFLIGHT PASS


In [9]:
#Train Exp21 seed 0 + D4
from pathlib import Path
import subprocess
import sys

P = Path("/kaggle/working/WeedsGalore_Ipr28")

DS = (
    "/kaggle/input/datasets/thothanh28/"
    "weedsgalore-dataset/weedsgalore-dataset"
)

subprocess.run(
    [
        sys.executable,
        "src/scripts/exp21_seed0.py",
        f"--dataset_path={DS}",
    ],
    cwd=P,
    check=True,
)

print("EXP21 SEED0 COMPLETED")

RUN: /usr/bin/python3 src/scripts/exp21_label_audit.py --dataset_path=/kaggle/input/datasets/thothanh28/weedsgalore-dataset/weedsgalore-dataset --project=/kaggle/working/WeedsGalore_Ipr28
EXP21 LABEL AUDIT PASS: 104 train images, fine IDs 0..5
PIXEL COUNTS: {0: 34950932, 1: 915459, 2: 1016336, 3: 336285, 4: 67903, 5: 153085}
TEST SEALED; label IDs are not species names until confirmed in official documentation
RUN: /usr/bin/python3 src/scripts/exp21_arms.py
EXP21 ARMS PASS: /kaggle/working/WeedsGalore_Ipr28/configs/exp21/exp21_arms.json
RUN: /usr/bin/python3 src/train_exp21.py --dataset_path=/kaggle/input/datasets/thothanh28/weedsgalore-dataset/weedsgalore-dataset --seed=0 --run_name=E21_B2_HIER_s0 --out_dir=/kaggle/working/WeedsGalore_Ipr28/results/exp21/E21_B2_HIER/seed0 --arm_id=E21_B2_HIER --exp12_common_config=/kaggle/working/WeedsGalore_Ipr28/configs/exp12/exp12_common_config.json --exp12_arms=/kaggle/working/WeedsGalore_Ipr28/configs/exp21/exp21_arms.json --log_interval=50 --ckp

In [8]:
from pathlib import Path
import subprocess
import sys

P = Path("/kaggle/working/WeedsGalore_Ipr28")

required = [
    "src/nets/exp20_icr.py",
    "src/train_exp20.py",
    "src/evaluate_exp20.py",
    "src/scripts/exp20_arms.py",
    "src/scripts/exp20_seed0.py",
    "src/scripts/exp20_test_once.py",
    "tests/test_exp20_icr.py",
    "configs/exp18/exp18_arms.json",
]

for rel in required:
    print("PASS" if (P / rel).is_file() else "MISSING", rel)

assert all((P / rel).is_file() for rel in required)

subprocess.run(
    [sys.executable, "src/scripts/exp20_arms.py"],
    cwd=P, check=True
)

subprocess.run(
    [sys.executable, "-m", "pytest", "-q",
     "tests/test_exp20_icr.py"],
    cwd=P, check=True
)

subprocess.run(
    [sys.executable, "-m", "py_compile",
     "src/train_exp20.py",
     "src/evaluate_exp20.py",
     "src/nets/exp20_icr.py"],
    cwd=P, check=True
)

print("EXP20 PREFLIGHT PASS")

PASS src/nets/exp20_icr.py
PASS src/train_exp20.py
PASS src/evaluate_exp20.py
PASS src/scripts/exp20_arms.py
PASS src/scripts/exp20_seed0.py
PASS src/scripts/exp20_test_once.py
PASS tests/test_exp20_icr.py
PASS configs/exp18/exp18_arms.json
EXP20 ARMS PASS: /kaggle/working/WeedsGalore_Ipr28/configs/exp20/exp20_arms.json
....                                                                     [100%]
4 passed in 3.00s
EXP20 PREFLIGHT PASS


In [9]:
#Cell train Exp20 seed 0
from pathlib import Path
import subprocess
import sys

P = Path("/kaggle/working/WeedsGalore_Ipr28")

DS = (
    "/kaggle/input/datasets/thothanh28/"
    "weedsgalore-dataset/weedsgalore-dataset"
)

subprocess.run(
    [
        sys.executable,
        "src/scripts/exp20_seed0.py",
        f"--dataset_path={DS}",
    ],
    cwd=P,
    check=True,
)

print("EXP20 SEED0 COMPLETED")

RUN: /usr/bin/python3 src/scripts/exp20_arms.py
EXP20 ARMS PASS: /kaggle/working/WeedsGalore_Ipr28/configs/exp20/exp20_arms.json
RUN: /usr/bin/python3 src/train_exp20.py --dataset_path=/kaggle/input/datasets/thothanh28/weedsgalore-dataset/weedsgalore-dataset --seed=0 --run_name=E20_B2_ICR_s0 --out_dir=/kaggle/working/WeedsGalore_Ipr28/results/exp20/E20_B2_ICR/seed0 --arm_id=E20_B2_ICR --exp12_common_config=/kaggle/working/WeedsGalore_Ipr28/configs/exp12/exp12_common_config.json --exp12_arms=/kaggle/working/WeedsGalore_Ipr28/configs/exp20/exp20_arms.json --log_interval=50 --ckpt_interval=1000000 --num_workers=4 --resume=True --dataset_size_train=104 --in_channels=5 --num_classes=3 --ignore_index=-1 --batch_size=8 --lr=0.0003 --weight_bg=0.9 --weight_crop=1.0 --weight_weed=1.3 --use_dice_loss=True --dice_weight=0.3 --epochs=100 --pretrained_backbone=True --spectral_guided=False --use_multiscale_fusion=True --segformer=True --ccmc=False --unet=False --dlv3p_do=False --use_attention=False 

In [8]:
from pathlib import Path
import subprocess
import sys

P = Path("/kaggle/working/WeedsGalore_Ipr28")

required = [
    "src/train_exp19.py",
    "src/exp19_ssl.py",
    "src/scripts/exp19_pretrain.py",
    "src/scripts/exp19_arms.py",
    "src/scripts/exp19_seed0.py",
    "tests/test_exp19_ssl.py",
    "configs/exp18/exp18_arms.json",
    "configs/exp12/exp12_common_config.json",
    "doc/exp02_results/results/exp02/instance_bins.json",
]

for rel in required:
    print(
        "PASS" if (P / rel).is_file() else "MISSING",
        rel
    )

assert all((P / rel).is_file() for rel in required)

subprocess.run(
    [
        sys.executable, "-m", "pytest", "-q",
        "tests/test_exp19_ssl.py",
    ],
    cwd=P,
    check=True,
)

subprocess.run(
    [
        sys.executable, "-m", "py_compile",
        "src/train_exp19.py",
        "src/exp19_ssl.py",
        "src/scripts/exp19_pretrain.py",
        "src/scripts/exp19_arms.py",
        "src/scripts/exp19_seed0.py",
    ],
    cwd=P,
    check=True,
)

print("EXP19 PREFLIGHT PASS")

PASS src/train_exp19.py
PASS src/exp19_ssl.py
PASS src/scripts/exp19_pretrain.py
PASS src/scripts/exp19_arms.py
PASS src/scripts/exp19_seed0.py
PASS tests/test_exp19_ssl.py
PASS configs/exp18/exp18_arms.json
PASS configs/exp12/exp12_common_config.json
PASS doc/exp02_results/results/exp02/instance_bins.json
....                                                                     [100%]
4 passed in 1.89s
EXP19 PREFLIGHT PASS


In [9]:
#Chạy Exp19 seed 0
from pathlib import Path
import subprocess
import sys

P = Path("/kaggle/working/WeedsGalore_Ipr28")

DS = (
    "/kaggle/input/datasets/thothanh28/"
    "weedsgalore-dataset/weedsgalore-dataset"
)

subprocess.run(
    [
        sys.executable,
        "src/scripts/exp19_seed0.py",
        f"--dataset_path={DS}",
    ],
    cwd=P,
    check=True,
)

print("EXP19 SEED0 COMPLETED")

RUN: /usr/bin/python3 src/scripts/exp19_pretrain.py --dataset_path=/kaggle/input/datasets/thothanh28/weedsgalore-dataset/weedsgalore-dataset
EXP19 SSL epoch=1/12 train_mask_loss=0.45447
EXP19 SSL epoch=2/12 train_mask_loss=0.14068
EXP19 SSL epoch=3/12 train_mask_loss=0.10584
EXP19 SSL epoch=4/12 train_mask_loss=0.10100
EXP19 SSL epoch=5/12 train_mask_loss=0.09757
EXP19 SSL epoch=6/12 train_mask_loss=0.09778
EXP19 SSL epoch=7/12 train_mask_loss=0.09758
EXP19 SSL epoch=8/12 train_mask_loss=0.09315
EXP19 SSL epoch=9/12 train_mask_loss=0.09189
EXP19 SSL epoch=10/12 train_mask_loss=0.09338
EXP19 SSL epoch=11/12 train_mask_loss=0.09425
EXP19 SSL epoch=12/12 train_mask_loss=0.09219
EXP19 SSL PASS: /kaggle/working/WeedsGalore_Ipr28/results/exp19/ssl_seed0/encoder_ssl.pth
RUN: /usr/bin/python3 src/scripts/exp19_arms.py
EXP19 ARMS PASS: /kaggle/working/WeedsGalore_Ipr28/configs/exp19/exp19_arms.json
RUN: /usr/bin/python3 src/train_exp19.py --dataset_path=/kaggle/input/datasets/thothanh28/weedsga

In [8]:
from pathlib import Path

P = Path("/kaggle/working/WeedsGalore_Ipr28")

required = [
    "doc/README_EXP18.md",
    "src/exp18_curriculum.py",
    "src/scripts/exp18_patch.py",
    "src/scripts/exp18_arms.py",
    "src/scripts/exp18_seed0.py",
    "tests/test_exp18_curriculum.py",
]

for name in required:
    print(
        "PASS" if (P / name).is_file() else "MISSING",
        name,
    )

assert all((P / name).is_file() for name in required)
print("\nEXP18 FILE STRUCTURE PASS")

PASS doc/README_EXP18.md
PASS src/exp18_curriculum.py
PASS src/scripts/exp18_patch.py
PASS src/scripts/exp18_arms.py
PASS src/scripts/exp18_seed0.py
PASS tests/test_exp18_curriculum.py

EXP18 FILE STRUCTURE PASS


In [11]:
from pathlib import Path
from datetime import datetime
import subprocess
import shutil
import sys

P = Path("/kaggle/working/WeedsGalore_Ipr28")
train = P / "src/train.py"

assert train.is_file(), "Không tìm thấy train.py"

def source():
    return train.read_text(encoding="utf-8")

def run(script):
    print("\nRUN:", script, flush=True)
    subprocess.run(
        [sys.executable, script],
        cwd=P,
        check=True,
    )

# Sao lưu trước khi thay đổi
backup = P / "src" / (
    "train_before_exp18_repair_"
    + datetime.now().strftime("%Y%m%d_%H%M%S")
    + ".py"
)

shutil.copy2(train, backup)
print("BACKUP:", backup)

# Chỉ hỗ trợ khôi phục từ train đã có Exp15
assert "exp15_lovasz_weight" in source(), (
    "train.py chưa tích hợp Exp15. "
    "Cần khôi phục train.py từ bản backup Exp17 "
    "trước khi chạy cell này."
)

# 1. Khôi phục Exp16 integration
if "EXP16_POINT_MODEL_TRAIN" not in source():
    run("src/scripts/exp16_patch.py")

assert "EXP16_POINT_MODEL_TRAIN" in source(), (
    "Khôi phục Exp16 thất bại"
)

# 2. Khôi phục Exp17 integration
if "EXP17_SAFETY_TRAIN_ONLY" not in source():
    run("src/scripts/exp17_patch.py")

assert "EXP17_SAFETY_TRAIN_ONLY" in source(), (
    "Khôi phục Exp17 thất bại"
)

# 3. Cài Exp18 curriculum
if "EXP18_CURRICULUM_HOOK" not in source():
    run("src/scripts/exp18_patch.py")

assert "EXP18_CURRICULUM_HOOK" in source(), (
    "Tích hợp Exp18 thất bại"
)

# Kiểm tra cú pháp
subprocess.run(
    [
        sys.executable,
        "-m", "py_compile",
        "src/train.py",
        "src/exp18_curriculum.py",
    ],
    cwd=P,
    check=True,
)

print("\nEXP18 SOURCE REPAIR PASS")

BACKUP: /kaggle/working/WeedsGalore_Ipr28/src/train_before_exp18_repair_20261112_140816.py

RUN: src/scripts/exp16_patch.py
EXP16 TRAIN PATCH PASS: /kaggle/working/WeedsGalore_Ipr28/src/train_pre_exp16.py
PATCH PASS: src/ccmc_common.py backup: /kaggle/working/WeedsGalore_Ipr28/src/ccmc_common.py.pre_exp16
PATCH PASS: src/evaluate_ccmc.py backup: /kaggle/working/WeedsGalore_Ipr28/src/evaluate_ccmc.py.pre_exp16
EXP16 PATCH PASS

RUN: src/scripts/exp17_patch.py
EXP17 PATCH PASS: src/train.py
EXP17 PATCH PASS: src/ccmc_common.py
EXP17 PATCH PASS: src/evaluate_ccmc.py
EXP17 INTEGRATION PASS

RUN: src/scripts/exp18_patch.py
EXP18 TRAIN PATCH PASS: /kaggle/working/WeedsGalore_Ipr28/src/train.py backup: /kaggle/working/WeedsGalore_Ipr28/src/train.py.pre_exp18

EXP18 SOURCE REPAIR PASS


In [13]:
from pathlib import Path
import shutil
import subprocess
import sys

P = Path("/kaggle/working/WeedsGalore_Ipr28")
script = P / "src/scripts/exp18_arms.py"

s = script.read_text(encoding="utf-8")

old = (
    "assert f['use_low_level_refine'] is True "
    "and f['batch_size']==8"
)
new = "assert f['use_low_level_refine'] is True"

if old in s:
    backup = script.with_suffix(".py.before_batch_fix")

    if not backup.exists():
        shutil.copy2(script, backup)

    s = s.replace(old, new, 1)
    script.write_text(s, encoding="utf-8")
    print("FIXED: Loại bỏ kiểm tra batch_size sai vị trí")

else:
    assert new in s, "Không tìm thấy dòng cần sửa"
    print("ALREADY FIXED")

# Kiểm tra cú pháp
subprocess.run(
    [sys.executable, "-m", "py_compile", str(script)],
    check=True,
)

# Tạo registry Exp18
subprocess.run(
    [sys.executable, "src/scripts/exp18_arms.py"],
    cwd=P,
    check=True,
)

# Chạy unit tests
subprocess.run(
    [
        sys.executable,
        "-m", "pytest", "-q",
        "tests/test_exp18_curriculum.py",
    ],
    cwd=P,
    check=True,
)

print("\nEXP18 REGISTRY + TESTS PASS")

FIXED: Loại bỏ kiểm tra batch_size sai vị trí
EXP18 ARMS PASS: /kaggle/working/WeedsGalore_Ipr28/configs/exp18/exp18_arms.json
....                                                                     [100%]
4 passed in 1.84s

EXP18 REGISTRY + TESTS PASS


In [14]:
from pathlib import Path
import subprocess
import sys

P = Path("/kaggle/working/WeedsGalore_Ipr28")

train = (P / "src/train.py").read_text(encoding="utf-8")
evaluate = (P / "src/evaluate_ccmc.py").read_text(
    encoding="utf-8"
)

checks = {
    "Exp16 train integration":
        "EXP16_POINT_MODEL_TRAIN" in train,
    "Exp17 train integration":
        "EXP17_SAFETY_TRAIN_ONLY" in train,
    "Exp18 curriculum":
        "EXP18_CURRICULUM_HOOK" in train,
    "Exp18 flag":
        "exp18_curriculum" in train,
    "Exp16 flag":
        "exp16_point_refine" in train,
    "Exp17 flag":
        "exp17_safety_gate" in train,
    "Exp16 evaluation flag":
        "exp16_point_refine" in evaluate,
    "Exp17 evaluation flag":
        "exp17_safety_gate" in evaluate,
    "Exp18 dataset":
        (P / "src/exp18_curriculum.py").is_file(),
}

for name, ok in checks.items():
    print(f"{'PASS' if ok else 'FAIL'}: {name}")

assert all(checks.values()), (
    "Project chưa đủ dependency để chạy Exp18"
)

subprocess.run(
    [
        sys.executable,
        "src/scripts/exp18_arms.py",
    ],
    cwd=P,
    check=True,
)

subprocess.run(
    [
        sys.executable,
        "-m", "pytest", "-q",
        "tests/test_exp18_curriculum.py",
    ],
    cwd=P,
    check=True,
)

print("\nEXP18 PREFLIGHT PASS")

PASS: Exp16 train integration
PASS: Exp17 train integration
PASS: Exp18 curriculum
PASS: Exp18 flag
PASS: Exp16 flag
PASS: Exp17 flag
PASS: Exp16 evaluation flag
PASS: Exp17 evaluation flag
PASS: Exp18 dataset
EXP18 ARMS PASS: /kaggle/working/WeedsGalore_Ipr28/configs/exp18/exp18_arms.json
....                                                                     [100%]
4 passed in 1.83s

EXP18 PREFLIGHT PASS


In [15]:
from pathlib import Path
import subprocess
import sys

P = Path("/kaggle/working/WeedsGalore_Ipr28")

DS = (
    "/kaggle/input/datasets/thothanh28/"
    "weedsgalore-dataset/weedsgalore-dataset"
)

subprocess.run(
    [
        sys.executable,
        "src/scripts/exp18_seed0.py",
        f"--dataset_path={DS}",
    ],
    cwd=P,
    check=True,
)

print("EXP18 SEED0 COMPLETED")

RUN: /usr/bin/python3 src/train.py --dataset_path=/kaggle/input/datasets/thothanh28/weedsgalore-dataset/weedsgalore-dataset --seed=0 --run_name=E18_B2_OBJECT_CURRICULUM_s0 --out_dir=/kaggle/working/WeedsGalore_Ipr28/results/exp18/E18_B2_OBJECT_CURRICULUM/seed0 --arm_id=E18_B2_OBJECT_CURRICULUM --exp12_common_config=/kaggle/working/WeedsGalore_Ipr28/configs/exp12/exp12_common_config.json --exp12_arms=/kaggle/working/WeedsGalore_Ipr28/configs/exp18/exp18_arms.json --log_interval=50 --ckpt_interval=1000000 --num_workers=4 --resume=True --dataset_size_train=104 --in_channels=5 --num_classes=3 --ignore_index=-1 --batch_size=8 --lr=0.0003 --weight_bg=0.9 --weight_crop=1.0 --weight_weed=1.3 --use_dice_loss=True --dice_weight=0.3 --epochs=100 --pretrained_backbone=True --spectral_guided=False --use_multiscale_fusion=True --segformer=True --ccmc=False --unet=False --dlv3p_do=False --use_attention=False --backbone=b2 --head=bilinear --use_low_level_refine=True --llr_source=raw --input_scale=1.0 

In [16]:
#Cell kiểm toán 5 band + thử khả năng phân biệt phổ
from pathlib import Path
import sys, json, inspect, re
import numpy as np

P = Path("/kaggle/working/WeedsGalore_Ipr28")
DS = "/kaggle/input/datasets/thothanh28/weedsgalore-dataset/weedsgalore-dataset"
OUT = P / "results/pre_exp19_audit"
OUT.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(P / "src"))
from datasets import WeedsGaloreDataset

def make_ds(split):
    return WeedsGaloreDataset(
        dataset_path=DS,
        dataset_size=104 if split == "train" else None,
        in_bands=5,
        num_classes=3,
        is_training=False,
        split=split,
        augmentation=False,
    )

train = make_ds("train")
val = make_ds("val")

assert len(train) == 104
assert len(val) == 26

rng = np.random.default_rng(2026)

# ---------- A. Kiểm tra source code ----------
print("\n========== A. SOURCE AUDIT ==========")

for rel in [
    "src/datasets.py",
    "src/train.py",
    "src/nets/segformer.py",
    "src/nets/ccmc.py",
]:
    f = P / rel
    if not f.is_file():
        continue

    lines = f.read_text(
        encoding="utf-8", errors="replace"
    ).splitlines()

    keywords = re.compile(
        r"normaliz|mean|std|percentile|clip|resize|"
        r"interpolat|pretrain|patch_embed|conv1|"
        r"in_channels|in_bands|input_scale|"
        r"reflect|uint16|float32",
        re.I,
    )

    matches = [
        (i, line.strip())
        for i, line in enumerate(lines, 1)
        if keywords.search(line)
    ]

    print(f"\n{rel} ({len(matches)} matches)")
    for i, line in matches[:35]:
        print(f"  L{i}: {line[:150]}")

# ---------- B. Lấy mẫu pixel từ train/val ----------
print("\n========== B. BAND STATISTICS ==========")

def collect(ds, max_pixels_per_image=12000):
    xs, ys = [], []
    image_ranges = []
    invalid = 0
    total = 0

    for i in range(len(ds)):
        sample = ds[i]
        im = np.asarray(sample[0], dtype=np.float32)
        mask = np.asarray(sample[2])

        assert im.shape == (5, 600, 600), im.shape
        assert mask.shape == (600, 600), mask.shape

        finite = np.isfinite(im).all(axis=0)
        valid = finite & (mask >= 0) & (mask <= 2)

        invalid += int((~finite).sum())
        total += mask.size

        flat_idx = np.flatnonzero(valid.reshape(-1))
        assert len(flat_idx) > 0

        k = min(max_pixels_per_image, len(flat_idx))
        chosen = rng.choice(flat_idx, size=k, replace=False)

        x = im.reshape(5, -1)[:, chosen].T
        y = mask.reshape(-1)[chosen].astype(np.int64)

        xs.append(x)
        ys.append(y)

        image_ranges.append({
            "image_idx": i,
            "min": np.min(im, axis=(1,2)).tolist(),
            "max": np.max(im, axis=(1,2)).tolist(),
        })

        if (i + 1) % 10 == 0:
            print(f"Read {i+1}/{len(ds)}")

    return (
        np.concatenate(xs),
        np.concatenate(ys),
        image_ranges,
        invalid,
        total,
    )

Xtr, Ytr, ranges_tr, inv_tr, ntr = collect(train)
Xva, Yva, ranges_va, inv_va, nva = collect(val)

def band_report(x, title):
    print(f"\n{title}")
    print("band | min | p01 | median | p99 | max | mean | std")

    result = []
    for b in range(5):
        z = x[:, b]
        row = {
            "band": b,
            "min": float(z.min()),
            "p01": float(np.percentile(z, 1)),
            "median": float(np.median(z)),
            "p99": float(np.percentile(z, 99)),
            "max": float(z.max()),
            "mean": float(z.mean()),
            "std": float(z.std()),
        }
        result.append(row)

        print(
            f"{b:>4} | "
            f"{row['min']:.4g} | "
            f"{row['p01']:.4g} | "
            f"{row['median']:.4g} | "
            f"{row['p99']:.4g} | "
            f"{row['max']:.4g} | "
            f"{row['mean']:.4g} | "
            f"{row['std']:.4g}"
        )
    return result

stats_train = band_report(Xtr, "TRAIN")
stats_val = band_report(Xva, "VALIDATION")

print("\nNon-finite pixels:")
print("train:", inv_tr, "/", ntr)
print("val:  ", inv_va, "/", nva)

# ---------- C. Band distribution shift ----------
print("\n========== C. TRAIN-VAL SHIFT ==========")

mean_train = Xtr.mean(axis=0)
std_train = Xtr.std(axis=0).clip(min=1e-6)
mean_val = Xva.mean(axis=0)

shift = np.abs(mean_val - mean_train) / std_train

for b, v in enumerate(shift):
    print(f"Band {b}: mean shift = {v:.3f} train SD")

# ---------- D. Spectral class separation ----------
print("\n========== D. SPECTRAL SEPARATION ==========")

# Chuẩn hóa hoàn toàn bằng train; không fit trên val
Ztr = (Xtr - mean_train) / std_train
Zva = (Xva - mean_train) / std_train

centroids = []
for c in range(3):
    z = Ztr[Ytr == c]
    assert len(z) > 100, f"Class {c} has too few samples"
    center = z.mean(axis=0)
    centroids.append(center)
    print(
        f"Class {c} | n={len(z)} | "
        f"center={np.round(center, 3)}"
    )

centroids = np.stack(centroids)

print("\nDistances between spectral class centroids:")
for a, b in [(0, 1), (0, 2), (1, 2)]:
    d = float(np.linalg.norm(centroids[a] - centroids[b]))
    print(f"{a} vs {b}: {d:.4f}")

# Chẩn đoán phổ đơn giản: nearest centroid
dist = ((Zva[:, None, :] - centroids[None, :, :]) ** 2).sum(axis=2)
pred = dist.argmin(axis=1)

cm = np.bincount(
    3 * Yva + pred, minlength=9
).reshape(3, 3)

print("\nSpectral-only confusion matrix")
print("GT rows / predicted columns:")
print(cm)

ious = []
for c in range(3):
    tp = cm[c, c]
    fp = cm[:, c].sum() - tp
    fn = cm[c, :].sum() - tp

    iou = 100 * tp / max(tp + fp + fn, 1)
    rec = 100 * tp / max(cm[c, :].sum(), 1)

    ious.append(float(iou))
    print(f"Class {c}: IoU={iou:.2f}% | recall={rec:.2f}%")

print(
    "\nSpectral-only sampled mIoU:",
    round(float(np.mean(ious)), 3),
)

print(
    "NOTE: This is NOT comparable to full-image "
    "SegFormer validation mIoU."
)

# ---------- E. Save report ----------
report = {
    "split": "train_and_val_only",
    "train_n": len(train),
    "val_n": len(val),
    "test_opened": False,
    "train_band_stats": stats_train,
    "val_band_stats": stats_val,
    "mean_shift_train_std": shift.tolist(),
    "class_centroids": centroids.tolist(),
    "spectral_only_confusion": cm.tolist(),
    "spectral_only_sampled_miou": float(np.mean(ious)),
    "nonfinite_train_pixels": inv_tr,
    "nonfinite_val_pixels": inv_va,
}

path = OUT / "spectral_audit.json"
path.write_text(json.dumps(report, indent=2))

print("\nAUDIT SAVED:", path)
print("PRE-EXP19 SPECTRAL AUDIT PASS")


========== A. SOURCE AUDIT ==========

src/train.py (41 matches)
  L62: scalar loss (1 - mean Dice over classes), lower is better, same convention as CE.
  L84: loss = 1. - dice_per_class.mean()
  L94: flags.DEFINE_integer('in_channels', 5, 'options: 3 (RGB), 5 (MSI)')
  L112: flags.DEFINE_boolean('pretrained_backbone', True, 'set True to use pretrained ResNet50 backbone')
  L113: flags.DEFINE_string('ckpt_resnet', 'ckpts/resnet50-19c8e357.pth', 'ckpt path for pretrained backbone')
  L183: flags.DEFINE_float('input_scale', 1.0, 'input scale: images bilinear, labels nearest; validation logits are resized '
  L184: 'back to the original size (pure interpolation: adds no spatial information)')
  L222: stderr=subprocess.DEVNULL).decode().strip()
  L235: def measure_model_stats(model, device, in_channels, input_size=(512, 512)):
  L249: dummy_input = torch.randn(1, in_channels, input_size[0], input_size[1]).to(device)
  L261: dummy_input = torch.randn(1, in_channels, input_size[0], input_s

In [17]:
#Cell kiểm tra checkpoint có nhận đủ 5 band không
from pathlib import Path
import torch

P = Path("/kaggle/working/WeedsGalore_Ipr28")

candidates = {
    "Exp15-A":
        P / "results/exp15/E15_A_CW_LOVASZ/seed0/final_ema.pth",
    "Exp18":
        P / "results/exp18/E18_B2_OBJECT_CURRICULUM/seed0/final_ema.pth",
}

for name, path in candidates.items():
    print(f"\n========== {name} ==========")

    if not path.exists():
        print("SKIP: checkpoint not found")
        continue

    ckpt = torch.load(
        path,
        map_location="cpu",
        weights_only=False,
    )

    if isinstance(ckpt, dict):
        for key in [
            "state_dict", "model_state_dict",
            "model", "net", "ema_state_dict",
        ]:
            if key in ckpt and isinstance(ckpt[key], dict):
                ckpt = ckpt[key]
                break

    assert isinstance(ckpt, dict), "Unknown checkpoint format"

    five_channel = []

    for k, v in ckpt.items():
        if isinstance(v, torch.Tensor) and v.ndim == 4:
            if v.shape[1] == 5:
                five_channel.append((k, tuple(v.shape)))

    print("5-channel convolution weights:")
    for key, shape in five_channel[:20]:
        print(" ", key, shape)

    if not five_channel:
        print(
            "WARNING: No direct 5-channel conv found. "
            "Inspect input projection architecture."
        )

print("\nCHECKPOINT AUDIT PASS")


========== Exp15-A ==========
SKIP: checkpoint not found

========== Exp18 ==========
5-channel convolution weights:
  encoder.encoder.patch_embeddings.0.proj.weight (64, 5, 7, 7)
  low_level.proj.0.weight (48, 5, 3, 3)

CHECKPOINT AUDIT PASS


In [18]:
#cell này để xem chính xác code đang lấy band nào, từ file nào, và trọng số 5 kênh sau huấn luyện có khác nhau hay không:
from pathlib import Path
import sys
import inspect
import numpy as np
import torch

P = Path("/kaggle/working/WeedsGalore_Ipr28")
sys.path.insert(0, str(P / "src"))

from datasets import WeedsGaloreDataset

print("=== DATASET SOURCE ===")
print(inspect.getfile(WeedsGaloreDataset))

source, start = inspect.getsourcelines(WeedsGaloreDataset)
keywords = (
    "band", "channel", "stack", "normal", "scale",
    "transpose", "read", "image", "nir", "rededge",
    "red_edge", "order"
)

for n, line in enumerate(source, start):
    if any(k in line.lower() for k in keywords):
        print(f"L{n}: {line.rstrip()[:180]}")

print("\n=== EXP18 TRAINED INPUT WEIGHTS ===")

ckpt = (
    P / "results/exp18/E18_B2_OBJECT_CURRICULUM/"
    "seed0/final_ema.pth"
)

assert ckpt.is_file(), "Thiếu checkpoint Exp18"

state = torch.load(
    ckpt, map_location="cpu", weights_only=False
)

for key in ("state_dict", "model_state_dict", "model", "net"):
    if isinstance(state, dict) and isinstance(state.get(key), dict):
        state = state[key]
        break

for key in [
    "encoder.encoder.patch_embeddings.0.proj.weight",
    "low_level.proj.0.weight",
]:
    w = state.get(key)
    if w is None:
        print("MISSING:", key)
        continue

    w = w.float()
    norms = w.square().sum(dim=(0, 2, 3)).sqrt().numpy()
    rel = norms / (norms.mean() + 1e-12)

    print("\n", key)
    print("Weight norm:", np.round(norms, 4))
    print("Relative norm:", np.round(rel, 3))

print("\nBAND ORDER AUDIT PASS")

=== DATASET SOURCE ===
/kaggle/working/WeedsGalore_Ipr28/src/datasets/weedsgalore/weedsgalore.py
L15:     def __init__(self, dataset_path, dataset_size, in_bands, num_classes, is_training, split, augmentation):
L17:         self.in_bands = in_bands
L40:         # load single bands and construct input image
L41:         img_path = os.path.join(self.img_dir, self.img_list[idx][:10], 'images', self.img_list[idx])
L42:         red_band = plt.imread(img_path + '_R.png')
L43:         green_band = plt.imread(img_path + '_G.png')
L44:         blue_band = plt.imread(img_path + '_B.png')
L45:         nir_band = plt.imread(img_path + '_NIR.png')
L46:         re_band = plt.imread(img_path + '_RE.png')
L47:         if self.in_bands == 3:
L48:             img = np.stack((red_band, green_band, blue_band))
L49:         elif self.in_bands == 5:
L50:             img = np.stack((red_band, green_band, blue_band, nir_band, re_band))
L54:         label = Image.open(label_path + '.png')

=== EXP18 TRAINED IN

In [19]:
#Cell chẩn đoán mức đóng góp của từng band
from pathlib import Path
import numpy as np

P = Path("/kaggle/working/WeedsGalore_Ipr28")
audit = P / "results/pre_exp19_audit/spectral_audit.json"

import json
report = json.loads(audit.read_text())

medians = [
    x["median"]
    for x in report["train_band_stats"]
]

print("TRAIN band medians (R,G,B,NIR,RE):")
print(np.round(medians, 5))
print("\nBAND ABLATION CONFIG PASS")

TRAIN band medians (R,G,B,NIR,RE):
[0.47266 0.44825 0.4043  0.39454 0.42872]

BAND ABLATION CONFIG PASS


In [20]:
from pathlib import Path
import zipfile

P = Path("/kaggle/working/WeedsGalore_Ipr28")
OUT = Path("/kaggle/working/exp19_source_for_integration.zip")

files = [
    "src/train.py",
    "src/nets/segformer.py",
    "src/evaluate_ccmc.py",
    "src/ccmc_common.py",
    "src/datasets/weedsgalore/weedsgalore.py",
    "src/exp18_curriculum.py",
    "src/scripts/exp18_seed0.py",
    "src/scripts/exp18_arms.py",
    "src/exp12_config.py",
    "configs/exp12/exp12_common_config.json",
]

with zipfile.ZipFile(OUT, "w", zipfile.ZIP_DEFLATED) as z:
    for rel in files:
        path = P / rel
        if path.is_file():
            z.write(path, rel)
            print("ADDED:", rel)
        else:
            print("MISSING:", rel)

with zipfile.ZipFile(OUT) as z:
    assert z.testzip() is None

print("SOURCE EXPORT PASS:", OUT)

ADDED: src/train.py
ADDED: src/nets/segformer.py
ADDED: src/evaluate_ccmc.py
ADDED: src/ccmc_common.py
ADDED: src/datasets/weedsgalore/weedsgalore.py
ADDED: src/exp18_curriculum.py
ADDED: src/scripts/exp18_seed0.py
ADDED: src/scripts/exp18_arms.py
ADDED: src/exp12_config.py
ADDED: configs/exp12/exp12_common_config.json
SOURCE EXPORT PASS: /kaggle/working/exp19_source_for_integration.zip


In [21]:
import shutil

src = "/kaggle/working/WeedsGalore_Ipr28/results/exp18"
dst = "/kaggle/working/exp18.zip"

shutil.make_archive(
    dst.replace(".zip", ""),
    "zip",
    root_dir=src
)

print("ZIP CREATED:", dst)

ZIP CREATED: /kaggle/working/exp18.zip


In [ ]:
#Kiểm tra và sửa dependency Exp16
from pathlib import Path
import subprocess
import sys
import shutil
from datetime import datetime

PROJECT = Path("/kaggle/working/WeedsGalore_Ipr28")
train = PROJECT / "src/train.py"
exp16_patch = PROJECT / "src/scripts/exp16_patch.py"
exp16_model = PROJECT / "src/nets/exp16_point.py"

assert train.is_file(), "Thiếu src/train.py"
assert exp16_patch.is_file(), "Thiếu exp16_patch.py"
assert exp16_model.is_file(), "Thiếu exp16_point.py"

s = train.read_text(encoding="utf-8")

print("Exp15 present:", "exp15_lovasz_weight" in s)
print("Exp16 present:", "EXP16_POINT_MODEL_TRAIN" in s)
print("Exp17 present:", "EXP17_SAFETY_TRAIN_ONLY" in s)

if "EXP16_POINT_MODEL_TRAIN" not in s:
    assert "exp15_lovasz_weight" in s, (
        "train.py thiếu cả Exp15: cần khôi phục phiên bản "
        "train.py phù hợp trước khi patch"
    )

    backup = train.with_name(
        "train_before_exp17_repair_"
        + datetime.now().strftime("%Y%m%d_%H%M%S")
        + ".py"
    )
    shutil.copy2(train, backup)
    print("BACKUP:", backup)

    subprocess.run(
        [sys.executable, str(exp16_patch)],
        cwd=PROJECT,
        check=True,
    )
else:
    print("Exp16 đã tích hợp, không patch lại.")

assert "EXP16_POINT_MODEL_TRAIN" in train.read_text(
    encoding="utf-8"
), "Exp16 integration chưa hoàn tất"

print("EXP16 DEPENDENCY PASS")

In [ ]:
#Chạy preflight Exp17
from pathlib import Path
import subprocess
import sys

PROJECT = Path("/kaggle/working/WeedsGalore_Ipr28")

commands = [
    ["src/scripts/exp17_patch.py"],
    ["src/scripts/exp17_arms.py"],
    ["-m", "pytest", "-q", "tests/test_exp17_gate.py"],
    [
        "-m", "py_compile",
        "src/train.py",
        "src/ccmc_common.py",
        "src/evaluate_ccmc.py",
    ],
]

for cmd in commands:
    subprocess.run(
        [sys.executable, *cmd],
        cwd=PROJECT,
        check=True,
    )

print("EXP17 PREFLIGHT PASS")

In [ ]:
#Kiểm tra tích hợp trước khi train exp17
from pathlib import Path

P = Path("/kaggle/working/WeedsGalore_Ipr28")

checks = {
    "Exp16 train integration":
        "EXP16_POINT_MODEL_TRAIN" in
        (P / "src/train.py").read_text(),

    "Exp17 train integration":
        "EXP17_SAFETY_TRAIN_ONLY" in
        (P / "src/train.py").read_text(),

    "Exp17 model":
        (P / "src/nets/exp17_gate.py").is_file(),

    "Exp17 registry":
        (P / "configs/exp17/exp17_arms.json").is_file(),

    "Exp17 evaluation integration":
        "exp17_safety_gate" in
        (P / "src/evaluate_ccmc.py").read_text(),
}

for name, passed in checks.items():
    print(("PASS" if passed else "FAIL"), name)

assert all(checks.values()), "Exp17 chưa sẵn sàng"
print("\nEXP17 READY FOR TRAINING")

In [ ]:
#Train Exp17 seed 0 + validation D4
from pathlib import Path
import subprocess
import sys

PROJECT = Path("/kaggle/working/WeedsGalore_Ipr28")

DS = (
    "/kaggle/input/datasets/thothanh28/"
    "weedsgalore-dataset/weedsgalore-dataset"
)

subprocess.run(
    [
        sys.executable,
        "src/scripts/exp17_seed0.py",
        f"--dataset_path={DS}",
    ],
    cwd=PROJECT,
    check=True,
)

print("EXP17 SEED0 COMPLETED")

In [ ]:
from pathlib import Path
import zipfile

PROJECT = Path("/kaggle/working/WeedsGalore_Ipr28")
OUT = Path("/kaggle/working/exp17_seed0_backup.zip")

items = [
    "results/exp17",
    "results/exp17_diagnostics",
    "configs/exp17",
    "src/nets/exp17_gate.py",
    "src/nets/exp16_point.py",
    "src/scripts/exp17_patch.py",
    "src/scripts/exp17_arms.py",
    "src/scripts/exp17_seed0.py",
    "tests/test_exp17_gate.py",
    "README_EXP17.md",
    "src/train.py",
    "src/ccmc_common.py",
    "src/evaluate_ccmc.py",
]

with zipfile.ZipFile(
    OUT, "w",
    compression=zipfile.ZIP_DEFLATED,
    compresslevel=6,
    allowZip64=True,
) as z:
    for rel in items:
        p = PROJECT / rel
        if not p.exists():
            print("SKIP:", rel)
            continue
        files = [p] if p.is_file() else p.rglob("*")
        for f in files:
            if f.is_file():
                z.write(f, f.relative_to(PROJECT))

with zipfile.ZipFile(OUT) as z:
    assert z.testzip() is None
    names = z.namelist()
    assert any(n.endswith("/final_ema.pth") for n in names)
    assert any(n.endswith("/seed0_screen.json") for n in names)

print("EXP17 BACKUP PASS:", OUT)
print("Size GB:", round(OUT.stat().st_size / 1024**3, 2))
print("Files:", len(names))

In [ ]:
#Cell sửa lỗi import và chạy lại diagnose
from pathlib import Path
import subprocess
import sys

PROJECT = Path("/kaggle/working/WeedsGalore_Ipr28")
p = PROJECT / "src/scripts/exp16_diagnose.py"

s = p.read_text(encoding="utf-8")

old = "from datasets import WeedsGaloreDataset"

new = """import sys
sys.path.insert(0, str(Path(__file__).resolve().parents[1]))
from datasets import WeedsGaloreDataset"""

if old in s and new not in s:
    s = s.replace(old, new, 1)
    p.write_text(s, encoding="utf-8")

subprocess.run(
    [sys.executable, "-m", "py_compile", str(p)],
    check=True
)

DS = (
    "/kaggle/input/datasets/thothanh28/"
    "weedsgalore-dataset/weedsgalore-dataset"
)

subprocess.run(
    [
        sys.executable,
        "src/scripts/exp16_diagnose.py",
        f"--dataset_path={DS}",
    ],
    cwd=PROJECT,
    check=True
)

print("EXP16 DIAGNOSE PASS")

In [ ]:
#exp16_Sửa lỗi AMP dtype
from pathlib import Path
import shutil
import subprocess
import sys

PROJECT = Path("/kaggle/working/WeedsGalore_Ipr28")
p = PROJECT / "src/nets/exp16_point.py"

s = p.read_text(encoding="utf-8")

old = "delta.transpose(1, 2))"
new = "delta.transpose(1, 2).to(dtype=correction.dtype))"

if old in s:
    backup = p.with_suffix(".py.before_amp_fix")
    if not backup.exists():
        shutil.copy2(p, backup)

    s = s.replace(old, new, 1)
    p.write_text(s, encoding="utf-8")

assert new in s, "Không tìm thấy đoạn scatter cần sửa"

subprocess.run(
    [sys.executable, "-m", "py_compile", str(p)],
    check=True,
)

subprocess.run(
    [
        sys.executable,
        "-m", "pytest", "-q",
        "tests/test_exp16_point.py",
    ],
    cwd=PROJECT,
    check=True,
)

print("EXP16 AMP DTYPE FIX PASS")

In [ ]:
#exp16 - train seed 0
from pathlib import Path
import subprocess
import sys

PROJECT = Path("/kaggle/working/WeedsGalore_Ipr28")

DS = (
    "/kaggle/input/datasets/thothanh28/"
    "weedsgalore-dataset/weedsgalore-dataset"
)

subprocess.run(
    [
        sys.executable,
        "src/scripts/exp16_seed0.py",
        f"--dataset_path={DS}",
    ],
    cwd=PROJECT,
    check=True,
)

print("EXP16 SEED0 COMPLETED")

In [ ]:
from pathlib import Path
import zipfile

P = Path("/kaggle/working/WeedsGalore_Ipr28")
OUT = Path("/kaggle/working/exp16_point_seed0_backup.zip")

items = [
    "results/exp16",
    "configs/exp16",
    "src/nets/exp16_point.py",
    "src/scripts/exp16_patch.py",
    "src/scripts/exp16_arms.py",
    "src/scripts/exp16_diagnose.py",
    "src/scripts/exp16_seed0.py",
    "tests/test_exp16_point.py",
    "README_EXP16.md",
    "src/train.py",
    "src/ccmc_common.py",
    "src/evaluate_ccmc.py",
    "src/train_pre_exp16.py",
    "src/ccmc_common.py.pre_exp16",
    "src/evaluate_ccmc.py.pre_exp16",
]

with zipfile.ZipFile(
    OUT, "w",
    compression=zipfile.ZIP_DEFLATED,
    compresslevel=6,
    allowZip64=True
) as z:
    for rel in items:
        p = P / rel
        if not p.exists():
            print("SKIP:", rel)
            continue
        files = [p] if p.is_file() else p.rglob("*")
        for f in files:
            if f.is_file():
                z.write(f, f.relative_to(P))

with zipfile.ZipFile(OUT) as z:
    assert z.testzip() is None
    names = z.namelist()
    assert (
        "results/exp16/E16_B2_SPECTRAL_POINT/"
        "seed0/final_ema.pth"
    ) in names
    assert any(n.endswith("seed0_screen.json") for n in names)

print("BACKUP PASS:", OUT)
print("Size (GB):", round(OUT.stat().st_size / 1024**3, 2))

In [ ]:
#Cell chẩn đoán Exp16 trước khi thiết kế Exp17
from pathlib import Path
import subprocess
import sys

PROJECT = Path("/kaggle/working/WeedsGalore_Ipr28")

DS = (
    "/kaggle/input/datasets/thothanh28/"
    "weedsgalore-dataset/weedsgalore-dataset"
)

CKPT = (
    PROJECT
    / "results/exp16/E16_B2_SPECTRAL_POINT/seed0/final_ema.pth"
)

assert CKPT.is_file(), f"Thiếu checkpoint: {CKPT}"

SCRIPT = PROJECT / "src/scripts/exp17_point_audit.py"

SCRIPT.write_text(r'''
import argparse
import csv
import json
import sys
from pathlib import Path

ROOT = Path(__file__).resolve().parents[2]
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import torch
import torch.nn.functional as F

from datasets import WeedsGaloreDataset
from nets.exp16_point import SpectralPointRefineSegFormer
from ccmc_common import load_checkpoint


def scores(cm):
    result = {}
    for c, name in [(1, "crop"), (2, "weed")]:
        tp = int(cm[c, c])
        fp = int(cm[:, c].sum() - tp)
        fn = int(cm[c, :].sum() - tp)

        result[name] = {
            "TP": tp,
            "FP": fp,
            "FN": fn,
            "precision_pct": round(100 * tp / max(tp + fp, 1), 3),
            "recall_pct": round(100 * tp / max(tp + fn, 1), 3),
            "IoU_pct": round(100 * tp / max(tp + fp + fn, 1), 3),
        }
    return result


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--dataset_path", required=True)
    ap.add_argument("--ckpt", required=True)
    args = ap.parse_args()

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    net = SpectralPointRefineSegFormer(
        num_classes=3,
        pretrained_backbone=True,
        in_channels=5,
        use_low_level_refine=True,
        use_multiscale_fusion=True,
        llr_source="raw",
        backbone="b2",
        point_budget=4096,
    )

    net = load_checkpoint(net, args.ckpt, device).to(device).eval()

    ds = WeedsGaloreDataset(
        dataset_path=args.dataset_path,
        dataset_size=None,
        in_bands=5,
        num_classes=3,
        is_training=False,
        split="val",
        augmentation=False,
    )

    assert len(ds) == 26, "Validation split không đúng 26 ảnh"

    captured = {}

    def hook(module, inputs, output):
        captured["coarse_half"] = output.detach()

    handle = net.classifier.register_forward_hook(hook)

    cm = np.zeros((3, 3), dtype=np.int64)
    coarse_cm = np.zeros((3, 3), dtype=np.int64)

    totals = {
        "selected": 0,
        "selected_wrong_before": 0,
        "selected_wrong_after": 0,
        "selected_corrected": 0,
        "selected_damaged": 0,
        "selected_near_gt_border": 0,
        "all_wrong_before": 0,
        "all_wrong_after": 0,
        "all_pixels": 0,
        "all_near_gt_border": 0,
        "crop_weed_selected": 0,
    }

    rows = []

    try:
        with torch.inference_mode():
            for i in range(len(ds)):
                img, _, lab = ds[i]
                x = torch.from_numpy(
                    np.ascontiguousarray(img)
                ).float().unsqueeze(0).to(device)

                gt = torch.as_tensor(
                    np.asarray(lab),
                    device=device
                ).long()

                captured.clear()

                # Full forward: coarse + point refinement
                refined_logits = net(x).float()

                coarse_half = captured["coarse_half"]
                coarse_logits = F.interpolate(
                    coarse_half.float(),
                    size=gt.shape,
                    mode="bilinear",
                    align_corners=False,
                )

                prob = coarse_logits.softmax(dim=1)
                top = prob.topk(2, dim=1).values

                uncertainty = -(
                    top[:, 0] - top[:, 1]
                ).flatten(1)

                n = min(net.point_budget, gt.numel())

                # Giống lựa chọn điểm của Exp16 lúc inference
                ids = uncertainty.topk(n, dim=1).indices[0]

                coarse = coarse_logits.argmax(dim=1)[0]
                refined = refined_logits.argmax(dim=1)[0]

                valid = (gt >= 0) & (gt < 3)

                # GT border: chỉ dùng phân tích sau inference
                g = gt.clamp(0, 2)[None, None].float()
                mx = F.max_pool2d(g, 7, stride=1, padding=3)
                mn = -F.max_pool2d(-g, 7, stride=1, padding=3)
                border = (mx != mn)[0, 0] & valid

                gt_flat = gt.flatten()
                before_flat = coarse.flatten()
                after_flat = refined.flatten()
                valid_flat = valid.flatten()

                selected_valid = valid_flat[ids]
                chosen = ids[selected_valid]

                wrong_before = (coarse != gt) & valid
                wrong_after = (refined != gt) & valid

                wrong_before_sel = (
                    before_flat[chosen] != gt_flat[chosen]
                )
                wrong_after_sel = (
                    after_flat[chosen] != gt_flat[chosen]
                )

                corrected = wrong_before_sel & ~wrong_after_sel
                damaged = ~wrong_before_sel & wrong_after_sel

                totals["selected"] += chosen.numel()
                totals["selected_wrong_before"] += int(
                    wrong_before_sel.sum().item()
                )
                totals["selected_wrong_after"] += int(
                    wrong_after_sel.sum().item()
                )
                totals["selected_corrected"] += int(
                    corrected.sum().item()
                )
                totals["selected_damaged"] += int(
                    damaged.sum().item()
                )
                totals["selected_near_gt_border"] += int(
                    border.flatten()[chosen].sum().item()
                )
                totals["crop_weed_selected"] += int(
                    ((gt_flat[chosen] == 1) |
                     (gt_flat[chosen] == 2)).sum().item()
                )
                totals["all_wrong_before"] += int(
                    wrong_before.sum().item()
                )
                totals["all_wrong_after"] += int(
                    wrong_after.sum().item()
                )
                totals["all_pixels"] += int(valid.sum().item())
                totals["all_near_gt_border"] += int(border.sum().item())

                y = gt[valid].cpu().numpy()
                p = refined[valid].cpu().numpy()
                q = coarse[valid].cpu().numpy()

                cm += np.bincount(
                    3 * y + p, minlength=9
                ).reshape(3, 3)

                coarse_cm += np.bincount(
                    3 * y + q, minlength=9
                ).reshape(3, 3)

                row = {
                    "image": str(ds.img_list[i]),
                    "selected": int(chosen.numel()),
                    "selected_errors": int(wrong_before_sel.sum()),
                    "corrected": int(corrected.sum()),
                    "damaged": int(damaged.sum()),
                    "total_errors_before": int(wrong_before.sum()),
                    "total_errors_after": int(wrong_after.sum()),
                }
                rows.append(row)

                print(
                    f"{i+1}/26 | selected errors="
                    f"{row['selected_errors']} | corrected="
                    f"{row['corrected']} | damaged="
                    f"{row['damaged']}",
                    flush=True,
                )
    finally:
        handle.remove()

    t = totals
    pct = lambda a, b: round(100 * a / max(b, 1), 3)

    report = {
        "split": "val",
        "n_images": 26,
        "tta": "none",
        "checkpoint": "final_ema.pth",
        "point_budget": 4096,
        "test_opened": False,
        "gt_used_only_for_diagnostics": True,
        "coarse_crop_weed": scores(coarse_cm),
        "refined_crop_weed": scores(cm),
        "confusion_gt_rows_pred_cols": cm.tolist(),
        "point_analysis": {
            "selected_pixels": t["selected"],
            "selected_error_rate_pct": pct(
                t["selected_wrong_before"], t["selected"]
            ),
            "global_error_rate_before_pct": pct(
                t["all_wrong_before"], t["all_pixels"]
            ),
            "selected_gt_border_pct": pct(
                t["selected_near_gt_border"], t["selected"]
            ),
            "global_gt_border_pct": pct(
                t["all_near_gt_border"], t["all_pixels"]
            ),
            "selected_crop_weed_gt_pct": pct(
                t["crop_weed_selected"], t["selected"]
            ),
            "errors_corrected": t["selected_corrected"],
            "correct_predictions_damaged": t["selected_damaged"],
            "net_corrected_pixels": (
                t["selected_corrected"] - t["selected_damaged"]
            ),
            "error_count_before": t["all_wrong_before"],
            "error_count_after": t["all_wrong_after"],
        },
    }

    out = ROOT / "results/exp17_diagnostics"
    out.mkdir(parents=True, exist_ok=True)

    (out / "exp16_point_audit.json").write_text(
        json.dumps(report, indent=2)
    )

    with (out / "exp16_point_per_image.csv").open(
        "w", newline=""
    ) as f:
        writer = csv.DictWriter(f, fieldnames=rows[0].keys())
        writer.writeheader()
        writer.writerows(rows)

    print("\n=== EXP16 POINT AUDIT ===")
    print(json.dumps(report, indent=2))
    print("\nSAVED:", out)


if __name__ == "__main__":
    main()
''', encoding="utf-8")

subprocess.run(
    [sys.executable, "-m", "py_compile", str(SCRIPT)],
    check=True
)

subprocess.run(
    [
        sys.executable,
        str(SCRIPT),
        f"--dataset_path={DS}",
        f"--ckpt={CKPT}",
    ],
    cwd=PROJECT,
    check=True
)

print("EXP17 DIAGNOSTICS PASS")

In [ ]:
#exp15-Kiểm tra và khôi phục VBB nếu cần
from pathlib import Path
import subprocess
import sys

PROJECT = Path("/kaggle/working/WeedsGalore_Ipr28")
p = PROJECT / "src/train.py"

s = p.read_text(encoding="utf-8")

print("ASCD flags:", "flags.DEFINE_float('ascd_weight'" in s)
print("VBB flags:", "flags.DEFINE_float('vbb_weight'" in s)
print("VBB import:", "from exp14_vbb import vbb_loss" in s)
print("VBB loss call:", "vbb_loss(" in s)

# Chỉ khôi phục VBB khi chưa được tích hợp
if "flags.DEFINE_float('vbb_weight'" not in s:
    assert "EXP14_VBB_TRAIN_ONLY" not in s, (
        "train.py có dấu vết patch VBB nhưng thiếu flags. "
        "Cần kiểm tra file trước khi sửa."
    )

    subprocess.run(
        [sys.executable, "src/scripts/exp14_vbb_patch.py"],
        cwd=PROJECT,
        check=True,
    )

s = p.read_text(encoding="utf-8")

assert "flags.DEFINE_float('vbb_weight'" in s
assert "from exp14_vbb import vbb_loss" in s
assert "vbb_loss(" in s

print("EXP14 VBB SOURCE READY")

In [ ]:
from pathlib import Path
import subprocess
import sys

PROJECT = Path("/kaggle/working/WeedsGalore_Ipr28")

def run(*args):
    subprocess.run(
        [sys.executable, *args],
        cwd=PROJECT,
        check=True
    )

# 1. Khôi phục registry ASCD nếu thiếu
ascd = PROJECT / "configs/exp14/exp14_ascd_arms.json"

if not ascd.is_file():
    run("src/scripts/exp14_ascd_arms.py")

# 2. Khôi phục registry VBB nếu thiếu
vbb = PROJECT / "configs/exp14/exp14_vbb_arms.json"

if not vbb.is_file():
    run("src/scripts/exp14_vbb_arms.py")

assert ascd.is_file()
assert vbb.is_file()

# 3. Tạo registry Exp15
run("src/scripts/exp15_arms.py")

# 4. Kiểm thử Exp15
run("-m", "pytest", "-q", "tests/test_exp15_losses.py")
run("-m", "py_compile", "src/train.py")

# 5. Xác minh tích hợp
s = (PROJECT / "src/train.py").read_text(encoding="utf-8")

assert "flags.DEFINE_float('vbb_weight'" in s
assert "vbb_loss(" in s

print("\nEXP15 PREFLIGHT PASS")
print("ASCD registry:", ascd.is_file())
print("VBB registry:", vbb.is_file())
print(
    "EXP15 registry:",
    (PROJECT / "configs/exp15/exp15_arms.json").is_file()
)

In [ ]:
#Exp15-A và tự chuyển sang B nếu A không đạt
from pathlib import Path
import subprocess
import sys

PROJECT = Path("/kaggle/working/WeedsGalore_Ipr28")

DS = (
    "/kaggle/input/datasets/thothanh28/"
    "weedsgalore-dataset/weedsgalore-dataset"
)

subprocess.run(
    [
        sys.executable,
        "src/scripts/exp15_seed0.py",
        f"--dataset_path={DS}",
        "--mode=auto",
    ],
    cwd=PROJECT,
    check=True
)

In [ ]:
#ZIP Exp15-A/B trên Kaggle
from pathlib import Path
import zipfile

PROJECT = Path("/kaggle/working/WeedsGalore_Ipr28")
OUT = Path("/kaggle/working/exp15_ab_backup.zip")

paths = [
    "results/exp15",
    "configs/exp15",
    "configs/exp14",
    "src/exp15_losses.py",
    "src/train.py",
    "src/train_pre_exp15.py",
    "src/scripts/exp15_patch.py",
    "src/scripts/exp15_arms.py",
    "src/scripts/exp15_seed0.py",
    "tests/test_exp15_losses.py",
]

with zipfile.ZipFile(
    OUT, "w",
    compression=zipfile.ZIP_DEFLATED,
    compresslevel=6,
    allowZip64=True
) as z:
    for rel in paths:
        p = PROJECT / rel

        if not p.exists():
            print("SKIP:", rel)
            continue

        files = [p] if p.is_file() else p.rglob("*")

        for f in files:
            if f.is_file():
                z.write(f, f.relative_to(PROJECT))

with zipfile.ZipFile(OUT) as z:
    assert z.testzip() is None
    names = z.namelist()

    for arm in ("E15_A_CW_LOVASZ", "E15_B_SPEC_SPATIAL"):
        expected = (
            f"results/exp15/{arm}/seed0/final_ema.pth"
        )
        assert expected in names, f"Thiếu: {expected}"

print("BACKUP PASS:", OUT)
print("Size:", round(OUT.stat().st_size / 1024**3, 2), "GB")
print("Files:", len(names))

In [ ]:
#cell này để tạo lại registry ASCD trước:
from pathlib import Path
import subprocess
import sys

PROJECT = Path("/kaggle/working/WeedsGalore_Ipr28")

ascd_registry = PROJECT / "configs/exp14/exp14_ascd_arms.json"

if not ascd_registry.exists():
    subprocess.run(
        [sys.executable, "src/scripts/exp14_ascd_arms.py"],
        cwd=PROJECT,
        check=True,
    )

assert ascd_registry.is_file(), "Không tạo được registry ASCD"

# Tiếp tục tạo VBB registry và kiểm thử
for cmd in [
    [sys.executable, "src/scripts/exp14_vbb_arms.py"],
    [sys.executable, "-m", "pytest", "-q", "tests/test_exp14_vbb.py"],
    [sys.executable, "-m", "py_compile", "src/train.py"],
]:
    subprocess.run(cmd, cwd=PROJECT, check=True)

print("EXP14 VBB PREFLIGHT PASS")

In [ ]:
#Thử nghiệm tiếp: Exp14-VBB (preflight)
from pathlib import Path
import subprocess
import sys

PROJECT = Path("/kaggle/working/WeedsGalore_Ipr28")

# Không chạy lại exp14_vbb_patch.py
for cmd in [
    [sys.executable, "src/scripts/exp14_vbb_arms.py"],
    [sys.executable, "-m", "pytest", "-q", "tests/test_exp14_vbb.py"],
    [sys.executable, "-m", "py_compile", "src/train.py"],
]:
    subprocess.run(cmd, cwd=PROJECT, check=True)

print("EXP14 VBB PREFLIGHT PASS")

In [ ]:
#Exp14-VBB Train và evaluate seed 0
from pathlib import Path
import subprocess
import sys

PROJECT = Path("/kaggle/working/WeedsGalore_Ipr28")

DS = (
    "/kaggle/input/datasets/thothanh28/"
    "weedsgalore-dataset/weedsgalore-dataset"
)

for arm in ("E14_B2_AMP_REF_VBB", "E14_B2_VBB"):
    subprocess.run(
        [
            sys.executable,
            "src/scripts/exp14_vbb_seed0.py",
            f"--dataset_path={DS}",
            f"--arm={arm}",
        ],
        cwd=PROJECT,
        check=True,
    )

print("EXP14 VBB SEED0 COMPLETED")

In [ ]:
from pathlib import Path
import zipfile

PROJECT = Path("/kaggle/working/WeedsGalore_Ipr28")
OUT = Path("/kaggle/working/exp14_ascd_vbb_backup.zip")

paths = [
    "results/exp14_ascd",
    "results/exp14_vbb",
    "configs/exp14",
    "src/exp14_ascd.py",
    "src/exp14_vbb.py",
    "src/train.py",
    "src/train_pre_ascd.py",
    "src/train_pre_vbb.py",
    "src/scripts/exp14_ascd_patch.py",
    "src/scripts/exp14_ascd_arms.py",
    "src/scripts/exp14_seed0.py",
    "src/scripts/exp14_diagnose.py",
    "src/scripts/exp14_vbb_patch.py",
    "src/scripts/exp14_vbb_arms.py",
    "src/scripts/exp14_vbb_seed0.py",
    "tests/test_exp14_ascd.py",
    "tests/test_exp14_vbb.py",
]

with zipfile.ZipFile(
    OUT, "w", compression=zipfile.ZIP_DEFLATED,
    compresslevel=6, allowZip64=True
) as z:
    for rel in paths:
        p = PROJECT / rel
        if not p.exists():
            print("SKIP:", rel)
            continue
        files = [p] if p.is_file() else p.rglob("*")
        for f in files:
            if f.is_file():
                z.write(f, f.relative_to(PROJECT))

with zipfile.ZipFile(OUT) as z:
    assert z.testzip() is None
    names = z.namelist()
    assert any("exp14_vbb/E14_B2_VBB/seed0/final_ema.pth" in n
               for n in names)
    print("ZIP PASS:", OUT)
    print("Files:", len(names))
    print("Size:", round(OUT.stat().st_size / 1024**3, 2), "GB")

In [ ]:
from pathlib import Path
import shutil
import subprocess
import sys

PROJECT = Path("/kaggle/working/WeedsGalore_Ipr28")
p = PROJECT / "src/train.py"
s = p.read_text(encoding="utf-8")

# Không ghi đè bản gốc
backup = PROJECT / "src/train_before_exp14_amp.py"
if not backup.exists():
    shutil.copy2(p, backup)

# 1. Đăng ký AMP nếu chưa có
if "flags.DEFINE_bool('amp'," not in s:
    anchor = "flags.DEFINE_integer('batch_size',"
    assert s.count(anchor) == 1
    pos = s.index("\n", s.index(anchor)) + 1
    s = (
        s[:pos]
        + "flags.DEFINE_bool('amp', False, 'CUDA AMP')\n"
        + s[pos:]
    )

# 2. Khởi tạo GradScaler
if "scaler = torch.amp.GradScaler(" not in s:
    anchor = "    # Metric\n"
    assert s.count(anchor) == 1
    s = s.replace(
        anchor,
        "    scaler = torch.amp.GradScaler('cuda', enabled=FLAGS.amp)\n\n"
        + anchor
    )

# 3. Autocast forward
if "with torch.autocast(" not in s:
    old = "            out, extras = forward_train(features, labels, binary_labels)"
    assert s.count(old) == 1
    s = s.replace(
        old,
        "            with torch.autocast("
        "device_type='cuda', dtype=torch.float16, enabled=FLAGS.amp):\n"
        "                out, extras = forward_train(features, labels, binary_labels)"
    )

# 4. Backward bằng GradScaler
if "scaler.scale(loss).backward()" not in s:
    old = "            loss.backward()"
    assert s.count(old) == 1
    s = s.replace(old, "            scaler.scale(loss).backward()")

# 5. Optimizer step bằng GradScaler
if "scaler.step(optimizer)" not in s:
    old = "            optimizer.step()"
    assert s.count(old) == 1
    s = s.replace(
        old,
        "            scaler.step(optimizer)\n"
        "            scaler.update()"
    )

# 6. Kiểm tra cấu trúc AMP
assert s.count("scaler.scale(loss).backward()") == 1
assert s.count("scaler.step(optimizer)") == 1
assert s.count("scaler.update()") == 1
assert "factor = lr_factor(" in s

compile(s, str(p), "exec")
p.write_text(s, encoding="utf-8")

subprocess.run(
    [sys.executable, "-m", "py_compile", str(p)],
    check=True
)

print("EXP14 AMP FIX PASS")
print("Backup:", backup)

In [ ]:
from pathlib import Path
import subprocess
import sys

PROJECT = Path("/kaggle/working/WeedsGalore_Ipr28")

for script in (
    "src/scripts/exp14_ascd_patch.py",
    "src/scripts/exp14_ascd_arms.py",
):
    subprocess.run(
        [sys.executable, script],
        cwd=PROJECT,
        check=True
    )

subprocess.run(
    [sys.executable, "-m", "pytest", "-q",
     "tests/test_exp14_ascd.py"],
    cwd=PROJECT,
    check=True
)

subprocess.run(
    [sys.executable, "-m", "py_compile", "src/train.py"],
    cwd=PROJECT,
    check=True
)

print("EXP14 ASCD PREFLIGHT PASS")

In [ ]:
#exp14 chạy seed 0:
from pathlib import Path
import subprocess
import sys

PROJECT = Path("/kaggle/working/WeedsGalore_Ipr28")

DS = (
    "/kaggle/input/datasets/thothanh28/"
    "weedsgalore-dataset/weedsgalore-dataset"
)

BASELINE = Path(
    "/kaggle/input/datasets/thanhthao28/"
    "exp12-backup/exp12/P_B2_LLR/seed0/final_ema.pth"
)

assert BASELINE.is_file(), f"Không tìm thấy checkpoint: {BASELINE}"

print("Baseline:", BASELINE)
print("Checkpoint exists:", BASELINE.is_file())

subprocess.run(
    [
        sys.executable,
        "src/scripts/exp14_seed0.py",
        f"--dataset_path={DS}",
        f"--baseline_ckpt={BASELINE}",
    ],
    cwd=PROJECT,
    check=True
)

In [ ]:
from pathlib import Path
import zipfile

PROJECT = Path("/kaggle/working/WeedsGalore_Ipr28")
ZIP = Path("/kaggle/working/exp14_ascd_backup.zip")

paths = [
    "results/exp14_ascd",
    "configs/exp14",
    "src/exp14_ascd.py",
    "src/scripts/exp14_ascd_patch.py",
    "src/scripts/exp14_ascd_arms.py",
    "src/scripts/exp14_seed0.py",
    "src/scripts/exp14_diagnose.py",
    "tests/test_exp14_ascd.py",
    "src/train.py",
    "src/exp12_config.py",
]

with zipfile.ZipFile(
    ZIP, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=6
) as z:
    for rel in paths:
        p = PROJECT / rel
        if not p.exists():
            print("SKIP:", rel)
            continue
        if p.is_file():
            z.write(p, rel)
        else:
            for f in p.rglob("*"):
                if f.is_file():
                    z.write(f, f.relative_to(PROJECT))

with zipfile.ZipFile(ZIP) as z:
    assert z.testzip() is None
    assert any(x.endswith("final_ema.pth") for x in z.namelist())
    print("BACKUP PASS:", ZIP)
    print("Files:", len(z.namelist()))
    print("Size:", round(ZIP.stat().st_size / 1024**3, 2), "GB")

In [ ]:
from pathlib import Path

PROJECT = Path("/kaggle/working/WeedsGalore_Ipr28")

assert PROJECT.exists(), f"Không tìm thấy project: {PROJECT}"
assert (PROJECT / "src/train.py").is_file()

print("PROJECT:", PROJECT)
print("PROJECT OK")

In [ ]:
from pathlib import Path

train_file = PROJECT / "src/train.py"
text = train_file.read_text()

print("=== EXP13 IMPORTS ===")
for i, line in enumerate(text.splitlines(), 1):
    if "exp13" in line.lower() or "VCF" in line:
        print(f"{i}: {line}")

print("\n=== V2 PATCH EXPECTED IMPORT ===")
print(
    "Expected import exists:",
    "from nets.exp13 import EXP13_HEADS, VCFSegFormer\n" in text
)

print("\n=== EXP13 FILES ===")
for f in [
    "src/nets/exp13.py",
    "src/nets/exp13_v2.py",
    "src/scripts/exp13_v2_apply_patch.py",
    "src/scripts/exp13_v2_make_arms.py",
]:
    p = PROJECT / f
    print(f, p.exists())

In [ ]:
#Cell cài Exp13 v2 trên Kaggle
from pathlib import Path
import subprocess
import sys

PROJECT = Path("/kaggle/working/WeedsGalore_Ipr28")

def run(script):
    print(f"\n=== RUN {script} ===", flush=True)
    subprocess.run(
        [sys.executable, script],
        cwd=PROJECT,
        check=True
    )

train = PROJECT / "src/train.py"
common = PROJECT / "src/ccmc_common.py"

v1_import = "from nets.exp13 import EXP13_HEADS, VCFSegFormer"
v2_import = "from nets.exp13_v2 import VCFContextSegFormer"

# Bước 1: kiểm tra Exp13 v1
if not all(v1_import in p.read_text(encoding="utf-8")
           for p in (train, common)):
    print("Thiếu đăng ký Exp13 v1 -> áp dụng patch v1")
    run("src/scripts/exp13_apply_patch.py")
else:
    print("Exp13 v1 imports OK")

# Bước 2: kiểm tra và áp dụng Exp13 v2
if not all(v2_import in p.read_text(encoding="utf-8")
           for p in (train, common)):
    run("src/scripts/exp13_v2_apply_patch.py")
else:
    print("Exp13 v2 imports OK")

# Bước 3: sinh cấu hình v2
run("src/scripts/exp13_v2_make_arms.py")

# Bước 4: kiểm tra
assert all(
    v2_import in p.read_text(encoding="utf-8")
    for p in (train, common)
), "V2 registration FAILED"

subprocess.run(
    [
        sys.executable, "-m", "pytest", "-q",
        "tests/test_exp13.py",
        "tests/test_exp13_v2.py"
    ],
    cwd=PROJECT,
    check=True
)

print("\nEXP13 V2 PREFLIGHT PASS")

In [ ]:
#Sửa AMP đúng cấu trúc train.py
from pathlib import Path
import subprocess
import sys
import shutil

p = PROJECT / "src/train.py"
backup = PROJECT / "src/train_before_amp.py"

s = p.read_text(encoding="utf-8")

# Chặn patch trùng
assert "flags.DEFINE_bool('amp'," not in s
assert "scaler.scale(loss).backward()" not in s

# Đảm bảo cấu trúc đúng
assert s.count("flags.DEFINE_integer('batch_size',") == 1
assert s.count("    # Metric\n") == 1
assert s.count(
    "            out, extras = forward_train(features, labels, binary_labels)"
) == 1
assert s.count("            loss.backward()") == 1
assert s.count("            optimizer.step()") == 1

shutil.copy2(p, backup)

# 1. Thêm cờ AMP
anchor = "flags.DEFINE_integer('batch_size',"
pos = s.index(anchor)
line_end = s.index("\n", pos)

s = (
    s[:line_end + 1]
    + "flags.DEFINE_bool('amp', False, 'CUDA mixed precision')\n"
    + s[line_end + 1:]
)

# 2. Khởi tạo GradScaler
s = s.replace(
    "    # Metric\n",
    "    scaler = torch.amp.GradScaler('cuda', enabled=FLAGS.amp)\n\n"
    "    # Metric\n"
)

# 3. Autocast cho model forward
old = "            out, extras = forward_train(features, labels, binary_labels)"

new = (
    "            with torch.autocast("
    "device_type='cuda', dtype=torch.float16, enabled=FLAGS.amp):\n"
    "                out, extras = forward_train(features, labels, binary_labels)"
)

s = s.replace(old, new)

# 4. Gradient scaling
s = s.replace(
    "            loss.backward()",
    "            scaler.scale(loss).backward()"
)

# 5. Giữ nguyên LR scheduler, chỉ sửa optimizer step
s = s.replace(
    "            optimizer.step()",
    "            scaler.step(optimizer)\n"
    "            scaler.update()"
)

# Kiểm tra trước khi ghi
compile(s, str(p), "exec")

assert "scaler.scale(loss).backward()" in s
assert "scaler.step(optimizer)" in s
assert "scaler.update()" in s
assert "factor = lr_factor(" in s

p.write_text(s, encoding="utf-8")

subprocess.run(
    [sys.executable, "-m", "py_compile", str(p)],
    check=True
)

print("AMP PATCH PASS")
print("Backup:", backup)

In [ ]:
#Kiểm tra AMP đã được chèn đúng
s = (PROJECT / "src/train.py").read_text()
lines = s.splitlines()

for i, line in enumerate(lines):
    if "scaler.scale(loss).backward()" in line:
        for j in range(i, min(i + 12, len(lines))):
            print(f"{j+1}: {lines[j]}")

In [ ]:
#cell tạo arm P13_B2_VCFCTX_AMP8 và train seed 0. Giữ batch size = 8, bật AMP FP16, không ensemble và không ghi đè các arm cũ.
from pathlib import Path
from exp12_config import load_json, build_cli, check_run_config
import copy
import json
import subprocess
import sys

PROJECT = Path("/kaggle/working/WeedsGalore_Ipr28")

# 1. Đường dẫn cấu hình
common_path = PROJECT / "configs/exp12/exp12_common_config.json"
old_arms_path = PROJECT / "configs/exp13/exp13_v2_arms.json"
new_arms_path = PROJECT / "configs/exp13/exp13_v2_amp_arms.json"

common = load_json(common_path)
arms = copy.deepcopy(load_json(old_arms_path))

# 2. Tạo arm riêng
base_arm = "P13_B2_VCFCTX"
arm = "P13_B2_VCFCTX_AMP8"
seed = 0

assert base_arm in arms["arms"]
assert common["common"]["batch_size"] == 8

assert "flags.DEFINE_bool('amp'," in (
    PROJECT / "src/train.py"
).read_text(), "Chưa patch AMP vào train.py"

arms["arms"][arm] = copy.deepcopy(arms["arms"][base_arm])
arms["arms"][arm]["flags"]["amp"] = True
arms["arms"][arm]["note"] = (
    "VCF-Context B2, AMP FP16, batch_size=8, seed0 exploratory"
)
arms["version"] = "exp13-v2-amp8-dev"

# 3. Lưu registry riêng, không ghi đè cấu hình cũ
content = json.dumps(arms, indent=2) + "\n"

if new_arms_path.exists():
    assert new_arms_path.read_text() == content, (
        f"Registry đã tồn tại nhưng khác nội dung: {new_arms_path}"
    )
else:
    new_arms_path.write_text(content)

# 4. Tạo output mới
out = PROJECT / "results/exp13_v2" / arm / f"seed{seed}"

assert not out.exists(), f"Run đã tồn tại: {out}"
out.mkdir(parents=True)

# 5. Tạo lệnh train
cmd = [sys.executable, "src/train.py"] + build_cli(
    common,
    arms,
    arm,
    seed,
    DS,
    str(out),
    f"{arm}_s{seed}",
    str(common_path),
    str(new_arms_path)
)

# 6. Kiểm tra trước khi chạy
assert "--batch_size=8" in cmd
assert "--amp=True" in cmd
assert "--head=vcfctx" in cmd
assert "--backbone=b2" in cmd

print("=== EXP13 V2 AMP TRAIN ===", flush=True)
print("Arm:", arm, flush=True)
print("Seed:", seed, flush=True)
print("Batch size: 8", flush=True)
print("AMP: True", flush=True)
print("Backbone: B2", flush=True)
print("Head: VCF-Context", flush=True)

# 7. Huấn luyện
subprocess.run(cmd, cwd=PROJECT, check=True)

# 8. Kiểm tra kết quả
assert (out / "final_ema.pth").is_file()
assert (out / "run_summary.json").is_file()

assert not check_run_config(
    out / "run_config.json",
    common,
    arms
)

print("\nTRAIN PASS:", arm, seed)

In [ ]:
from exp12_config import eval_cli
import subprocess
import sys

arm = "P13_B2_VCFCTX_AMP8"
seed = 0

out = PROJECT / "results/exp13_v2" / arm / f"seed{seed}"
ckpt = out / "final_ema.pth"

arms_path = PROJECT / "configs/exp13/exp13_v2_amp_arms.json"
arms = load_json(arms_path)

bins = PROJECT / "doc/exp02_results/results/exp02/instance_bins.json"

assert ckpt.is_file()
assert bins.is_file()

cmd = [sys.executable, "src/evaluate_ccmc.py"] + eval_cli(
    arms,
    arm,
    str(ckpt),
    DS,
    "val",
    str(bins),
    str(out / "val_d4"),
    f"{arm}_s{seed}",
    str(arms_path),
    tta="d4"
)

subprocess.run(cmd, cwd=PROJECT, check=True)

In [ ]:
from pathlib import Path
import subprocess
import zipfile

PROJECT = Path("/kaggle/working/WeedsGalore_Ipr28")
dst = Path("/kaggle/working/exp13_v2.zip")

# Các thư mục cần sao lưu
paths = [
    "results/exp13_v2",
]

for p in paths:
    assert (PROJECT / p).exists(), f"Không tồn tại: {p}"

# Nén, giữ nguyên cấu trúc thư mục
subprocess.run(
    ["zip", "-q", "-0", "-r", str(dst), *paths],
    cwd=PROJECT,
    check=True
)

# Kiểm tra ZIP
with zipfile.ZipFile(dst) as z:
    assert z.testzip() is None, "ZIP bị lỗi!"
    print("Số file:", len(z.namelist()))

print("BACKUP PASS:", dst)
print("Dung lượng:", round(dst.stat().st_size / 1024**3, 2), "GB")

In [ ]:
#Cell E6.0 — Biến dùng chung
import os, glob, json
os.chdir('/kaggle/working/WeedsGalore_Ipr28')
EXP = 'exp06'
OUT = f'results/{EXP}'
BINS = 'results/instance_bins.json'
os.makedirs(f'{OUT}/logs', exist_ok=True)
COMMON = (f"--dataset_path={DS} --dataset_size_train=104 --in_channels=5 --num_classes=3 "
          "--batch_size=8 --num_workers=4 --lr=0.0003 "
          "--weight_bg=0.9 --weight_crop=1.0 --weight_weed=1.3 --use_dice_loss=True --dice_weight=0.3 "
          "--epochs=100 --log_interval=50 --ckpt_interval=1000000 "
          "--pretrained_backbone=True --spectral_guided=False --use_multiscale_fusion=True --resume=True")
SEG = "--ccmc=False --segformer=True"
TRAIN_ARCH = {
    'L0':       "--ccmc=True --prior_mode=none --use_low_level_refine=False",
    'MCSDg':    f"{SEG} --use_low_level_refine=False --head=mcsd --spectral_guide=True",
    'MCSD':     f"{SEG} --use_low_level_refine=False --head=mcsd",
    'PS':       f"{SEG} --use_low_level_refine=False --head=pixelshuffle",
    'Dnone':    f"{SEG} --use_low_level_refine=True --llr_source=none",
    'LiteSent': f"{SEG} --use_low_level_refine=True --llr_source=raw",
}
EVAL_ARCH = {k: v.replace('--segformer=True ', '') for k, v in TRAIN_ARCH.items()}
for p in glob.glob('/kaggle/input/**/runs/*/best_val.pth', recursive=True):
    dst = os.path.join('runs', os.path.basename(os.path.dirname(p)))
    os.makedirs(dst, exist_ok=True)
    if not os.path.exists(f'{dst}/best_val.pth'):
        os.symlink(p, f'{dst}/best_val.pth')
print(sorted(glob.glob('runs/*/best_val.pth')))
!python src/scripts/make_instance_bins.py --dataset_path={DS} --out={BINS} > {OUT}/logs/bins.log 2>&1
assert json.load(open(BINS))['edges'] == [45.0, 147.0]

In [ ]:
#Cell E6.1 — Train seed 3, 4 (ưu tiên cặp xác nhận L0/MCSD-g)
ORDER = [('L0', 3), ('MCSDg', 3), ('L0', 4), ('MCSDg', 4), ('MCSD', 3), ('MCSD', 4),
         ('Dnone', 3), ('Dnone', 4), ('LiteSent', 3), ('LiteSent', 4), ('PS', 3), ('PS', 4)]
for model, seed in ORDER:
    run = f'{model}_s{seed}'
    if os.path.exists(f'runs/{run}/run_summary.json'):
        print('skip (done)', run); continue
    print('=====', run)
    !python src/train.py {COMMON} {TRAIN_ARCH[model]} --seed={seed} --run_name={model}__s{seed} --out_dir=runs/{run} 2>&1 | tee {OUT}/logs/train_{run}.log | grep --line-buffered -E "Saved best|Traceback|Error" | tail -n 2

In [ ]:
#Cell E6.2 — Đánh giá validation, seed 0–4
def evaluate(model, seed, arch, extra='', tag=None):
    ck = f'runs/{model}_s{seed}/best_val.pth'
    if not os.path.exists(ck):
        print('NOT RUN', model, seed); return
    name = f'{tag or model}_s{seed}'
    !python src/evaluate_ccmc.py --dataset_path={DS} --split=val --ckpt={ck} {arch} {extra} --bins_json={BINS} --boundary_px=3 --run_name={name} --out_prefix={OUT}/{name}_val > {OUT}/logs/eval_{name}.log 2>&1
    !grep -E "^mIoU|  small" {OUT}/logs/eval_{name}.log

for model in ('L0', 'MCSDg', 'MCSD', 'PS', 'Dnone', 'LiteSent'):
    for s in range(5):
        evaluate(model, s, EVAL_ARCH[model])
# Ablation: số vòng Sinkhorn lúc suy luận (cùng checkpoint MCSD-g)
for it in (1, 20):
    for s in range(5):
        evaluate('MCSDg', s, EVAL_ARCH['MCSDg'], f'--sinkhorn_iters={it}', tag=f'MCSDgIt{it}')

In [ ]:
#Cell E6.3 — Xác nhận (seed 3–4) và bảng 5 seed
def compare(a, b, gate='none', seeds=range(5), extra='', name=None):
    common = [s for s in seeds if os.path.exists(f'{OUT}/{a}_s{s}_val_per_image.csv')
              and os.path.exists(f'{OUT}/{b}_s{s}_val_per_image.csv')]
    if not common:
        print('NOT RUN', a, b); return
    fa = ','.join(f'{OUT}/{a}_s{s}_val_per_image.csv' for s in common)
    fb = ','.join(f'{OUT}/{b}_s{s}_val_per_image.csv' for s in common)
    name = name or f'{a}_vs_{b}'
    !python src/scripts/compare_runs.py --a={fa} --b={fb} --name_a={a} --name_b={b} --gate={gate} {extra} --out={OUT}/cmp_{name}.json 2>&1 | tee {OUT}/logs/cmp_{name}.log

# 1) XÁC NHẬN trên seed mới
compare('MCSDg', 'L0', 'abs', seeds=(3, 4), extra='--min_delta=0 --min_seeds_pass=2', name='CONFIRM_MCSDg_vs_L0_s34')
# 2) Bảng 5 seed
compare('MCSDg', 'L0', 'abs')
compare('MCSD', 'L0', 'abs')
compare('MCSD', 'PS', 'g4')
compare('MCSDg', 'MCSD')
compare('PS', 'L0')
compare('Dnone', 'L0')
compare('LiteSent', 'L0')
compare('MCSDg', 'Dnone')
compare('MCSDg', 'LiteSent')
# 3) Ablation số vòng Sinkhorn
compare('MCSDgIt1', 'MCSDg')
compare('MCSDgIt20', 'MCSDg')

In [ ]:
#Cell E6.4 — Bảo toàn khối lượng theo số vòng Sinkhorn (MCSD-g seed 0)
for it in (1, 5, 20):
    !python src/scripts/mass_check.py --dataset_path={DS} --ckpt=runs/MCSDg_s0/best_val.pth --ccmc=False --head=mcsd --spectral_guide=True --sinkhorn_iters={it} --out={OUT}/mass_check_MCSDg_s0_it{it}.json

In [ ]:
#Cell E6.5 — Gom và đóng gói
import shutil
for run_dir in sorted(glob.glob('runs/*')):
    if os.path.islink(f'{run_dir}/best_val.pth') or not os.path.exists(f'{run_dir}/history.jsonl'):
        continue
    dst = f'{OUT}/runs_meta/{os.path.basename(run_dir)}'
    os.makedirs(dst, exist_ok=True)
    for f in ('history.jsonl', 'run_config.json', 'run_summary.json'):
        if os.path.exists(f'{run_dir}/{f}'):
            shutil.copy(f'{run_dir}/{f}', dst)
!python src/scripts/collect_results.py --results_dir={OUT} --runs_dir=runs --out={OUT}/metrics.csv > {OUT}/logs/collect.log 2>&1
!zip -qr /kaggle/working/{EXP}_results.zip {OUT}
new_ckpts = ' '.join(p for p in sorted(glob.glob('runs/*/best_val.pth')) if not os.path.islink(p))
!zip -qr /kaggle/working/{EXP}_ckpts.zip {new_ckpts}
!ls -lh /kaggle/working/*.zip

In [ ]:
#Cell E5.0 — Biến dùng chung
import os, glob, json
os.chdir('/kaggle/working/WeedsGalore_Ipr28')
EXP = 'exp05'
OUT = f'results/{EXP}'
BINS = 'results/instance_bins.json'
os.makedirs(f'{OUT}/logs', exist_ok=True)
COMMON = (f"--dataset_path={DS} --dataset_size_train=104 --in_channels=5 --num_classes=3 "
          "--batch_size=8 --num_workers=4 --lr=0.0003 "
          "--weight_bg=0.9 --weight_crop=1.0 --weight_weed=1.3 --use_dice_loss=True --dice_weight=0.3 "
          "--epochs=100 --log_interval=50 --ckpt_interval=1000000 "
          "--pretrained_backbone=True --spectral_guided=False --use_multiscale_fusion=True --resume=True")
SEG = "--ccmc=False --segformer=True --use_low_level_refine=False"
TRAIN_ARCH = {
    'MCSD':  f"{SEG} --head=mcsd",
    'PS':    f"{SEG} --head=pixelshuffle",
    'MCSDg': f"{SEG} --head=mcsd --spectral_guide=True",
}
EVAL_ARCH = {
    'L0':       "--ccmc=True --prior_mode=none --use_low_level_refine=False",
    'Dnone':    "--ccmc=False --use_low_level_refine=True --llr_source=none",
    'LiteSent': "--ccmc=False --use_low_level_refine=True --llr_source=raw",
    'MCSD':     "--ccmc=False --use_low_level_refine=False --head=mcsd",
    'PS':       "--ccmc=False --use_low_level_refine=False --head=pixelshuffle",
    'MCSDg':    "--ccmc=False --use_low_level_refine=False --head=mcsd --spectral_guide=True",
}
for p in glob.glob('/kaggle/input/**/runs/*/best_val.pth', recursive=True):
    dst = os.path.join('runs', os.path.basename(os.path.dirname(p)))
    os.makedirs(dst, exist_ok=True)
    if not os.path.exists(f'{dst}/best_val.pth'):
        os.symlink(p, f'{dst}/best_val.pth')
print(sorted(glob.glob('runs/*/best_val.pth')))
!python src/scripts/make_instance_bins.py --dataset_path={DS} --out={BINS} > {OUT}/logs/bins.log 2>&1
assert json.load(open(BINS))['edges'] == [45.0, 147.0]

In [ ]:
#Cell E5.1 — Train (theo thứ tự ưu tiên)
ORDER = [('MCSD', 0), ('PS', 0), ('MCSD', 1), ('PS', 1), ('MCSD', 2), ('PS', 2),
         ('MCSDg', 0), ('MCSDg', 1), ('MCSDg', 2)]
for model, seed in ORDER:
    run = f'{model}_s{seed}'
    if os.path.exists(f'runs/{run}/run_summary.json'):
        print('skip (done)', run); continue
    print('=====', run)
    !python src/train.py {COMMON} {TRAIN_ARCH[model]} --seed={seed} --run_name={model}__s{seed} --out_dir=runs/{run} 2>&1 | tee {OUT}/logs/train_{run}.log | grep --line-buffered -E "Saved best|Traceback|Error" | tail -n 2

In [ ]:
#Cell E5.2 — Đánh giá validation (mô hình mới + tham chiếu)
def evaluate(run, arch):
    ck = f'runs/{run}/best_val.pth'
    if not os.path.exists(ck):
        print('NOT RUN', run); return
    !python src/evaluate_ccmc.py --dataset_path={DS} --split=val --ckpt={ck} {arch} --bins_json={BINS} --boundary_px=3 --run_name={run} --out_prefix={OUT}/{run}_val > {OUT}/logs/eval_{run}.log 2>&1
    !grep -E "^mIoU|  small" {OUT}/logs/eval_{run}.log

for model in ('L0', 'Dnone', 'LiteSent', 'MCSD', 'PS', 'MCSDg'):
    for s in (0, 1, 2):
        evaluate(f'{model}_s{s}', EVAL_ARCH[model])

In [ ]:
#Cell E5.3 — Gate và so sánh
def compare(a, b, gate='none', seeds=(0, 1, 2)):
    common = [s for s in seeds if os.path.exists(f'{OUT}/{a}_s{s}_val_per_image.csv')
              and os.path.exists(f'{OUT}/{b}_s{s}_val_per_image.csv')]
    if not common:
        print('NOT RUN', a, b); return
    fa = ','.join(f'{OUT}/{a}_s{s}_val_per_image.csv' for s in common)
    fb = ','.join(f'{OUT}/{b}_s{s}_val_per_image.csv' for s in common)
    !python src/scripts/compare_runs.py --a={fa} --b={fb} --name_a={a} --name_b={b} --gate={gate} --out={OUT}/cmp_{a}_vs_{b}.json 2>&1 | tee {OUT}/logs/cmp_{a}_vs_{b}.log

compare('MCSD', 'L0', 'abs')       # H-MCSD
compare('MCSD', 'PS', 'g4')        # H-mech
compare('PS', 'L0')                # head sub-pixel không bảo toàn khối lượng
compare('MCSD', 'Dnone')           # H-eff
compare('MCSD', 'LiteSent')
compare('MCSDg', 'MCSD')           # H-spec
compare('MCSDg', 'L0', 'abs')

In [ ]:
#Cell E5.4 — Profile (600×600, cùng quy ước)
for label in ('MCSD', 'PS', 'MCSDg'):
    !python src/scripts/profile_model.py {EVAL_ARCH[label]} --label={label} --out={OUT}/profile_{label}.json 2>&1 | grep -E "GFLOPs|latency_ms_median|params_M"

In [ ]:
%%writefile /kaggle/working/WeedsGalore_Ipr28/mass_check.py
import json, sys
import numpy as np, torch
sys.path.insert(0, 'src')
from nets.mcsd import MCSDSegFormer
from ccmc_common import load_checkpoint
from datasets import WeedsGaloreDataset
ds_path, ckpt, out = sys.argv[1], sys.argv[2], sys.argv[3]
dev = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
net = load_checkpoint(MCSDSegFormer(head='mcsd'), ckpt, dev).to(dev).eval()
ds = WeedsGaloreDataset(ds_path, None, 5, 3, False, 'val', False)
errs, pred_mass, gt_mass = [], 0.0, 0.0
with torch.no_grad():
    for i in range(len(ds)):
        img, _, lab = ds[i]
        x = torch.from_numpy(np.ascontiguousarray(img)).float()[None].to(dev)
        logp, ex = net(x, return_aux=True)
        p = logp.exp()
        q = torch.softmax(ex['occ_logits'].float(), 1)
        errs.append((torch.nn.functional.avg_pool2d(p, 4) - q).abs().max().item())
        pred_mass += p[:, 2].sum().item()
        gt_mass += float((np.asarray(lab) == 2).sum())
res = {'max_row_marginal_error': max(errs), 'mean_row_marginal_error': float(np.mean(errs)),
       'pred_weed_mass_over_gt': pred_mass / gt_mass}
print(res)
json.dump(res, open(out, 'w'), indent=2)

In [ ]:
!python mass_check.py {DS} runs/MCSD_s0/best_val.pth {OUT}/mass_check_MCSD_s0.json

In [ ]:
#Cell E5.6 — Gom và đóng gói
import shutil
for run_dir in sorted(glob.glob('runs/*')):
    if os.path.islink(f'{run_dir}/best_val.pth') or not os.path.exists(f'{run_dir}/history.jsonl'):
        continue
    dst = f'{OUT}/runs_meta/{os.path.basename(run_dir)}'
    os.makedirs(dst, exist_ok=True)
    for f in ('history.jsonl', 'run_config.json', 'run_summary.json'):
        if os.path.exists(f'{run_dir}/{f}'):
            shutil.copy(f'{run_dir}/{f}', dst)
!python src/scripts/collect_results.py --results_dir={OUT} --runs_dir=runs --out={OUT}/metrics.csv > {OUT}/logs/collect.log 2>&1
!zip -qr /kaggle/working/{EXP}_results.zip {OUT}
new_ckpts = ' '.join(p for p in sorted(glob.glob('runs/*/best_val.pth')) if not os.path.islink(p))
!zip -qr /kaggle/working/{EXP}_ckpts.zip {new_ckpts}
!ls -lh /kaggle/working/*.zip

In [ ]:
#Cell E4.0 — Biến dùng chung
import os, glob, json
os.chdir('/kaggle/working/WeedsGalore_Ipr28')
EXP = 'exp04'
OUT = f'results/{EXP}'
BINS = 'results/instance_bins.json'
os.makedirs(f'{OUT}/logs', exist_ok=True)
COMMON = (f"--dataset_path={DS} --dataset_size_train=104 --in_channels=5 --num_classes=3 "
          "--batch_size=8 --num_workers=4 --lr=0.0003 "
          "--weight_bg=0.9 --weight_crop=1.0 --weight_weed=1.3 --use_dice_loss=True --dice_weight=0.3 "
          "--epochs=100 --log_interval=50 --ckpt_interval=1000000 "
          "--pretrained_backbone=True --spectral_guided=False --use_multiscale_fusion=True --resume=True")
TRAIN_ARCH = {
    'L0':       "--ccmc=True --prior_mode=none --use_low_level_refine=False",
    'LiteSent': "--ccmc=False --segformer=True --use_low_level_refine=True --llr_source=raw",
    'Dnone':    "--ccmc=False --segformer=True --use_low_level_refine=True --llr_source=none",
    'DF1':      "--ccmc=False --segformer=True --use_low_level_refine=True --llr_source=f1",
}
EVAL_ARCH = {k: v.replace('--segformer=True ', '') for k, v in TRAIN_ARCH.items()}
for p in glob.glob('/kaggle/input/**/runs/*/best_val.pth', recursive=True):
    dst = os.path.join('runs', os.path.basename(os.path.dirname(p)))
    os.makedirs(dst, exist_ok=True)
    if not os.path.exists(f'{dst}/best_val.pth'):
        os.symlink(p, f'{dst}/best_val.pth')
print(sorted(glob.glob('runs/*/best_val.pth')))
!python src/scripts/make_instance_bins.py --dataset_path={DS} --out={BINS} > {OUT}/logs/bins.log 2>&1
assert json.load(open(BINS))['edges'] == [45.0, 147.0]

In [ ]:
#Cell E4.1 — Train
ORDER = [('Dnone', 0), ('DF1', 0), ('LiteSent', 1), ('Dnone', 1), ('DF1', 1),
         ('LiteSent', 2), ('Dnone', 2), ('DF1', 2)]
for model, seed in ORDER:
    run = f'{model}_s{seed}'
    if os.path.exists(f'runs/{run}/best_val.pth') and (os.path.islink(f'runs/{run}/best_val.pth')
                                                       or os.path.exists(f'runs/{run}/run_summary.json')):
        print('skip (done)', run); continue
    print('=====', run)
    !python src/train.py {COMMON} {TRAIN_ARCH[model]} --seed={seed} --run_name={model}__s{seed} --out_dir=runs/{run} 2>&1 | tee {OUT}/logs/train_{run}.log | grep --line-buffered -E "Saved best|Traceback|Error" | tail -n 2

In [ ]:
#Cell E4.2 — Đánh giá validation (cả L0 0–2 và LiteSent s0 lấy từ các Exp trước)
def evaluate(run, arch):
    ck = f'runs/{run}/best_val.pth'
    if not os.path.exists(ck):
        print('NOT RUN', run); return
    !python src/evaluate_ccmc.py --dataset_path={DS} --split=val --ckpt={ck} {arch} --bins_json={BINS} --boundary_px=3 --run_name={run} --out_prefix={OUT}/{run}_val > {OUT}/logs/eval_{run}.log 2>&1
    !grep -E "^mIoU|  small" {OUT}/logs/eval_{run}.log

for model in ('L0', 'LiteSent', 'Dnone', 'DF1'):
    for s in (0, 1, 2):
        evaluate(f'{model}_s{s}', EVAL_ARCH[model])

In [ ]:
#Cell E4.3 — So sánh (bootstrap phân tầng, 3 seed)
def compare(a, b, gate='none', seeds=(0, 1, 2)):
    common = [s for s in seeds if os.path.exists(f'{OUT}/{a}_s{s}_val_per_image.csv')
              and os.path.exists(f'{OUT}/{b}_s{s}_val_per_image.csv')]
    if not common:
        print('NOT RUN', a, b); return
    fa = ','.join(f'{OUT}/{a}_s{s}_val_per_image.csv' for s in common)
    fb = ','.join(f'{OUT}/{b}_s{s}_val_per_image.csv' for s in common)
    !python src/scripts/compare_runs.py --a={fa} --b={fb} --name_a={a} --name_b={b} --gate={gate} --out={OUT}/cmp_{a}_vs_{b}.json 2>&1 | tee {OUT}/logs/cmp_{a}_vs_{b}.log

compare('LiteSent', 'L0')      # tổng lợi ích của LLR (3 seed)
compare('Dnone', 'L0')         # chỉ tăng độ phân giải decoder
compare('LiteSent', 'Dnone')   # giá trị thêm của chi tiết từ băng thô
compare('LiteSent', 'DF1')     # băng thô so với F1
compare('DF1', 'Dnone')        # F1 có thêm gì không

In [ ]:
#Cell E4.4 — Profile (600×600, cùng quy ước)
for label in ('Dnone', 'DF1', 'LiteSent'):
    !python src/scripts/profile_model.py {EVAL_ARCH[label]} --label={label} --out={OUT}/profile_{label}.json 2>&1 | grep -E "GFLOPs|latency_ms_median|params_M"

In [ ]:
#Cell E4.5 — Gom và đóng gói
import shutil
for run_dir in sorted(glob.glob('runs/*')):
    if os.path.islink(f'{run_dir}/best_val.pth') or not os.path.exists(f'{run_dir}/history.jsonl'):
        continue
    dst = f'{OUT}/runs_meta/{os.path.basename(run_dir)}'
    os.makedirs(dst, exist_ok=True)
    for f in ('history.jsonl', 'run_config.json', 'run_summary.json'):
        if os.path.exists(f'{run_dir}/{f}'):
            shutil.copy(f'{run_dir}/{f}', dst)
!python src/scripts/collect_results.py --results_dir={OUT} --runs_dir=runs --out={OUT}/metrics.csv > {OUT}/logs/collect.log 2>&1
!zip -qr /kaggle/working/{EXP}_results.zip {OUT}
new_ckpts = ' '.join(p for p in sorted(glob.glob('runs/*/best_val.pth')) if not os.path.islink(p))
!zip -qr /kaggle/working/{EXP}_ckpts.zip {new_ckpts}
!ls -lh /kaggle/working/*.zip

In [ ]:
#Cell E3.0 — Biến dùng chung, nối checkpoint cũ
import os, glob, json
os.chdir('/kaggle/working/WeedsGalore_Ipr28')
EXP = 'exp03'
OUT = f'results/{EXP}'
BINS = 'results/instance_bins.json'
os.makedirs(f'{OUT}/logs', exist_ok=True)
COMMON = (f"--dataset_path={DS} --dataset_size_train=104 --in_channels=5 --num_classes=3 "
          "--batch_size=8 --num_workers=4 --lr=0.0003 "
          "--weight_bg=0.9 --weight_crop=1.0 --weight_weed=1.3 --use_dice_loss=True --dice_weight=0.3 "
          "--epochs=100 --log_interval=50 --ckpt_interval=1000000 "
          "--pretrained_backbone=True --spectral_guided=False --use_multiscale_fusion=True --resume=True")
BASE = "--ccmc=True --use_low_level_refine=False"
LAM_LEARN = "--lam_init=0.5 --lam_lr=0.01 --learn_lambda=True"
LAM_FIXED1 = "--lam_init=0.999 --learn_lambda=False"   # sigmoid(logit(0.999)) ≈ 1
ARCH = {
    'L0':    f"{BASE} --prior_mode=none",
    'C3a':   f"{BASE} --prior_mode=gt_weed --injection=ccmc_sr {LAM_LEARN}",
    'C2a':   f"{BASE} --prior_mode=gt_veg  --injection=ccmc_sr {LAM_LEARN}",
    'C3f':   f"{BASE} --prior_mode=gt_weed --injection=ccmc_sr {LAM_FIXED1}",
    'C3K5a': f"{BASE} --prior_mode=gt_weed --injection=pre_sr_gate {LAM_LEARN}",
}
# Cờ chỉ dùng cho train (evaluate_ccmc không có lam_*)
EVAL_ARCH = {k: ' '.join(t for t in v.split() if not t.startswith(('--lam_', '--learn_lambda')))
             for k, v in ARCH.items()}
# Nối checkpoint Exp01/Exp02 (chỉ đọc)
for p in glob.glob('/kaggle/input/**/runs/*/best_val.pth', recursive=True):
    dst = os.path.join('runs', os.path.basename(os.path.dirname(p)))
    os.makedirs(dst, exist_ok=True)
    if not os.path.exists(f'{dst}/best_val.pth'):
        os.symlink(p, f'{dst}/best_val.pth')
print(sorted(glob.glob('runs/*/best_val.pth')))

In [ ]:
#Cell E3.1 — Bin (phải trùng [45, 147])
!python src/scripts/make_instance_bins.py --dataset_path={DS} --out={BINS} > {OUT}/logs/bins.log 2>&1
assert json.load(open(BINS))['edges'] == [45.0, 147.0]

In [ ]:
# Cell E3.2 — Train (theo thứ tự ưu tiên)
ORDER = [('L0', 2), ('C3a', 0), ('C2a', 0), ('C3f', 0), ('C3a', 1), ('C2a', 1), ('C3a', 2), ('C2a', 2),
         ('C3K5a', 0), ('C3K5a', 1), ('C3K5a', 2), ('C3f', 1), ('C3f', 2)]
for model, seed in ORDER:
    run = f'{model}_s{seed}'
    if os.path.exists(f'runs/{run}/run_summary.json'):
        print('skip (done)', run); continue
    print('=====', run)
    !python src/train.py {COMMON} {ARCH[model]} --seed={seed} --run_name={model}__s{seed} --out_dir=runs/{run} 2>&1 | tee {OUT}/logs/train_{run}.log | grep --line-buffered -E "Saved best|Traceback|Error" | tail -n 2
    !grep -E "lambdas" {OUT}/logs/train_{run}.log | tail -n 1

In [ ]:
#Cell E3.3 — Quỹ đạo λ (kiểm tra λ đã thực sự di chuyển)
for h in sorted(glob.glob('runs/*/history.jsonl')):
    rows = [json.loads(l) for l in open(h)]
    if not rows[0]['lambdas']:
        continue
    lam = [r['lambdas'] for r in rows]; best = rows[-1]['best_epoch']
    print(f"{h.split('/')[1]:10s} ep0={[round(v,3) for v in lam[0]]} ep10={[round(v,3) for v in lam[min(10,len(lam)-1)]]} "
          f"ep50={[round(v,3) for v in lam[min(50,len(lam)-1)]]} best(ep{best})={[round(v,3) for v in lam[best]]}")

In [ ]:
#Cell E3.4 — Đánh giá validation
def evaluate(run, arch, extra='', tag=None):
    ck = f'runs/{run}/best_val.pth'
    if not os.path.exists(ck):
        print('NOT RUN', run); return
    tag = tag or run
    !python src/evaluate_ccmc.py --dataset_path={DS} --split=val --ckpt={ck} {arch} {extra} --bins_json={BINS} --boundary_px=3 --run_name={tag} --out_prefix={OUT}/{tag}_val > {OUT}/logs/eval_{tag}.log 2>&1
    !grep -E "^mIoU|  small" {OUT}/logs/eval_{tag}.log

for s in (0, 1, 2):
    evaluate(f'L0_s{s}', EVAL_ARCH['L0'])
for model, seed in ORDER:
    if model != 'L0':
        evaluate(f'{model}_s{seed}', EVAL_ARCH[model])

In [ ]:
#Cell E3.5 — Cơ chế có hoạt động không? (cùng checkpoint, chỉ đổi prior lúc suy luận)
for model in ('C3a', 'C3f'):
    for s in (0, 1, 2):
        run = f'{model}_s{s}'
        evaluate(run, EVAL_ARCH[model], '--prior_override=uniform', tag=f'{run}_ovr-uniform')
for noise in ('dilate:3', 'drop:0.3', 'shift:4'):
    evaluate('C3a_s0', EVAL_ARCH['C3a'], f'--prior_noise={noise}', tag=f"C3a_s0_noise-{noise.replace(':','')}")

In [ ]:
#Cell E3.6 — So sánh (bootstrap phân tầng theo seed + ảnh) và A/A test
def compare(a, b, gate, seeds=(0, 1, 2), suffix_a='', suffix_b='', name=None, extra=''):
    common = [s for s in seeds if os.path.exists(f'{OUT}/{a}_s{s}{suffix_a}_val_per_image.csv')
              and os.path.exists(f'{OUT}/{b}_s{s}{suffix_b}_val_per_image.csv')]
    if not common:
        print('NOT RUN', a, b); return
    fa = ','.join(f'{OUT}/{a}_s{s}{suffix_a}_val_per_image.csv' for s in common)
    fb = ','.join(f'{OUT}/{b}_s{s}{suffix_b}_val_per_image.csv' for s in common)
    name = name or f'{a}{suffix_a}_vs_{b}{suffix_b}'
    !python src/scripts/compare_runs.py --a={fa} --b={fb} --name_a={a}{suffix_a} --name_b={b}{suffix_b} --gate={gate} {extra} --out={OUT}/cmp_{name}.json 2>&1 | tee {OUT}/logs/cmp_{name}.log

# 1) Cơ chế: prior GT so với prior đồng đều trên CÙNG checkpoint
compare('C3a', 'C3a', 'none', suffix_b='_ovr-uniform', name='C3a_activity')
compare('C3f', 'C3f', 'none', suffix_b='_ovr-uniform', name='C3f_activity')
# 2) G2-screen
compare('C3a', 'C2a', 'screen')
# 3) Trần mô tả và đối chứng cùng leakage
compare('C3a', 'L0', 'none'); compare('C3f', 'L0', 'none'); compare('C2a', 'L0', 'none')
compare('C3a', 'C3K5a', 'g4')
# 4) A/A test: cùng mô hình L0, khác seed, chỉ bootstrap theo ảnh
!python src/scripts/compare_runs.py --a={OUT}/L0_s0_val_per_image.csv --b={OUT}/L0_s1_val_per_image.csv --name_a=L0_s0 --name_b=L0_s1 --seed_bootstrap=False --out={OUT}/cmp_AA_L0s0_vs_L0s1.json 2>&1 | tee {OUT}/logs/cmp_AA.log

In [ ]:
#Cell E3.7 — Gom kết quả và đóng gói
import shutil
for run_dir in sorted(glob.glob('runs/*')):
    if not os.path.exists(f'{run_dir}/history.jsonl') or os.path.islink(f'{run_dir}/best_val.pth'):
        continue
    dst = f'{OUT}/runs_meta/{os.path.basename(run_dir)}'
    os.makedirs(dst, exist_ok=True)
    for f in ('history.jsonl', 'run_config.json', 'run_summary.json'):
        if os.path.exists(f'{run_dir}/{f}'):
            shutil.copy(f'{run_dir}/{f}', dst)
!python src/scripts/collect_results.py --results_dir={OUT} --runs_dir=runs --out={OUT}/metrics.csv > {OUT}/logs/collect.log 2>&1
!zip -qr /kaggle/working/{EXP}_results.zip {OUT}
new_ckpts = ' '.join(p for p in sorted(glob.glob('runs/*/best_val.pth')) if not os.path.islink(p))
!zip -qr /kaggle/working/{EXP}_ckpts.zip {new_ckpts}
!ls -lh /kaggle/working/*.zip

In [ ]:
#Cell E2.0 — Biến dùng chung và khôi phục checkpoint Exp01
import os, glob, json, zipfile
os.chdir('/kaggle/working/WeedsGalore_Ipr28')
EXP = 'exp02'
OUT = f'results/{EXP}'
BINS = 'results/instance_bins.json'
os.makedirs(f'{OUT}/logs', exist_ok=True)
COMMON = (f"--dataset_path={DS} --dataset_size_train=104 --in_channels=5 --num_classes=3 "
          "--batch_size=8 --num_workers=4 --lr=0.0003 "
          "--weight_bg=0.9 --weight_crop=1.0 --weight_weed=1.3 --use_dice_loss=True --dice_weight=0.3 "
          "--epochs=100 --log_interval=50 --ckpt_interval=1000000 "
          "--pretrained_backbone=True --spectral_guided=False --use_multiscale_fusion=True --resume=True")
L0_ARCH = "--ccmc=True --prior_mode=none --use_low_level_refine=False"
ARCH = {
    'C2':   "--ccmc=True --prior_mode=gt_veg  --injection=ccmc_sr      --use_low_level_refine=False",
    'C3':   "--ccmc=True --prior_mode=gt_weed --injection=ccmc_sr      --use_low_level_refine=False",
    'C3K5': "--ccmc=True --prior_mode=gt_weed --injection=pre_sr_gate  --use_low_level_refine=False",
    'C3K6': "--ccmc=True --prior_mode=gt_weed --injection=post_sr_gate --use_low_level_refine=False",
}
# Exp01 checkpoints (zip hoặc thư mục đã giải nén trong /kaggle/input)
for z in glob.glob('/kaggle/input/**/exp01_ckpts.zip', recursive=True):
    zipfile.ZipFile(z).extractall('.')
for p in glob.glob('/kaggle/input/**/runs/*/best_val.pth', recursive=True):
    dst = os.path.join('runs', os.path.basename(os.path.dirname(p)))
    os.makedirs(dst, exist_ok=True)
    if not os.path.exists(f'{dst}/best_val.pth'):
        os.symlink(p, f'{dst}/best_val.pth')
print('Exp01 ckpts:', sorted(glob.glob('runs/*/best_val.pth')))

In [ ]:
#Cell E2.1 — Bin kích thước (phải trùng Exp01: edges = [45, 147])
!python src/scripts/make_instance_bins.py --dataset_path={DS} --out={BINS} > {OUT}/logs/bins.log 2>&1
edges = json.load(open(BINS))['edges']
print('edges', edges)
assert edges == [45.0, 147.0], 'Bin khác Exp01 -> dừng lại và gửi logs/bins.log'
!cp {BINS} {OUT}/instance_bins.json

In [ ]:
#Cell E2.2 — Train C2, C3, C3-K5, C3-K6 (seed 0 rồi seed 1)
ORDER = [('C3', 0), ('C2', 0), ('C3K5', 0), ('C3', 1), ('C2', 1), ('C3K6', 0), ('C3K5', 1), ('C3K6', 1)]
for model, seed in ORDER:
    run = f'{model}_s{seed}'
    if os.path.exists(f'runs/{run}/run_summary.json'):
        print('skip (done)', run); continue
    print('=====', run)
    !python src/train.py {COMMON} {ARCH[model]} --seed={seed} --run_name={model}__s{seed} --out_dir=runs/{run} 2>&1 | tee {OUT}/logs/train_{run}.log | grep --line-buffered -E "Saved best|lambdas|Traceback|Error" | tail -n 3

In [ ]:
#Cell E2.3 — Đánh giá validation (Boundary IoU 3 px)
def evaluate(run, arch, extra=''):
    ck = f'runs/{run}/best_val.pth'
    if not os.path.exists(ck):
        print('NOT RUN', run); return
    tag = run + (extra.replace('--prior_noise=', '_noise-').replace(':', '') if extra else '')
    !python src/evaluate_ccmc.py --dataset_path={DS} --split=val --ckpt={ck} {arch} {extra} --bins_json={BINS} --boundary_px=3 --run_name={tag} --out_prefix={OUT}/{tag}_val > {OUT}/logs/eval_{tag}.log 2>&1
    !grep -E "mIoU|small|GT prior" {OUT}/logs/eval_{tag}.log

# L0 / LiteSent đánh giá lại (BIoU 3 px), từ checkpoint Exp01
for seed in (0, 1):
    evaluate(f'L0_s{seed}', L0_ARCH)
evaluate('LiteSent_s0', '--ccmc=False --use_low_level_refine=True')
# Oracle models (GT leakage)
for model, seed in ORDER:
    evaluate(f'{model}_s{seed}', ARCH[model])

In [ ]:
#Cell E2.4 — Độ nhạy với prior GT bị nhiễu (C3 seed 0, cùng checkpoint)
for noise in ('dilate:3', 'erode:1', 'drop:0.3', 'shift:4'):
    evaluate('C3_s0', ARCH['C3'], f'--prior_noise={noise}')

In [ ]:
# Cell E2.4b — λ cuối và quỹ đạo λ của các mô hình oracle
import json, glob
for h in sorted(glob.glob('runs/C*/history.jsonl')):
    rows = [json.loads(l) for l in open(h)]
    lam = [r['lambdas'] for r in rows]
    best = rows[-1]['best_epoch']
    print(f"{h.split('/')[1]:9s} λ@ep0={[round(v,3) for v in lam[0]]} "
          f"λ@ep50={[round(v,3) for v in lam[min(50,len(lam)-1)]]} "
          f"λ@best(ep{best})={[round(v,3) for v in lam[best]]} λ@last={[round(v,3) for v in lam[-1]]}")

In [ ]:
#Cell E2.5 — So sánh paired và gate
def compare(a, b, gate, seeds=(0, 1)):
    # chỉ dùng các seed có ở CẢ HAI mô hình (paired)
    common = [s for s in seeds if os.path.exists(f'{OUT}/{a}_s{s}_val_per_image.csv')
              and os.path.exists(f'{OUT}/{b}_s{s}_val_per_image.csv')]
    if not common:
        print('NOT RUN', a, 'vs', b); return
    fa = ','.join(f'{OUT}/{a}_s{s}_val_per_image.csv' for s in common)
    fb = ','.join(f'{OUT}/{b}_s{s}_val_per_image.csv' for s in common)
    name = f'{a}_vs_{b}'
    !python src/scripts/compare_runs.py --a={fa} --b={fb} --name_a={a} --name_b={b} --gate={gate} --out={OUT}/cmp_{name}.json 2>&1 | tee {OUT}/logs/cmp_{name}.log

compare('C3', 'C2', 'screen')      # G2-screen (một chiều)
compare('C3', 'C3K5', 'g4')        # cùng leakage: CCMC-SR so với gate trước SR
compare('C3', 'C3K6', 'g4')        # cùng leakage: CCMC-SR so với gate sau SR
compare('C3', 'L0', 'none')        # biên độ trần, chỉ mô tả
compare('C2', 'L0', 'none')

In [ ]:
#Cell E2.6 — Gom kết quả và đóng gói
import shutil
for run_dir in sorted(glob.glob('runs/*')):
    if not os.path.exists(f'{run_dir}/run_config.json'):
        continue
    dst = f'{OUT}/runs_meta/{os.path.basename(run_dir)}'
    os.makedirs(dst, exist_ok=True)
    for f in ('history.jsonl', 'run_config.json', 'run_summary.json'):
        if os.path.exists(f'{run_dir}/{f}'):
            shutil.copy(f'{run_dir}/{f}', dst)
!python src/scripts/collect_results.py --results_dir={OUT} --runs_dir=runs --out={OUT}/metrics.csv
!zip -qr /kaggle/working/{EXP}_results.zip {OUT}
!zip -qr /kaggle/working/{EXP}_ckpts.zip runs/C*/best_val.pth
!ls -lh /kaggle/working/*.zip

In [ ]:
#Cell E1.0 — Biến dùng chung
import os
os.chdir('/kaggle/working/WeedsGalore_Ipr28')
EXP = 'exp01'
OUT = f'results/{EXP}'
BINS = 'results/instance_bins.json'   # bin cố định cho MỌI thí nghiệm sau này
os.makedirs(f'{OUT}/logs', exist_ok=True)
COMMON = (f"--dataset_path={DS} --dataset_size_train=104 --in_channels=5 --num_classes=3 "
          "--batch_size=8 --num_workers=4 --lr=0.0003 "
          "--weight_bg=0.9 --weight_crop=1.0 --weight_weed=1.3 --use_dice_loss=True --dice_weight=0.3 "
          "--epochs=100 --log_interval=50 --ckpt_interval=1000000 "
          "--pretrained_backbone=True --spectral_guided=False --use_multiscale_fusion=True --resume=True")
L0_ARCH = "--ccmc=True --prior_mode=none --use_low_level_refine=False"
LITE_ARCH = "--ccmc=False --segformer=True --use_low_level_refine=True"
print(COMMON)

In [ ]:
#Cell E1.1 — Bin kích thước cỏ (tam phân vị của train, khóa trước khi xem mô hình)
!python src/scripts/make_instance_bins.py --dataset_path={DS} --out={BINS} 2>&1 | tee {OUT}/logs/bins.log | head -n 60
!cp {BINS} {OUT}/instance_bins.json

In [ ]:
#Cell E1.2 — Train L0 seed 0 và seed 1
for seed in (0, 1):
    run = f'L0_s{seed}'
    !python src/train.py {COMMON} {L0_ARCH} --seed={seed} --run_name=L0__s{seed} --out_dir=runs/{run} 2>&1 | tee {OUT}/logs/train_{run}.log | grep --line-buffered -E "Val mIoU|Saved best|Traceback|Error|FLOPs|Total Parameters"

In [ ]:
#Cell E1.3 — Train LiteSent seed 0 (cùng pipeline, để làm Diag-D0 trên checkpoint có LLR)
!python src/train.py {COMMON} {LITE_ARCH} --seed=0 --run_name=LiteSent__s0 --out_dir=runs/LiteSent_s0 2>&1 | tee {OUT}/logs/train_LiteSent_s0.log | grep --line-buffered -E "Val mIoU|Saved best|Traceback|Error|FLOPs|Total Parameters"

In [ ]:
#Cell E1.4 — Diag-D0: đánh giá validation (IoU, Boundary IoU, recall theo bin, theo ngày)
for seed in (0, 1):
    !python src/evaluate_ccmc.py --dataset_path={DS} --split=val --ckpt=runs/L0_s{seed}/best_val.pth {L0_ARCH} --bins_json={BINS} --run_name=L0__s{seed} --out_prefix={OUT}/L0_s{seed}_val 2>&1 | tee {OUT}/logs/eval_L0_s{seed}.log
if os.path.exists('runs/LiteSent_s0/best_val.pth'):
    !python src/evaluate_ccmc.py --dataset_path={DS} --split=val --ckpt=runs/LiteSent_s0/best_val.pth --ccmc=False --use_low_level_refine=True --bins_json={BINS} --run_name=LiteSent__s0 --out_prefix={OUT}/LiteSent_s0_val 2>&1 | tee {OUT}/logs/eval_LiteSent_s0.log

In [ ]:
#Cell E1.5 — Diag-D0: recall cỏ nhỏ so với cỏ lớn trong cùng mô hình (bootstrap theo ảnh)
import csv, json, numpy as np
def small_vs_large(path, n_boot=10000, seed=0):
    rows = list(csv.DictReader(open(path)))
    g = lambda k: np.array([float(r[k]) for r in rows])
    out = {}
    for t in ('50', '30'):
        tp_s, n_s, tp_l, n_l = g(f'tp{t}_small'), g('n_small'), g(f'tp{t}_large'), g('n_large')
        point = 100 * (tp_l.sum() / n_l.sum() - tp_s.sum() / n_s.sum())
        rng = np.random.default_rng(seed)
        W = rng.multinomial(len(rows), np.full(len(rows), 1 / len(rows)), size=n_boot)
        with np.errstate(invalid='ignore', divide='ignore'):
            boot = 100 * ((W @ tp_l) / (W @ n_l) - (W @ tp_s) / (W @ n_s))
        boot = boot[np.isfinite(boot)]
        out[f'recall{t}_large_minus_small'] = {'delta': point, 'ci95': np.percentile(boot, [2.5, 97.5]).tolist()}
    return out
d0 = {}
for name in ['L0_s0', 'L0_s1', 'LiteSent_s0']:
    p = f'{OUT}/{name}_val_per_image.csv'
    if os.path.exists(p):
        d0[name] = small_vs_large(p)
print(json.dumps(d0, indent=2))
json.dump(d0, open(f'{OUT}/d0_small_vs_large.json', 'w'), indent=2)

In [ ]:
#Cell E1.6 — LiteSent so với L0 (paired bootstrap, seed 0, mang tính bổ trợ)
if os.path.exists(f'{OUT}/LiteSent_s0_val_per_image.csv'):
    !python src/scripts/compare_runs.py --a={OUT}/LiteSent_s0_val_per_image.csv --b={OUT}/L0_s0_val_per_image.csv --name_a=LiteSent --name_b=L0 --out={OUT}/cmp_LiteSent_vs_L0_s0.json 2>&1 | tee {OUT}/logs/cmp_LiteSent_vs_L0.log

In [ ]:
#Cell E1.7 — Diag-D2: linear probe trước/sau SR (stage 1–3)
for seed in (0, 1):
    !python src/scripts/probe_sr.py --dataset_path={DS} --ckpt=runs/L0_s{seed}/best_val.pth {L0_ARCH} --bins_json={BINS} --out_prefix={OUT}/d2_probe_L0_s{seed} 2>&1 | tee {OUT}/logs/probe_L0_s{seed}.log | tail -n 60

In [ ]:
#Cell E1.8 — Profile (cùng quy ước: 1×5×600×600, fp32, batch 1, T4)
profiles = {
    'L0': L0_ARCH,
    'LiteSent': LITE_ARCH.replace('--segformer=True ', ''),
    'L3': '--ccmc=True --prior_mode=learned_weed --injection=ccmc_sr --use_low_level_refine=False',
    'K5': '--ccmc=True --prior_mode=learned_weed --injection=pre_sr_gate --use_low_level_refine=False',
    'K6': '--ccmc=True --prior_mode=learned_weed --injection=post_sr_gate --use_low_level_refine=False',
}
for label, arch in profiles.items():
    !python src/scripts/profile_model.py {arch} --label={label} --out={OUT}/profile_{label}.json 2>&1 | tail -n 3

In [ ]:
#Cell E1.9 — Gom kết quả và đóng gói để gửi review
import shutil, glob
for run_dir in sorted(glob.glob('runs/*')):
    dst = f'{OUT}/runs_meta/{os.path.basename(run_dir)}'
    os.makedirs(dst, exist_ok=True)
    for f in ('history.jsonl', 'run_config.json', 'run_summary.json'):
        if os.path.exists(f'{run_dir}/{f}'):
            shutil.copy(f'{run_dir}/{f}', dst)
shutil.copytree('results/env', f'{OUT}/env', dirs_exist_ok=True)
!python src/scripts/collect_results.py --results_dir={OUT} --runs_dir=runs --out={OUT}/metrics.csv
!cd /kaggle/working/WeedsGalore_Ipr28 && zip -qr /kaggle/working/{EXP}_results.zip {OUT} {BINS}
!cd /kaggle/working/WeedsGalore_Ipr28 && zip -qr /kaggle/working/{EXP}_ckpts.zip runs/*/best_val.pth
!ls -lh /kaggle/working/*.zip

In [ ]:
!pip install thop

In [ ]:
#Cell chạy E1/E2:
!python /kaggle/working/WeedsGalore_Ipr28/src/train.py \
    --dataset_path=/kaggle/working/weedsgalore-dataset \
    --dataset_size_train=104 \
    --segformer=True \
    --spectral_guided=False \
    --use_spectral_indices=False \
    --use_attention=False \
    --dlv3p_do=False \
    --in_channels=5 \
    --num_classes=3 \
    --pretrained_backbone=True \
    --use_low_level_refine=False \
    --batch_size=8 \
    --num_workers=4 \
    --lr=0.0003 \
    --weight_bg=0.9 \
    --weight_crop=1.0 \
    --weight_weed=1.3 \
    --use_dice_loss=True \
    --dice_weight=0.3 \
    --epochs=100 \
    --seed=0 \
    --out_dir=/kaggle/working/out_dir_segformer_no_refine_seed0 \
    --log_interval=25 \
    --ckpt_interval=100

In [ ]:
!python /kaggle/working/WeedsGalore_Ipr28/src/evaluate.py \
    --dataset_path=/kaggle/working/weedsgalore-dataset \
    --split=test \
    --ckpt=/kaggle/working/out_dir_segformer_no_refine_seed0/best_val.pth \
    --segformer=True \
    --pretrained_backbone=True \
    --use_low_level_refine=False \
    --spectral_guided=False \
    --in_channels=5 \
    --num_classes=3 \
    --temperature=1.0

In [ ]:
#Cell E5, RGB-only nhiều seed:
for seed in [0, 1, 2, 3, 4]:
    !python /kaggle/working/WeedsGalore_Ipr28/src/train.py \
        --dataset_path=/kaggle/working/weedsgalore-dataset \
        --dataset_size_train=104 \
        --segformer=True \
        --spectral_guided=False \
        --use_spectral_indices=False \
        --use_attention=False \
        --dlv3p_do=False \
        --in_channels=3 \
        --num_classes=3 \
        --pretrained_backbone=True \
        --use_low_level_refine=True \
        --batch_size=8 \
        --num_workers=4 \
        --lr=0.0003 \
        --weight_bg=0.9 \
        --weight_crop=1.0 \
        --weight_weed=1.3 \
        --use_dice_loss=True \
        --dice_weight=0.3 \
        --epochs=100 \
        --seed={seed} \
        --out_dir=/kaggle/working/out_dir_rgb_seed{seed} \
        --log_interval=25 \
        --ckpt_interval=100

In [ ]:
!python /kaggle/working/WeedsGalore_Ipr28/src/evaluate.py \
    --dataset_path=/kaggle/working/weedsgalore-dataset \
    --split=val \
    --ckpt=/kaggle/working/out_dir_rgb_seed0/best_val.pth \
    --segformer=True \
    --pretrained_backbone=True \
    --use_low_level_refine=False \
    --spectral_guided=False \
    --in_channels=3 \
    --num_classes=3 \
    --use_low_level_refine=True \
    --temperature=1.0

In [ ]:
!python /kaggle/working/WeedsGalore_Ipr28/src/evaluate.py \
    --dataset_path=/kaggle/working/weedsgalore-dataset \
    --split=test \
    --ckpt=/kaggle/working/out_dir_rgb_seed0/best_val.pth \
    --segformer=True \
    --pretrained_backbone=True \
    --use_low_level_refine=False \
    --spectral_guided=False \
    --in_channels=3 \
    --num_classes=3 \
    --use_low_level_refine=True \
    --temperature=1.0

In [ ]:
!python /kaggle/working/WeedsGalore_Ipr28/src/evaluate.py \
    --dataset_path=/kaggle/working/weedsgalore-dataset \
    --split=val \
    --ckpt=/kaggle/working/out_dir_rgb_seed1/best_val.pth \
    --segformer=True \
    --pretrained_backbone=True \
    --use_low_level_refine=False \
    --spectral_guided=False \
    --in_channels=3 \
    --num_classes=3 \
    --use_low_level_refine=True \
    --temperature=1.0

In [ ]:
!python /kaggle/working/WeedsGalore_Ipr28/src/evaluate.py \
    --dataset_path=/kaggle/working/weedsgalore-dataset \
    --split=test \
    --ckpt=/kaggle/working/out_dir_rgb_seed1/best_val.pth \
    --segformer=True \
    --pretrained_backbone=True \
    --use_low_level_refine=False \
    --spectral_guided=False \
    --in_channels=3 \
    --num_classes=3 \
    --use_low_level_refine=True \
    --temperature=1.0

In [ ]:
!python /kaggle/working/WeedsGalore_Ipr28/src/evaluate.py \
    --dataset_path=/kaggle/working/weedsgalore-dataset \
    --split=val \
    --ckpt=/kaggle/working/out_dir_rgb_seed2/best_val.pth \
    --segformer=True \
    --pretrained_backbone=True \
    --use_low_level_refine=False \
    --spectral_guided=False \
    --in_channels=3 \
    --num_classes=3 \
    --use_low_level_refine=True \
    --temperature=1.0

In [ ]:
!python /kaggle/working/WeedsGalore_Ipr28/src/evaluate.py \
    --dataset_path=/kaggle/working/weedsgalore-dataset \
    --split=test \
    --ckpt=/kaggle/working/out_dir_rgb_seed2/best_val.pth \
    --segformer=True \
    --pretrained_backbone=True \
    --use_low_level_refine=False \
    --spectral_guided=False \
    --in_channels=3 \
    --num_classes=3 \
    --use_low_level_refine=True \
    --temperature=1.0

In [ ]:
!python /kaggle/working/WeedsGalore_Ipr28/src/evaluate.py \
    --dataset_path=/kaggle/working/weedsgalore-dataset \
    --split=val \
    --ckpt=/kaggle/working/out_dir_rgb_seed3/best_val.pth \
    --segformer=True \
    --pretrained_backbone=True \
    --use_low_level_refine=False \
    --spectral_guided=False \
    --in_channels=3 \
    --num_classes=3 \
    --use_low_level_refine=True \
    --temperature=1.0

In [ ]:
!python /kaggle/working/WeedsGalore_Ipr28/src/evaluate.py \
    --dataset_path=/kaggle/working/weedsgalore-dataset \
    --split=test \
    --ckpt=/kaggle/working/out_dir_rgb_seed3/best_val.pth \
    --segformer=True \
    --pretrained_backbone=True \
    --use_low_level_refine=False \
    --spectral_guided=False \
    --in_channels=3 \
    --num_classes=3 \
    --use_low_level_refine=True \
    --temperature=1.0

In [ ]:
!python /kaggle/working/WeedsGalore_Ipr28/src/evaluate.py \
    --dataset_path=/kaggle/working/weedsgalore-dataset \
    --split=val \
    --ckpt=/kaggle/working/out_dir_rgb_seed4/best_val.pth \
    --segformer=True \
    --pretrained_backbone=True \
    --use_low_level_refine=False \
    --spectral_guided=False \
    --in_channels=3 \
    --num_classes=3 \
    --use_low_level_refine=True \
    --temperature=1.0

In [ ]:
!python /kaggle/working/WeedsGalore_Ipr28/src/evaluate.py \
    --dataset_path=/kaggle/working/weedsgalore-dataset \
    --split=test \
    --ckpt=/kaggle/working/out_dir_rgb_seed4/best_val.pth \
    --segformer=True \
    --pretrained_backbone=True \
    --use_low_level_refine=False \
    --spectral_guided=False \
    --in_channels=3 \
    --num_classes=3 \
    --use_low_level_refine=True \
    --temperature=1.0

In [ ]:
for seed in [0, 1, 2, 3, 4]:
    !python /kaggle/working/WeedsGalore_Ipr28/src/train.py \
        --dataset_path=/kaggle/working/weedsgalore-dataset \
        --dataset_size_train=104 \
        --segformer=True \
        --spectral_guided=False \
        --use_spectral_indices=False \
        --use_attention=False \
        --dlv3p_do=False \
        --in_channels=5 \
        --num_classes=3 \
        --pretrained_backbone=True \
        --use_low_level_refine=True \
        --batch_size=8 \
        --num_workers=4 \
        --lr=0.0003 \
        --weight_bg=0.9 \
        --weight_crop=1.0 \
        --weight_weed=1.3 \
        --use_dice_loss=True \
        --dice_weight=0.3 \
        --epochs=100 \
        --seed={seed} \
        --out_dir=/kaggle/working/out_dir_5band_seed{seed} \
        --log_interval=25 \
        --ckpt_interval=100

In [ ]:
from pathlib import Path

file = Path("/kaggle/working/WeedsGalore_Ipr28/src/evaluate.py")
text = file.read_text()

old = 'print("Architecture: 5-band SegFormer")'
new = '''print(f"Architecture: SegFormer ({FLAGS.in_channels}-channel, "
              f"LowLevelRefine={FLAGS.use_low_level_refine})")'''

assert old in text, "Không tìm thấy dòng cũ hoặc file đã thay đổi!"

file.write_text(text.replace(old, new, 1))
print("Đã cập nhật thành công dòng 65!")

In [ ]:
%cd /kaggle/working/WeedsGalore_Ipr28

import subprocess
import sys

cmd = [
    sys.executable,
    "src/calibrate_temperature.py",
    "--dataset_path", "/kaggle/working/weedsgalore-dataset",
    "--ckpt", "/kaggle/working/out_dir_rgb_seed0/best_val.pth",
    "--segformer=True",
    "--pretrained_backbone=True",
    "--spectral_guided=False",
    "--in_channels", "3",
    "--num_classes", "3",
    "--use_low_level_refine=True",
]

result = subprocess.run(cmd, capture_output=True, text=True)

print("Exit code:", result.returncode)
print("\n===== STDOUT =====")
print(result.stdout)
print("\n===== STDERR =====")
print(result.stderr)

In [ ]:
import subprocess
import sys
import re
import numpy as np
from pathlib import Path

REPO = "/kaggle/working/WeedsGalore_Ipr28"
DATASET = "/kaggle/working/weedsgalore-dataset"

# Trong mọi subprocess, chạy từ thư mục repository
def run_command(command):
    result = subprocess.run(
        command,
        cwd=REPO,
        capture_output=True,
        text=True,
    )
    if result.returncode != 0:
        print("STDOUT:\n", result.stdout)
        print("STDERR:\n", result.stderr)
        raise RuntimeError(f"Command failed with exit code {result.returncode}")
    return result.stdout + result.stderr

experiments = {
    "RGB": [
        f"/kaggle/working/out_dir_rgb_seed{i}/best_val.pth"
        for i in range(5)
    ],
    "5-band": [
        f"/kaggle/working/out_dir_5band_seed{i}/best_val.pth"
        for i in range(5)
    ],
}

results = {}

for name, checkpoints in experiments.items():
    miou_values = []
    ece_values = []
    temperatures = []

    in_channels = 3 if name == "RGB" else 5

    for seed, checkpoint in enumerate(checkpoints):
        if not Path(checkpoint).exists():
            print(f"SKIP: {checkpoint}")
            continue

        common = [
            f"--dataset_path={DATASET}",
            f"--ckpt={checkpoint}",
            "--segformer=True",
            "--pretrained_backbone=True",
            "--spectral_guided=False",
            f"--in_channels={in_channels}",
            "--num_classes=3",
            "--use_low_level_refine=True",
        ]

        calibration_log = run_command(
            [sys.executable, "src/calibrate_temperature.py"] + common
        )

        match = re.search(
            r"Fitted temperature:\s*T\s*=\s*([0-9.]+)",
            calibration_log,
        )
        if match is None:
            raise RuntimeError(
                f"Không đọc được temperature cho {name}, seed {seed}"
            )

        temperature = float(match.group(1))
        temperatures.append(temperature)

        evaluation_log = run_command([
            sys.executable,
            "src/evaluate.py",
            "--split=test",
            *common,
            f"--temperature={temperature}",
        ])

        print(evaluation_log)

        miou_match = re.search(r"mIoU:\s*([0-9.]+)%", evaluation_log)
        ece_match = re.search(r"ECE:\s*(?:tensor\()?([0-9.]+)", evaluation_log)

        if miou_match is None or ece_match is None:
            raise RuntimeError(f"Không đọc được kết quả cho {name}, seed {seed}")

        miou_values.append(float(miou_match.group(1)))
        ece_values.append(float(ece_match.group(1)))

    results[name] = {
        "mIoU": miou_values,
        "ECE": ece_values,
        "T": temperatures,
    }

print("\n===== SUMMARY =====")
for name, values in results.items():
    print(f"\n{name}")
    print("Temperatures:", values["T"])
    print(
        f"Test mIoU: {np.mean(values['mIoU']):.2f} "
        f"+/- {np.std(values['mIoU'], ddof=1):.2f}%"
    )
    print(
        f"Test ECE: {np.mean(values['ECE']):.5f} "
        f"+/- {np.std(values['ECE'], ddof=1):.5f}"
    )